# Práctica 1 — Aplicación y rúbrica

**Aplicación:** PWA estática **Triaje financiero**. Recibe un fragmento de un business case y devuelve clase, probabilidad calibrada y umbral. La inferencia ocurre en el navegador; no hay LLM ni clasificador preentrenado.

Este notebook documenta el sistema, incluye el código de la app en las secciones 3 y 4, y deja embebido el dataset con el que se entrenó el modelo. No incluye identificadores de equipo, documento ni personas.

| Criterio de la rúbrica | Dónde se cumple | Cifra |
|---|---|---|
| 1. Aplicación funcional | `practica1/pwa/` | Texto nuevo → clase, probabilidad y umbral; OOD no tumba la app; ~0.05 ms por inferencia |
| 2. Datos y honestidad | corpus de PDF y split temporal por equipo | 500 train / 130 val / 202 test; test sin tocar |
| 3. Evaluación y umbral | baseline, matriz, barrido por costo, costo/1000, ECE | umbral 0.26; costo/1000 514.9; ECE 0.151 → 0.084 |
| 4. Errores y fuera de dominio | `evaluacion.html` + `evaluacion.js` | 20 errores agrupados; 8/10 OOD rechazados |
| 5. Razonamiento escrito | este notebook y el PDF de 6–8 páginas | el PDF aún no es la entrega |

La corrida sigue siendo **provisional**: las etiquetas de test son débiles. La rúbrica exige auditar el test al 100% antes de la corrida final. El video y el PDF todavía no se entregan.


## 2. Datos y etiquetado

El corpus contiene **44 PDF de competencia y 2 PPTX**, con 46 documentos y 34 equipos. Todos los PDF de competencia disponibles están incluidos. El documento que no entra es la guía del evento, porque no es un reporte de equipo.

- Dataset base con fragmentación principal: **666 fragmentos** (138 finanzas/riesgo, 528 mercado/propuesta).
- Entrenamiento: **500 fragmentos** (115 positivos, 385 negativos). De ellos, **166** son fragmentos extra más finos de documentos que ya estaban en train.
- Validación: **130**; test: **202** (30 positivos).
- El refuerzo de train nunca toca val/test. Test sigue con las mismas 202 filas.

Cómo se etiquetó: una sola persona, con etiqueta débil a partir del heading de la sección. Es un límite del montaje y debe aparecer en el PDF. El test todavía no está auditado.

Split: temporal y agrupado por equipo. Los equipos más recientes van a test; ningún equipo aparece en dos splits. El umbral y la temperatura se eligen solo con validación.

Alternativa descartada: split aleatorio, porque los documentos tienen fecha y la rúbrica pide split temporal cuando la hay. También se descartó reducir el tamaño mínimo de fragmento en todo el dataset: eso empeoró el costo de test (668.2 por 1000) y cambiaba el test. El refuerzo se aplicó solo a train.


## Dataset de entrenamiento

Las 500 filas siguientes son las que entran al ajuste. Las 166 marcadas con `augmented=1` son el refuerzo de train; val/test no están aquí. El texto ya pasó por el filtro de correos, URLs, teléfonos y nombres conocidos.


In [1]:
import json
import pandas as pd

TRAIN_JSON = "[{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":590,\"augmented\":0,\"text\":\"Ofrecemos dos variantes sobre la misma plataforma mecánica monoplaza 2WD para simplificar el inventario de refacciones. La versión utilitaria incorpora un módulo trasero de carga para 65 kg de herramientas, punto de remolque para arrastrar un carro ligero con 150 kg adicionales de piezas o insumos, recubrimiento anticorrosivo marino y faros de trabajo, mientras que la versión base se orienta a supervisión ágil y rutas recreativas guiadas. La inversión total por las 70 unidades es de $871,653.70 USD en flota con un paquete de soporte inicial y refacciones de $30,826.80 USD en soporte.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":282,\"augmented\":0,\"text\":\"Tan solo en los primeros 12 meses de obra, la compra del primer lote genera un ahorro neto de $51,023.66 USD en el primer año frente al arrendamiento comercial de unidades equivalentes, alcanzando un periodo de recuperación de 10.4 meses y una disponibilidad operativa meta del 96%.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":362,\"augmented\":0,\"text\":\"Nuestra plantilla está integrada por 20 profesionales de diversas disciplinas de ingeniería y administración, con una composición multidisciplinaria e inclusiva enfocada en la colaboración técnica y operativa. La estructura opera mediante áreas de dirección, gestión de proyectos, diseño mecánico, manufactura, chasis, suspensión, tren motriz, compras y calidad.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":479,\"augmented\":0,\"text\":\"Acorde a las necesidades de Outdoor Adventure Holdings, la configuración comercial soporta a un operador de hasta 110 kg e incorpora un módulo trasero desmontable con capacidad de 65 kg para herramientas, punto de remolque trasero para un carro utilitario ligero de hasta 150 kg de piezas e insumos, faros auxiliares LED, salpicaderas ampliadas contra lodo y silenciador adicional para zonas de reserva ecológica, manteniendo intacta la base del reporte de costos de competencia.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":545,\"augmented\":0,\"text\":\"Configuración elegida Nuestra plataforma mantiene una arquitectura de tracción trasera 2WD optimizada bajo el reglamento Baja SAE México 2026. La arquitectura 4WD se incorpora como una mejora conceptual modular para requerimientos de terreno extremo en zonas de arena suelta y lodo en Yucatán. Para las condiciones de suelo calizo, arena costera, lodo y vegetación densa de Yucatán, el vehículo presenta dimensiones generales de 2.20 m de largo, 0.88 m de ancho y 1.26 m de alto, con una altura libre al suelo de 305 mm y peso en seco de 215 kg.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":397,\"augmented\":0,\"text\":\"El sistema de dirección utiliza una cremallera reforzada con columna de aluminio 7075 de 1 pulgada y juntas universales de 36 estrías con fuelles guardapolvo. El frenado opera mediante doble circuito hidráulico con cilindros maestros Tilton Serie 74 de 7\\/8 de pulgada, barra de balance Tilton 600 y calipers con discos Kawasaki Brute Force, garantizando el bloqueo simultáneo de las cuatro ruedas.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":293,\"augmented\":0,\"text\":\"El habitáculo protege al operador con asiento ergonómico, arnés de cinco puntos SFI 16.1, sujeciones de brazos SFI 3.3, soporte de cuello, doble interruptor de paro Polaris, luz de freno LED y extintor Kidde con soporte rápido, permitiendo la salida por ambos lados en menos de cinco segundos.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":622,\"augmented\":0,\"text\":\"Requisitos de seguridad y costeo El chasis tubular se fabrica en acero tubular AISI 1020 DOM con corte láser rotativo y soldadura controlada en dispositivos de sujeción, verificado mediante análisis de elementos finitos con factor de seguridad superior a 1.8 en casos de impacto frontal, lateral y vuelco. Toda la estructura recibe recubrimiento anticorrosivo por aspersión líquida para resistir la intemperie costera. La suspensión independiente en las cuatro ruedas integra amortiguadores ajustables de largo recorrido y ocho rótulas de acero aleado de 5\\/8 de pulgada Rod End Supply que absorben irregularidades severas.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":501,\"augmented\":0,\"text\":\"Capacidad institucional Operamos desde los talleres de manufactura y laboratorios mecánicos en Torreón, Coahuila, equipados con estaciones de soldadura MIG, corte, maquinado convencional y CNC, dobladoras hidráulicas y dispositivos de alineación, complementados por proveedores industriales certificados en la Comarca Lagunera para corte láser rotativo de tubería y tratamientos térmicos. El trabajo se administra con planos controlados, trazabilidad por pieza e inspecciones dimensionales por unidad.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":443,\"augmented\":0,\"text\":\"La propulsión emplea el motor Rehlko CH440 acoplado a una transmisión variable continua CVTech y un tren motriz de tracción trasera 2WD con diferencial y mazas de uso rudo Kawasaki Brute Force 750, alimentado por un tanque certificado Pyrotect anti-derrames. El sistema 2WD proporciona tracción eficiente y confiable sin requerir intervención compleja del operador, facilitando el manejo para huéspedes en rutas recreativas y personal de obra.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":258,\"augmented\":1,\"text\":\"La propulsión emplea el motor Rehlko CH440 acoplado a una transmisión variable continua CVTech y un tren motriz de tracción trasera 2WD con diferencial y mazas de uso rudo Kawasaki Brute Force 750, alimentado por un tanque certificado Pyrotect anti-derrames.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":184,\"augmented\":1,\"text\":\"El sistema 2WD proporciona tracción eficiente y confiable sin requerir intervención compleja del operador, facilitando el manejo para huéspedes en rutas recreativas y personal de obra.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":583,\"augmented\":0,\"text\":\"En el ejercicio actual de desarrollo del prototipo base, registramos ingresos por $14,246.22 USD respaldados por alianzas estratégicas con Enlace Digital por $2,034.88 USD, Canacintra por $1,162.79 USD, la Escuela de Ingeniería por $872.09 USD y Amher por $290.70 USD, sumados a $9,885.76 USD generados mediante actividades comerciales y patrocinios en especie del equipo. Con una inversión ejecutada de $12,107.49 USD en el desarrollo integral del prototipo demostrador acorde al reporte de costos oficial, mantenemos un flujo de caja positivo de $1,002.03 USD y cero endeudamiento.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":639,\"augmented\":0,\"text\":\"Criterio de aceptación de ascenso en pendientes del 35 por ciento con tracción 2WD activa, temperatura de transmisión CVTech inferior a 95 °C, consumo promedio menor a 1.8 litros por hora y recorrido continuo de cuatro horas sin pérdida de presión en amortiguadores ni holgura en rótulas. 4. Operación y mantenimiento. Responsable Área de Manufactura y Ensamble. Programada del 19 al 22 de noviembre de 2026. Cronometraje de rutinas de servicio con criterio de aceptación menor a 15 minutos para cambio de filtro de aire, ajuste de banda CVT y revisión de niveles, más reemplazo de rueda en menos de seis minutos con herramienta básica. 5.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":552,\"augmented\":0,\"text\":\"Purgado de líneas independientes con cilindros Tilton Serie 74, verificación de retorno libre del acelerador, operación de ambos interruptores de paro Polaris, ausencia total de fugas en celda Pyrotect y bloqueo efectivo de las cuatro ruedas sobre asfalto a 40 km\\/h en una distancia máxima de 8.5 metros. 3. Pruebas de terreno. Responsables Elías Lucero Marmolejo en Tren Motriz y Azael Soto Díaz en Suspensión. Programada del 9 al 18 de noviembre de 2026 en terrenos de arena, terracería pedregosa y pendientes equivalentes a la topografía de Yucatán.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":219,\"augmented\":0,\"text\":\"Ejecutamos un protocolo de validación por etapas con métricas cuantitativas de aceptación y bitácoras firmadas antes de presentar el vehículo a Outdoor Adventure Holdings para su prueba de manejo y evaluación de compra.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":320,\"augmented\":1,\"text\":\"Responsable Área de Manufactura y Ensamble. Programada del 19 al 22 de noviembre de 2026. Cronometraje de rutinas de servicio con criterio de aceptación menor a 15 minutos para cambio de filtro de aire, ajuste de banda CVT y revisión de niveles, más reemplazo de rueda en menos de seis minutos con herramienta básica. 5.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":266,\"augmented\":1,\"text\":\"Prueba presencial de manejo para Outdoor Adventure Holdings en circuito mixto de arena, lodo y obstáculos, evaluando maniobrabilidad con radio de giro de 3.8 metros, estabilidad con 65 kg de carga útil en el módulo trasero y simulación de inspección diaria de flota.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":526,\"augmented\":0,\"text\":\"1. Diseño y seguridad estructural. Responsable Johan Benjamín García Domínguez, área de Chasis. Completada en octubre de 2026. Inspección dimensional de jaula frente a planos CAD, verificación visual de cordones de soldadura, prueba de anclajes de arnés SFI 16.1 y cronometraje de egreso del operador con criterio de aceptación menor a 4.2 segundos por ambos lados. 2. Integración y controles. Responsables Juan Pablo Rangel Magaña en Frenos y Fernando Landeros Fierro en Dirección. Programada del 1 al 8 de noviembre de 2026.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":448,\"augmented\":0,\"text\":\"Demostración ejecutiva al cliente. Responsables David de la Fuente Adame en Negocios y Elías Lucero Marmolejo en Coordinación. Programada para la última semana de noviembre de 2026. Prueba presencial de manejo para Outdoor Adventure Holdings en circuito mixto de arena, lodo y obstáculos, evaluando maniobrabilidad con radio de giro de 3.8 metros, estabilidad con 65 kg de carga útil en el módulo trasero y simulación de inspección diaria de flota.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":481,\"augmented\":0,\"text\":\"Programa del cliente Etapa Meta Condición Meses 0-5 20 unidades Producción escalonada de cuatro a cinco unidades aceptadas al mes Mes 6 Primera entrega 20 unidades utilitarias liberadas y puestas en Yucatán Meses 7-17 50 unidades adicionales Ritmo sostenido de cinco unidades al mes con mejoras de campo Mes 18 Segunda entrega 50 unidades adicionales liberadas para operación diaria Cada vehículo requiere 45 horas-hombre de ensamble integradas en el costo unitario de manufactura.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":611,\"augmented\":0,\"text\":\"La producción en serie se organiza mediante celdas de manufactura esbelta con trazabilidad por número de serie y control dimensional en cada estación. Nuestra planta de ensamble en Torreón, Coahuila cuenta con un área operativa de 450 metros cuadrados distribuida en seis zonas secuenciales que comprenden recepción y control de calidad de insumos, preparación y soldadura de chasis en dispositivos Poka-Yoke, pintura anticorrosiva por aspersión líquida en cabina, subensamble de tren motriz 2WD y suspensión, línea de integración final con torqueo certificado, y patio de pruebas dinámicas previas al embarque.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":421,\"augmented\":0,\"text\":\"Los componentes especializados se adquieren directamente de fabricantes líderes como CVTech para transmisiones, Tilton para sistemas de frenado, Rod End Supply para rótulas de suspensión y Pyrotect para celdas de combustible, manteniendo un inventario de seguridad de 10 juegos en almacén. Aplicamos metodología 5S, tableros Kanban para reabastecimiento automático y juntas semanales Kaizen para reducir tiempos de ciclo.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":113,\"augmented\":1,\"text\":\"suspensión, línea de integración final con torqueo certificado, y patio de pruebas dinámicas previas al embarque.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":350,\"augmented\":1,\"text\":\"Programa del cliente Etapa Meta Condición Meses 0-5 20 unidades Producción escalonada de cuatro a cinco unidades aceptadas al mes Mes 6 Primera entrega 20 unidades utilitarias liberadas y puestas en Yucatán Meses 7-17 50 unidades adicionales Ritmo sostenido de cinco unidades al mes con mejoras de campo Mes 18 Segunda entrega 50 unidades adicionales\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":150,\"augmented\":1,\"text\":\"La producción en serie se organiza mediante celdas de manufactura esbelta con trazabilidad por número de serie y control dimensional en cada estación.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":130,\"augmented\":1,\"text\":\"liberadas para operación diaria Cada vehículo requiere 45 horas-hombre de ensamble integradas en el costo unitario de manufactura.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":131,\"augmented\":1,\"text\":\"Aplicamos metodología 5S, tableros Kanban para reabastecimiento automático y juntas semanales Kaizen para reducir tiempos de ciclo.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":94,\"augmented\":1,\"text\":\"Trabajamos con una cadena de suministro validada durante la construcción de nuestro prototipo.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":246,\"augmented\":0,\"text\":\"Con una plantilla operativa de cinco técnicos y un supervisor disponemos de 800 horas-hombre mensuales en planta para producir hasta ocho unidades al mes frente al requerimiento de cinco unidades, garantizando holgura de calidad sin sobretiempos.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":386,\"augmented\":0,\"text\":\"Trabajamos con una cadena de suministro validada durante la construcción de nuestro prototipo. Los aceros y el aluminio estructural provienen de distribuidores nacionales como Compañía Metalúrgica La Paloma con certificados de colada, mientras que el corte láser rotativo y doblado CNC de tubería se contrata con proveedores regionales bajo auditoría dimensional del 100 % de los lotes.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":6,\"char_len\":323,\"augmented\":0,\"text\":\"El pedido comprende el Lote 1 con 20 unidades Cargo en el mes 6 por $255,118.31 USD y el Lote 2 con 50 unidades en el mes 18 por $616,535.39 USD con 30 unidades Cargo y 20 Base, totalizando $871,653.70 USD en flota más $30,826.80 USD de soporte inicial, refacciones y taller, para un valor llave en mano de $902,480.50 USD.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":6,\"char_len\":193,\"augmented\":1,\"text\":\"Para la unidad monoplaza 2WD base el costo unitario de producción es de $9,301.19 USD con precio de venta de $11,692.92 USD o $220,000.00 MXN y margen de $2,391.73 USD correspondiente al 20.5%.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":6,\"char_len\":438,\"augmented\":0,\"text\":\"Para la unidad monoplaza 2WD base el costo unitario de producción es de $9,301.19 USD con precio de venta de $11,692.92 USD o $220,000.00 MXN y margen de $2,391.73 USD correspondiente al 20.5%. Para la unidad monoplaza 2WD utilitaria de carga con remolque con módulo para 65 kg y remolque para 150 kg el costo unitario es de $10,098.43 USD con precio de $12,755.92 USD o $240,000.00 MXN y margen de $2,657.49 USD correspondiente al 20.8%.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":6,\"char_len\":211,\"augmented\":0,\"text\":\"El modelo financiero parte del reporte oficial de costos del prototipo de $12,107.49 USD. La escala a 70 unidades optimiza compras por volumen y herramentales propios manteniendo componentes de grado industrial.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":279,\"augmented\":0,\"text\":\"El fondeo combina $32,400.00 USD de capital institucional y aportaciones de socios con una línea de crédito comercial PyME por $60,000.00 USD a tasa de 14.00% anual que suma TIIE de 6.80% y sobretasa bancaria de 7.20%, amortizada contra las liquidaciones de entrega de cada lote.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":638,\"augmented\":0,\"text\":\"Para iniciar operaciones se requiere una inversión inicial de $92,400.00 USD distribuida en adecuación de planta y depósito de nave en Torreón por $16,579.07 USD, herramentales y fixtures Poka-Yoke por $26,900.00 USD, equipo de metrología y software por $8,700.00 USD, permisos y pólizas por $4,900.00 USD, arranque y capacitación por $6,709.30 USD, y capital de trabajo por $28,611.63 USD. En los gastos fijos mensuales, el componente salarial corresponde a sueldo y gestión administrativa de $15,000 MXN equivalentes a $797.24 USD mensuales de administración y gerencia de planta alineado al rubro Administrativo del control financiero.\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":645,\"augmented\":0,\"text\":\"Operar con requiere $164.76 USD mensuales por unidad en mantenimiento, generando un ahorro neto de $51,023.66 USD en el primer año con recuperación en 10.4 meses y retorno proyectado a tres años del Flujos financieros semestrales a 24 meses en USD Rubro Semestre 1 Semestre 2 Semestre 3 Semestre 4 Total Consolidado Hito operativo Lote 1 con 20 Soporte y Lote Lote 2 con 50 Flota 70 y tours Balance mes 24 Cargo 2 unidades UTV Ingresos USD $255,118.31 $19,771.35 $616,535.39 $195,595.66 $1,087,020.71 Costos y $230,765.14 $11,427.71 $523,912.72 $59,422.37 $825,527.94 amortizaciones USD Flujo neto USD $24,353.17 $8,343.64 $92,622.67 $136,173.29\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":247,\"augmented\":1,\"text\":\"En los gastos fijos mensuales, el componente salarial corresponde a sueldo y gestión administrativa de $15,000 MXN equivalentes a $797.24 USD mensuales de administración y gerencia de planta alineado al rubro Administrativo del control financiero.\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":344,\"augmented\":1,\"text\":\"Operar con requiere $164.76 USD mensuales por unidad en mantenimiento, generando un ahorro neto de $51,023.66 USD en el primer año con recuperación en 10.4 meses y retorno proyectado a tres años del Flujos financieros semestrales a 24 meses en USD Rubro Semestre 1 Semestre 2 Semestre 3 Semestre 4 Total Consolidado Hito operativo Lote 1 con 20\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":648,\"augmented\":0,\"text\":\"Con costos fijos de $50,492.17 USD alcanzamos punto de equilibrio en la unidad 20 en el mes 6. En el modelo de negocio, Outdoor Adventure Holdings no liquida la totalidad por anticipado sino contra entrega y aceptación de cada lote con 30% de anticipo comercial a la firma. financia la manufactura mediante crédito comercial PyME a condiciones de mercado mexicano con TIIE de 6.80% más sobretasa de 7.20% totalizando una tasa anual de 14.00%, saldada puntualmente con las liquidaciones de entrega. Para el cliente rentar flota utilitaria costaría $1,275.59 USD mensuales por vehículo sumando un costo evitado de $306,141.97 USD en 12 meses de obra.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":467,\"augmented\":0,\"text\":\"Proponemos destinar un circuito perimetral de 45 kilómetros dentro de las zonas de amortiguamiento del predio en Yucatán para crear la experiencia Eco-Ruta Maya, operada con las 20 unidades Base 2WD del segundo lote. Al ser vehículos monoplaza de fácil conducción con limitador de velocidad regulable y jaula certificada, los huéspedes del hotel pueden conducir su propia unidad en caravanas guiadas de ocho vehículos acompañados por dos guías en unidades de soporte.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":559,\"augmented\":0,\"text\":\"Parte del margen comercial se reinvierte en alianzas con proveedores para integrar materiales avanzados como horquillas de acero microaleado de alta resistencia, paneles compuestos ligeros y corte láser automatizado en serie que reducen el peso un 8% y disminuyen el costo de manufactura en lotes subsecuentes. La logística de traslado desde Torreón hasta Yucatán se realiza en plataformas madrina cerradas con capacidad de 10 unidades por viaje y seguro de carga total, con un costo logístico de $3,614.18 USD por embarque ya integrado en nuestra cotización.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":630,\"augmented\":0,\"text\":\"Posicionamiento La operación del circuito UTV dentro de propiedad privada cumple con la normatividad ambiental de SEMARNAT mediante el aprovechamiento de brechas existentes sin deforestación adicional, uso de silenciadores de baja emisión acústica por debajo de 82 decibeles para proteger la fauna de la reserva y lavado de chasis en trampas de sólidos. Cada participante firma responsiva digital, recibe capacitación práctica de 15 minutos en pista escuela, utiliza casco certificado ECE R22-06, gafas, collarín y arnés de cinco puntos, y circula por rutas clasificadas por dificultad con comunicación por radio en cada vehículo.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":511,\"augmented\":0,\"text\":\"Con tres recorridos diarios de dos horas a una tarifa de $98.33 USD por participante y una ocupación conservadora del 50%, tan solo 10 unidades destinadas a recorridos turísticos generan ingresos brutos de $29,498.05 USD mensuales para el complejo, amortizando el costo del segundo lote de 50 vehículos en menos de 15 meses. Adicionalmente, la ruta se comercializa mediante pases de día para visitantes externos, patrocinios de marcas deportivas en estaciones de descanso y paquetes fotográficos en el circuito.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":250,\"augmented\":1,\"text\":\"Al ser vehículos monoplaza de fácil conducción con limitador de velocidad regulable y jaula certificada, los huéspedes del hotel pueden conducir su propia unidad en caravanas guiadas de ocho vehículos acompañados por dos guías en unidades de soporte.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":248,\"augmented\":1,\"text\":\"La logística de traslado desde Torreón hasta Yucatán se realiza en plataformas madrina cerradas con capacidad de 10 unidades por viaje y seguro de carga total, con un costo logístico de $3,614.18 USD por embarque ya integrado en nuestra cotización.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":186,\"augmented\":1,\"text\":\"Adicionalmente, la ruta se comercializa mediante pases de día para visitantes externos, patrocinios de marcas deportivas en estaciones de descanso y paquetes fotográficos en el circuito.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":324,\"augmented\":1,\"text\":\"Con tres recorridos diarios de dos horas a una tarifa de $98.33 USD por participante y una ocupación conservadora del 50%, tan solo 10 unidades destinadas a recorridos turísticos generan ingresos brutos de $29,498.05 USD mensuales para el complejo, amortizando el costo del segundo lote de 50 vehículos en menos de 15 meses.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":532,\"augmented\":0,\"text\":\"Canales y entrega Nuestra estrategia comercial B2B se enfoca en desarrollos ecoturísticos, complejos hoteleros, parques de aventura y reservas agroindustriales en México mediante venta directa consultiva, demostraciones presenciales en terreno y alianzas con cámaras empresariales como Canacintra. Asignamos un presupuesto de mercadotecnia del 4% de las ventas, equivalente a $24,448.84 USD anuales, enfocado en pruebas de manejo para clientes institucionales, contenido técnico digital y presencia en ferias de turismo sustentable.\"},{\"section\":\"mercado\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":262,\"augmented\":0,\"text\":\"–IPN, Guanajuato. Operamos en el Grupo multidisciplinario de estudiantes Competimos en Baja SAE México y Bajío, junto a la proveeduría automotriz de ingeniería del IPN, organizado en nos adaptamos a tus proyectos. de León y Celaya. áreas técnicas par de gestión.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":288,\"augmented\":0,\"text\":\"13.7 % de las autopartes del país se Chasis tubular · Tratamiento Anticorrosión (León, Celaya) producen en Guanajuato Iluminación y componentes · Ensamble y pruebas ENTREGAR FLOTA 1 OBRA EN MARCHA SEGUNDO LOTE FLOTA DE 70 Mes 0 al mes 6 Mes 6 al mes 18 Mes 12 al mes 18 Mes 18 en adelante\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":5,\"char_len\":80,\"augmented\":0,\"text\":\"Entrega antes del inicio de obra Entrega en el mes 18 Flota de 70 frente a renta\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":5,\"char_len\":649,\"augmented\":0,\"text\":\"Pieza o proceso comprado Origen Costo por unidad (USD) Plazo de entrega Chasis tubular (acero) Talleres de Guanajuato $941.98 15 a 20 días hábiles Tratamiento anticorrosión Plantas de León y Celaya $95.00 5 a 7 días hábiles Iluminación y componentes Proveeduría local $107.53 3 a 5 dias hábiles Llantas todoterreno Distribuidor Nacional Maxxis 3831.72 10 a 15 días hábiles Asiento del pasajero y cinturones e de equipo de seguridad $214.00 15 a 20 días hábiles Techo abatible y gancho de arrastre Proveeduria local 3875.00 7 a 10 días hábiles LOTE 1 - 20 UNIDADES LOTE 2 - 50 UNIDADES COSTO TOTAL A 5 AÑOS $224,147.80 USD $560,369.50 USD $784,517.30\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":261,\"augmented\":0,\"text\":\"@ Homologacion PLANEAR HACER swore y muestra inicial aprobada antes del primer Metas de calidad y plan de Produccién y pruebas con pecico. inspección por pieza. instrucciones de trabajo. Especificación Planos, tolerancias y certificado de material en cada lote.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":325,\"augmented\":1,\"text\":\"Campo Descripción Empresa Actividad Diseño, fabricación, ensamble y venta de vehículos Baja monoplaza Etapa Empresa en crecimiento; prototipo en fabricación Equipo actual 10 integrantes Salvador González Lobo , República Oriente Nte., 25280 Instalaciones Saltillo, Coah. Modelo comercial Fabricante y proveedor independiente;\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":397,\"augmented\":0,\"text\":\"Campo Descripción Empresa Actividad Diseño, fabricación, ensamble y venta de vehículos Baja monoplaza Etapa Empresa en crecimiento; prototipo en fabricación Equipo actual 10 integrantes Salvador González Lobo , República Oriente Nte., 25280 Instalaciones Saltillo, Coah. Modelo comercial Fabricante y proveedor independiente; venta por contrato Cliente de esta propuesta Outdoor Adventure Holdings\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":359,\"augmented\":0,\"text\":\"• Venta directa y especificación de lote uniforme para facilitar piezas de repuesto. • Programa de pruebas del prototipo y aceptación por parte del cliente antes de la producción en serie. • Manual de inspección, mantenimiento preventivo y catálogo de refacciones. • Capacitación básica al personal operador y técnico del cliente como servicio presupuestable.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":3,\"char_len\":88,\"augmented\":0,\"text\":\"preliminar) Corte, soldadura, rolado y acabados $12,000 Mano de obra de ensamble $16,000\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":565,\"augmented\":0,\"text\":\"Prototipo, pruebas, planos, proveedores y Meses 0–1 Validación y preparación utillajes Primera entrega contractual (4 Meses 2–6 20 unidades unidades\\/mes de referencia) Meses 7–12 25 unidades Incremento gradual de producción Segunda entrega acumulada de 50 unidades Meses 13–18 25 unidades adicionales La estrategia propone manufactura interna para diseño, ensamble, supervisión y control de calidad, y subcontratación selectiva de corte, rolado y acabados cuando resulte necesario. La meta operativa de cinco unidades por mes no es todavía una capacidad comprobada.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":249,\"augmented\":0,\"text\":\"La capacidad mensual real aún no se ha medido. Para presentar una oferta competitiva y responsable, se plantea una meta escalonada, condicionada a completar el prototipo, medir horas de fabricación, validar seguridad y disponibilidad de proveedores.\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":278,\"augmented\":1,\"text\":\"propone un vehículo Baja monoplaza orientado a recorridos de inspección, supervisión y desplazamiento individual en superficies irregulares. No se ofrecerá como sustituto general de un vehículo de carga o transporte de varios ocupantes sin validación técnica de sus capacidades.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":143,\"augmented\":1,\"text\":\"Elemento Dato disponible Tipo Vehículo Baja monoplaza; se excluye la categoría UTV Largo 108 pulgadas (aprox. 2.74 m) Ancho 64 pulgadas (aprox.\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":600,\"augmented\":0,\"text\":\"El comprador desea reducir la dependencia del alquiler de vehículos durante la construcción y contar después con unidades de apoyo en operaciones diarias. Solicita pruebas de manejo antes de contratar, un nivel alto de disponibilidad y una propuesta integral para almacenamiento, mantenimiento, combustible y refacciones. propone un vehículo Baja monoplaza orientado a recorridos de inspección, supervisión y desplazamiento individual en superficies irregulares. No se ofrecerá como sustituto general de un vehículo de carga o transporte de varios ocupantes sin validación técnica de sus capacidades.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":415,\"augmented\":0,\"text\":\"Elemento Dato disponible Tipo Vehículo Baja monoplaza; se excluye la categoría UTV Largo 108 pulgadas (aprox. 2.74 m) Ancho 64 pulgadas (aprox. 1.63 m) Motor Kohler CH440, gasolina, 14 HP, con cuñero Suspensión Independiente de doble horquilla Neumáticos 22 × 10-10 Estado Prototipo en proceso de fabricación Peso final, altura, carga útil, consumo, autonomía, velocidad, Pendientes frenado y ensayos de durabilidad\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":287,\"augmented\":0,\"text\":\"La empresa puede proponer un esquema de pagos contractuales 40% de anticipo, 40% contra hito de inspección y 20% contra entrega aceptada. Los pagos serían anticipos sobre una compraventa. La adjudicación dependerá de que Outdoor Adventure Holdings considere la propuesta la mejor oferta.\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":646,\"augmented\":0,\"text\":\"El mercado inicial se centra en compradores institucionales que demandan movilidad individual en propiedades extensas y senderos no pavimentados, como administradores de complejos recreativos, propietarios de instalaciones rurales y empresas con equipos de inspección. Se excluye el desarrollo comercial de senderos UTV, por decisión de alcance del presente plan. • Preparar dossier técnico y comercial que incluya fichas, fotos reales, mantenimiento y resultados medidos de pruebas. • Ofrecer demostración controlada de manejo y una matriz de aceptación técnica. • Cotizar el costo total de suministro y soporte, no solamente el precio unitario.\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":554,\"augmented\":0,\"text\":\"Indicador 20 vehículos 50 vehículos Total Ingresos a $170,000 por unidad $3,400,000 $8,500,000 $11,900,000 Costo a $124,747 por unidad $2,494,940 $6,237,350 $8,732,290 Margen bruto estimado $905,060 $2,262,650 $3,167,710 El margen bruto unitario estimado es de $45,253 (26.6% sobre ventas). No constituye utilidad neta y no integra todavía costos internacionales de entrega, regulación, homologación, garantías extraordinarias, financiamiento ni impuestos. El precio definitivo se presentará como oferta competitiva una vez cuantificados estos conceptos.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":167,\"augmented\":0,\"text\":\"Herramientas y dispositivos de ensamble $100,000 Equipo adicional de soldadura y corte $100,000 Control de calidad $50,000 Seguridad industrial e instalaciones $40,000\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":97,\"augmented\":0,\"text\":\"• Documentar testimonios y métricas de funcionamiento solo después de obtener datos verificables.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":398,\"augmented\":0,\"text\":\"La inversión fija de $500,000 es adicional al capital de trabajo. La producción inicial de 20 unidades representa $2,494,940 de costo de fabricación acumulado, aunque los desembolsos se realizan por etapas. El anticipo de 40% sobre el primer lote sería de $1,360,000 y no cubre por sí solo todo el costo de ese lote; se requieren planeación de liquidez, crédito de proveedores y\\/o recursos propios.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":130,\"augmented\":0,\"text\":\"Detalle de procesos incluidos en los $12,000: corte $3,000; rolado y doblado $3,000; soldadura $4,500; limpieza y acabados $1,500.\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":290,\"augmented\":1,\"text\":\"Indicador 20 vehículos 50 vehículos Total Ingresos a $170,000 por unidad $3,400,000 $8,500,000 $11,900,000 Costo a $124,747 por unidad $2,494,940 $6,237,350 $8,732,290 Margen bruto estimado $905,060 $2,262,650 $3,167,710 El margen bruto unitario estimado es de $45,253 (26.6% sobre ventas).\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":180,\"augmented\":1,\"text\":\"• Cotizar el costo total de suministro y soporte, no solamente el precio unitario. • Documentar testimonios y métricas de funcionamiento solo después de obtener datos verificables.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":5,\"char_len\":250,\"augmented\":0,\"text\":\"Riesgo Acción recomendada Prototipo no validado Completar ensayos antes de aceptar producción en serie Capacidad no comprobada Medir tiempos reales y desarrollar programa maestro de producción Segundas fuentes, compras críticas anticipadas y stock de\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":252,\"augmented\":0,\"text\":\"seguridad Falta de capital de trabajo Presupuesto mensual de caja y anticipos por hitos Incumplimiento normativo internacional Revisión legal y de seguridad del vehículo Tiempo inactivo elevado Estandarización, repuestos y mantenimiento basado en horas\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":148,\"augmented\":0,\"text\":\"Almacenamiento Área cubierta, ventilada, segura y de fácil acceso para inspección Dos posiciones de servicio como hipótesis inicial, sujetas a carga\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":303,\"augmented\":0,\"text\":\"Se propone fabricar en México y entregar en Yucatán. Antes de cotizar en firme se deben revisar documentación de , seguros, transporte, requisitos de seguridad del producto y condiciones de operación dentro o fuera de vías públicas. Estos costos no se incluyen en la cifra base de $170,000 por vehículo.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":271,\"augmented\":0,\"text\":\"Nuestros vehículos cumplen con estándares estrictos de calidad, asegurándonos en cada paso del proceso y poniéndolos a prueba en el terreno. Algunas de las pruebas realizadas en los vehículos incluyen: Frenado en seco Para la aceptación de cada prueba se toman en cuenta:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":161,\"augmented\":0,\"text\":\"de Inversión (ROI) Nuestro modelo financiero está diseñado para maximizar el Retorno de Inversión mediante una reducción drástica en el costo total de propiedad.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":648,\"augmented\":0,\"text\":\"Tareas -Diseño estructural del -Recepción y -Desarrollo de chasis verificación de material soluciones en organización -Ingeniería de suspensión -Fabricación y Integración de sistemas soldadura del chasis -Gestión y relación con proveedores -Simulaciones y -Línea de ensamblaje validación virtual modular -Control de inventarios -Diseño para la -Control de procesos y -Programación de la manufactura pruebas finales producción -Pruebas de condición -Atención al cliente y postventa -Ergonomía aplicada -Planificación de -Registro compras y mitigación digital\\/seguimiento de de riesgos producto -Sistema de postventa Optimización Financiera y Retorno\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":174,\"augmented\":0,\"text\":\"Esta estructura, combinada con el soporte integral establece un retorno de inversión predecible y sostenible a largo plazo, pero sin afectar la calidad de nuestros productos.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":339,\"augmented\":1,\"text\":\"Nuestros diseños duraderos y una manufactura modular, así como el uso de componentes de alta calidad garantizan la continuidad operativa, lo que optimiza la productividad. Cardenal’s Industries no sólo busca reducir costos, sino que también transformar los vehículos en activos generadores de valor a través de la innovación en el mercado.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":194,\"augmented\":1,\"text\":\"Esto significa, recuperar la inversión. Así mismo, la estrategía de despliegue progresivo permite ajustar la inversión inicial a las necesidades operativas reales, mitigando riesgos financieros.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":116,\"augmented\":1,\"text\":\"está diseñado para maximizar el Retorno de Inversión mediante una reducción drástica en el costo total de propiedad.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":534,\"augmented\":0,\"text\":\"Nuestros diseños duraderos y una manufactura modular, así como el uso de componentes de alta calidad garantizan la continuidad operativa, lo que optimiza la productividad. Cardenal’s Industries no sólo busca reducir costos, sino que también transformar los vehículos en activos generadores de valor a través de la innovación en el mercado. Esto significa, recuperar la inversión. Así mismo, la estrategía de despliegue progresivo permite ajustar la inversión inicial a las necesidades operativas reales, mitigando riesgos financieros.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":542,\"augmented\":0,\"text\":\"Cuando una materia prima o componente sufre retrasos en el mercado, nuestros ingenieros adaptan los procesos para ensamblar la estructura con alternativas equivalentes (y que han sido aprobadas luego de una larga verificación) sin alterar o estancar la línea de producción ni comprometer la calidad de los vehículos. La implementación de esta estrategía busca el equilibrio entre la virtud del producto y el costo operativo. Pueden existir diversas complicaciones de fuerza mayor, pero en Cardenal’s Industries siempre encontramos soluciones.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":450,\"augmented\":0,\"text\":\"Ante la complejidad para adquirir ciertas aleaciones o piezas especializadas, en Cardenal’s Industries priorizamos el aprovechamiento al máximo de la materia prima, transformando la posible merma en soluciones inteligentes. Aseguramos que ningún recurso se pierda, gracias a la rediseñación de nuestros ingenieros y administradores es posible continuar con la línea de producción sin tener que modificar todo el proceso si falta una pieza específica.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":344,\"augmented\":1,\"text\":\"Ante una imposibilidad de los proveedores principales, contamos con planes de opciones alternativas. Cardenal’s Industries se posiciona en la actualidad bajo un perfil de mayor confianza y credibilidad, destacando frente a otras opciones en el mercado debido a que nunca detenemos la línea de ensamblaje a causa de retrasos logísticos externos.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":316,\"augmented\":1,\"text\":\"Cuando una materia prima o componente sufre retrasos en el mercado, nuestros ingenieros adaptan los procesos para ensamblar la estructura con alternativas equivalentes (y que han sido aprobadas luego de una larga verificación) sin alterar o estancar la línea de producción ni comprometer la calidad de los vehículos.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":243,\"augmented\":0,\"text\":\"Cardenal’s Industries se posiciona en la actualidad bajo un perfil de mayor confianza y credibilidad, destacando frente a otras opciones en el mercado debido a que nunca detenemos la línea de ensamblaje a causa de retrasos logísticos externos.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":635,\"augmented\":0,\"text\":\"Para asegurar la continuidad de 5 unidades por mes en nuestra línea de producción la empresa ha implementado una estrategia integral de abastecimiento y gestión de recursos. Contamos con una red sólida y confiable de proveedores dentro del mercado en materiales de alta calidad. Cada aliado estratégico es evaluado bajo estrictos controles de calidad, asegurando un suministro constante y predecible. Dentro del sistema de inventario existe un área dedicada a los elementos de mayor demanda y difícil acceso, para no sufrir desabasto. Ante una imposibilidad de los proveedores principales, contamos con planes de opciones alternativas.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":624,\"augmented\":0,\"text\":\"Sistema Descripción Costo estimado Chasis Tubo de cromolio 4130, placas, soportes, corte, doblez, $88,000 soldadura TIG y consumibles. Motor Motor de motocicleta 400 cc, mantenimiento, radiador, $45,000 ventilador, escape, admisión y adaptación eléctrica. Transmisión Reductora, reversa, diferencial trasero, semiejes, juntas $38,000 homocinéticas y adaptaciones. Suspensión Horquillas, manguetas, mazas, amortiguadores, rótulas, bujes y $49,000 fabricación. Ruedas y Cuatro rines, cuatro neumáticos y accesorios de montaje. $16,000 neumáticos Dirección Volante, columna, terminales, barras y mecanismos de $9,000 dirección.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":463,\"augmented\":0,\"text\":\"2.- Consideraciones de la proyección • Se contempla una fabricación mixta: algunos trabajos se realizarán en taller propio y otros mediante proveedores externos. • Se considera un motor de motocicleta de 400 cc, por lo que será necesario adaptar el sistema de transmisión para incorporar reducción y reversa. • La capacidad de carga de 200 a 300 kg se considera adicional al peso de los dos ocupantes. • La proyección contempla un sistema de tracción trasera 4x2.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":501,\"augmented\":0,\"text\":\"Frenos Discos, cálipers, cilindro maestro, líneas hidráulicas y pedal. $15,000 Cabina y seguridad Dos asientos, arneses, anclajes y paneles de protección. $14,000 Batea de carga Lámina de acero, refuerzos, bisagras, seguros y fabricación. $10,000 Sistema eléctrico y Batería, cableado, iluminación, tablero, pintura y tornillería. $11,000 acabados Integración y Montaje final, ajustes, pruebas de funcionamiento y $18,000 p ruebas correcciones. Subtotal $313,000 Reserva para imprevistos (15%) $46,950\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":103,\"augmented\":0,\"text\":\"Concepto Importe Costos directos de fabricación e integración $313,000 Reserva para imprevistos $46,950\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":268,\"augmented\":1,\"text\":\"Transmisión Reductora, reversa, diferencial trasero, semiejes, juntas $38,000 homocinéticas y adaptaciones. Suspensión Horquillas, manguetas, mazas, amortiguadores, rótulas, bujes y $49,000 fabricación. Ruedas y Cuatro rines, cuatro neumáticos y accesorios de montaje.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":154,\"augmented\":1,\"text\":\"• La capacidad de carga de 200 a 300 kg se considera adicional al peso de los dos ocupantes. • La proyección contempla un sistema de tracción trasera 4x2.\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":201,\"augmented\":0,\"text\":\"Aztlán evoca el punto de origen desde el que una marca mexicana sale a recorrer terrenos extremos. Paleta de negro carbón, rojo, naranja y gris arena; tipografía robusta y de lectura rápida. · Página 1\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":574,\"augmented\":0,\"text\":\"Dado nuestro volumen de producción de nicho (280 unidades\\/año), Aztlán no compite por volumen de compra, sino mediante un Plan de Mercadotecnia B2B de Proveeduría. Estructuramos alianzas de Co-Branding y visibilidad de marca con proveedores Tier 1 para adquirir materiales de máxima resistencia a costos rentables (precios OEM): • Materiales Mejorados vía Co-Branding: Negociamos contratos de suministro con Isuzu (tren motriz), Fox (suspensión), Visteon (interfaz digital) y Brembo (frenos) ofreciéndoles exposición directa en los complejos turísticos de nuestros clientes.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":557,\"augmented\":0,\"text\":\"• Mecánica propia: diseño y maquinado de brazos de suspensión, masas, barras estabilizadoras, chasis tubulares, semiejes y ejes cardán. • Electrónica avanzada: diseño, programación y fabricación de arneses eléctricos y sistemas de iluminación. Las interfaces del tablero se integran de Visteon. • Venta a terceros: además de abastecer a la flota Aztlán, la división vende componentes y refacciones a talleres off-road, equipos de competencia y otros armadores, esto representa aproximadamente el 58 % de los ingresos de componentes, refacciones y servicio).\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":180,\"augmented\":0,\"text\":\"Esta capacidad instalada reduce los costos de los componentes estructurales en un 35% frente a la compra externa, garantizando tolerancias perfectas para trabajo pesado. · Página 3\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":336,\"augmented\":1,\"text\":\"Para evitar los altos costos y tiempos de la tercerización de piezas complejas, Aztlán mercadea sus contratos B2B (pago por hitos) para apalancar líneas de crédito industrial. Esto nos permite adquirir e integrar tecnologías de grado aeroespacial en nuestra planta (Centros de Mecanizado CNC Haas VF-2 y equipos Miller Dynasty TIG\\/MIG).\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":328,\"augmented\":1,\"text\":\"Dado nuestro volumen de producción de nicho (280 unidades\\/año), Aztlán no compite por volumen de compra, sino mediante un Plan de Mercadotecnia B2B de Proveeduría. Estructuramos alianzas de Co-Branding y visibilidad de marca con proveedores Tier 1 para adquirir materiales de máxima resistencia a costos rentables (precios OEM):\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":327,\"augmented\":1,\"text\":\"• Materiales Mejorados vía Co-Branding: Negociamos contratos de suministro con Isuzu (tren motriz), Fox (suspensión), Visteon (interfaz digital) y Brembo (frenos) ofreciéndoles exposición directa en los complejos turísticos de nuestros clientes. El AZ 3 funciona como un escaparate tecnológico activo ante los huéspedes de OAH.\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":646,\"augmented\":0,\"text\":\"El AZ 3 funciona como un escaparate tecnológico activo ante los huéspedes de OAH. A cambio de esta visibilidad premium, obtenemos precios de volumen que reducen el costo unitario del vehículo en un 18%, logrando una plataforma robusta pero altamente rentable. • Adquisición de Manufactura Avanzada (In-House): Para evitar los altos costos y tiempos de la tercerización de piezas complejas, Aztlán mercadea sus contratos B2B (pago por hitos) para apalancar líneas de crédito industrial. Esto nos permite adquirir e integrar tecnologías de grado aeroespacial en nuestra planta (Centros de Mecanizado CNC Haas VF-2 y equipos Miller Dynasty TIG\\/MIG).\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":463,\"augmented\":0,\"text\":\"auditoría de proveedores críticos, inspección de recepción por lote e indicadores de entregas a tiempo y piezas defectuosas por millón (PPM). • Las 5S en planta: Seiri (clasificar piezas específicas por pedido), Seiton (zonas y flujos para componentes grandes de proveedores), Seiso (planta limpia y libre de derrames, también argumento de venta ante clientes B2B), Seiketsu (procedimiento estandarizado por estación) y Shitsuke (auditorías semanales). · Página 4\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":631,\"augmented\":0,\"text\":\"• Manufactura mixta: Build-to-Order para configuraciones personalizadas y programas de flotilla con orden de compra marco, producción programada y pago por hitos. El AZ 3 se mantiene en 15 unidades por trimestre (60 al año en curso): volumen controlado deliberadamente para personalizar y probar cada unidad antes de entregarla. • Componentes críticos importados (motor, transmisión, sist. de frenos, suspensión e interfaz de tablero): plazo de 8 semanas e inventario de seguridad equivalente al 5 % por flota, con una parte en consigna en el sitio del cliente (B2B). • Refacciones de carrocería, mecánica secundaria y electrónica:\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":235,\"augmented\":0,\"text\":\"stock central en Pachuca con envío nacional en 24 a 72 horas y reposición diaria. • Flota de demostración: 6 unidades propias (2 AZ 2, 2 AZ 3 y 2 AZ 1), registradas como activo fijo y disponibles para pruebas de manejo de clientes B2B.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":328,\"augmented\":1,\"text\":\"• Manufactura mixta: Build-to-Order para configuraciones personalizadas y programas de flotilla con orden de compra marco, producción programada y pago por hitos. El AZ 3 se mantiene en 15 unidades por trimestre (60 al año en curso): volumen controlado deliberadamente para personalizar y probar cada unidad antes de entregarla.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":282,\"augmented\":1,\"text\":\"Ingeniería adaptativa con configuraciones estandarizadas y respaldo absoluto en sitio. A diferencia de las marcas de producción en masa, Aztlán configura cada vehículo según el uso real del cliente: recreación de fin de semana, competencia, trabajo agrícola, industrial o turístico.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":161,\"augmented\":1,\"text\":\"auditoría de proveedores críticos, inspección de recepción por lote e indicadores de entregas a tiempo y piezas defectuosas por millón (PPM). • Las 5S en planta:\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":333,\"augmented\":1,\"text\":\"La personalización se entrega mediante paquetes predefinidos sobre una plataforma común, lo que mantiene bajos los costos y los plazos y conserva refacciones comunes entre unidades. El respaldo incluye mantenimiento preventivo programado, técnicos en sitio para flotillas y una disponibilidad objetivo de al menos 95 % bajo contrato.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":616,\"augmented\":0,\"text\":\"Ingeniería adaptativa con configuraciones estandarizadas y respaldo absoluto en sitio. A diferencia de las marcas de producción en masa, Aztlán configura cada vehículo según el uso real del cliente: recreación de fin de semana, competencia, trabajo agrícola, industrial o turístico. La personalización se entrega mediante paquetes predefinidos sobre una plataforma común, lo que mantiene bajos los costos y los plazos y conserva refacciones comunes entre unidades. El respaldo incluye mantenimiento preventivo programado, técnicos en sitio para flotillas y una disponibilidad objetivo de al menos 95 % bajo contrato.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":541,\"augmented\":0,\"text\":\"• Sistema de calidad: 5S como base de la cultura de planta y certificación ISO 9001:2015 programada para el 2S 2027. • Prueba de aceptación al 100 %: inspección dimensional y de soldadura de chasis, prueba de frenos, calibración electrónica y 30 km de ruta en pista interna. • Validación de diseño: bancos de fatiga de chasis y suspensión, pruebas de vadeo y de niebla salina (ASTM B117) para recubrimientos, con un protocolo específico para clima cálido-húmedo y ambiente salino antes de entregar la flota de OAH. • Calidad con proveedores:\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":232,\"augmented\":0,\"text\":\"El AZ 3 es un UTV de máxima resistencia, configurado con un propósito doble: operar como maquinaria de trabajo pesado durante la fase de construcción y como vehículo recreativo de alto rendimiento en la etapa de operación turística.\"},{\"section\":\"segmentacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":113,\"augmented\":0,\"text\":\"confiabilidad operativa, resistencia bajo carga constante y pólizas de mantenimiento preventivo que eviten paros.\"},{\"section\":\"segmentacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":557,\"augmented\":0,\"text\":\"• B2C, mercado entusiasta. Pilotos amateur de rutas off-road, aventureros de fin de semana y entusiastas de los deportes de motor. Se interesan en el AZ 1 y el AZ 3 por su agilidad, diseño agresivo y personalización de suspensión y mapeo, y valoran el trato directo y el servicio posventa. • B2B, mercado utilitario y corporativo. Agroindustrias, minería, constructoras en zonas remotas, parques de ecoturismo, desarrollos hoteleros y turísticos, y corporaciones de seguridad privada o rescate. Se interesan en el AZ 2 y el AZ 3 como herramienta de trabajo:\"},{\"section\":\"segmentacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":330,\"augmented\":1,\"text\":\"• B2C, mercado entusiasta. Pilotos amateur de rutas off-road, aventureros de fin de semana y entusiastas de los deportes de motor. Se interesan en el AZ 1 y el AZ 3 por su agilidad, diseño agresivo y personalización de suspensión y mapeo, y valoran el trato directo y el servicio posventa. • B2B, mercado utilitario y corporativo.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":103,\"augmented\":0,\"text\":\"un programa de primer empleo con -IPN incorpora cada año a egresados del AZTLÁN RACING TEAM. · Página 6\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":180,\"augmented\":1,\"text\":\"4 personas con capacidades diferentes operan dentro de las diferentes áreas; un programa de primer empleo con -IPN incorpora cada año a egresados del AZTLÁN RACING TEAM. · Página 6\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":331,\"augmented\":1,\"text\":\"Gerencia de Calidad y Pruebas Prueba de aceptación, validación de diseño y auditoría de 6 proveedores. Gerencia de Cadena de Compras, importaciones, inventario de seguridad y consigna. 6 Suministro y Logística Gerencia Comercial (B2B y Flotillas corporativas (ventas, contratos y pruebas de manejo) 8 B2C) y atención a entusiastas.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":629,\"augmented\":0,\"text\":\"Área Función Personas Dirección General (CEO y Visión a largo plazo y relaciones con proveedores clave y 2 COO) clientes estratégicos. Fabricación, ingeniería operativa, cadena de suministro, calidad y entregas. Dirección de Ingeniería e I+D Diseño CAD, nuevos chasis, evolución del software de 14 mapeo, pruebas de resistencia y transferencia desde el prototipo Baja SAE. Gerencia de Producción y Línea de ensamble (45 personas) y maquinado, soldadura de 67 Manufactura chasis y módulos electrónicos (22 personas); aplica 5S. Gerencia de Calidad y Pruebas Prueba de aceptación, validación de diseño y auditoría de 6 proveedores.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":643,\"augmented\":0,\"text\":\"Gerencia de Cadena de Compras, importaciones, inventario de seguridad y consigna. 6 Suministro y Logística Gerencia Comercial (B2B y Flotillas corporativas (ventas, contratos y pruebas de manejo) 8 B2C) y atención a entusiastas. Gerencia de Soporte y Mantenimiento, upgrades, contratos de disponibilidad y Base 12 Posventa Sureste (+6 técnicos a partir de 2027). Administración y Finanzas Control de costos, financiamiento y capital de trabajo. 5 Total 120 Diversidad e inclusión: 32 % de mujeres (38 de 120) y 29 % en ingeniería (4 de 14); edad promedio de 31 años; 4 personas con capacidades diferentes operan dentro de las diferentes áreas;\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":260,\"augmented\":0,\"text\":\"aporta 45 % de los ingresos y 59 % de la utilidad bruta. La rentabilidad de Aztlán no depende de las fallas de sus unidades: los contratos B2B se pactan con disponibilidad objetivo y costo por hora previsible, de modo que la empresa gana cuando la flota opera.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":135,\"augmented\":1,\"text\":\"los contratos B2B se pactan con disponibilidad objetivo y costo por hora previsible, de modo que la empresa gana cuando la flota opera.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":647,\"augmented\":0,\"text\":\"• Venta de vehículos (producción programada y preventa por hitos): B2C con anticipo de 50 %; flotillas B2B con pago por hitos (30 % al firmar, 40 % al avance de producción y 30 % contra entrega) y arrendamiento con opción a compra. El anticipo financia la compra de componentes externos (Isuzu, AISIN, Fox, Visteon,Brembo) sin inmovilizar inventario de unidades terminadas; las unidades de demostración se registran como activo fijo. • Componentes, refacciones y servicio: componentes y refacciones de manufactura propia vendidos a terceros y a la flota Aztlán, más contratos de mantenimiento, upgrades y afinación. Es la división de mayor margen:\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":130,\"augmented\":0,\"text\":\"Vehículo Utilitario Todoterreno (Baja Utility Diesel) - Configuración Ecoturismo\\/Táctica para Entornos Tropicales (Selva y Playa).\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":583,\"augmented\":0,\"text\":\"Para garantizar una producción continua y rentable sin detener la planta por cambios de vehículo, las instalaciones operan bajo un Sistema de Manufactura Flexible (FMS): • 25% Línea de Ensamblaje de Modelo Mixto: Corazón de la planta donde conviven los tres modelos sobre una línea transportadora principal, alimentada por bahías laterales de sub-ensamble (matrimonio del tren motriz). • 20% Fabricación y Mecanizado: Zonas de corte láser, doblado y mecanizado en Islas CNC que operan por lotes para abastecer de componentes estructurales a toda la planta. • 18% Soldadura de Chasis:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":595,\"augmented\":0,\"text\":\"Estaciones con fijaciones modulares (jigs intercambiables) de rápida reconfiguración y área de amortiguamiento (buffer) de chasis terminados. · Página 9 • 15% Ingeniería, I+D y Corporativo: Oficinas de diseño CAD, taller aislado para prototipado y Showroom B2B. • 12% Logística Dinámica (Kitting): Almacén Just-In-Time (JIT) donde se preparan carritos (kits) con las piezas exactas para el modelo en turno, previniendo errores en la línea mixta. • 10% Inspección, Pruebas y Posventa: Área de alineación, programación de la ECU, control dinámico de calidad (PDI) y empaque central de refacciones.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":10,\"char_len\":499,\"augmented\":0,\"text\":\"Para garantizar el 95% de disponibilidad durante la fase de obra y operación en Yucatán sin comprometer la reserva ecológica, Aztlán sugiere la instalación de un In-Site Hub modular dentro de la propiedad de OAH: • Almacenamiento de unidades: Diseño de cobertizos modulares de estructura ligera con techumbre de lona tensada. Protegen a los vehículos de la radiación UV, lluvias tropicales y salinidad, garantizando una ventilación cruzada que previene la corrosión. • Abastecimiento de combustible:\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":10,\"char_len\":636,\"augmented\":0,\"text\":\"Instalación de un tanque superficial de autoconsumo para diésel (norma NOM-006-ASEA-2017) de 5,000 litros, equipado con dique de contención de derrames al 110% de su capacidad. Esto elimina la necesidad de viajes a gasolineras externas, reduce tiempos muertos y protege el suelo de la reserva. • Mantenimiento y Refacciones (Base Sureste): Implementación de un taller móvil en contenedores marítimos modificados de 40 pies. Uno fungirá como bahía de servicio preventivo y el otro como almacén climatizado para resguardar la electrónica y componentes críticos (inventario de seguridad del 5%), operado por nuestros 6 técnicos residentes.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":10,\"char_len\":246,\"augmented\":1,\"text\":\"Esto elimina la necesidad de viajes a gasolineras externas, reduce tiempos muertos y protege el suelo de la reserva. • Mantenimiento y Refacciones (Base Sureste): Implementación de un taller móvil en contenedores marítimos modificados de 40 pies.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":10,\"char_len\":350,\"augmented\":1,\"text\":\"Protegen a los vehículos de la radiación UV, lluvias tropicales y salinidad, garantizando una ventilación cruzada que previene la corrosión. • Abastecimiento de combustible: Instalación de un tanque superficial de autoconsumo para diésel (norma NOM-006-ASEA-2017) de 5,000 litros, equipado con dique de contención de derrames al 110% de su capacidad.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":308,\"augmented\":0,\"text\":\"Comprar tubos de acero AISI 1018 con medidas y espesores adecuados para el chasis. Evaluar aceros de mayor resistencia únicamente en las piezas donde realmente se necesiten. Corte láser o CNC para obtener piezas con medidas precisas. Diseño CAD y simulación CAE para revisar la resistencia antes de fabricar.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":316,\"augmented\":0,\"text\":\"Solicitar al menos tres cotizaciones por material o componente. Comparar precio, calidad, tiempo de entrega y costo de transporte. Pedir certificados de material cuando corresponda. Organizar las áreas de corte, soldadura, ensamble y revisión. Fabricar lotes de piezas similares para aprovechar el tiempo de trabajo.\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":249,\"augmented\":0,\"text\":\"| Propuesta comercial B2B | Baja monoplaza Destinatario principal: Outdoor Adventure Holdings — cliente comprador que evalúa ofertas de distintos proveedores. Octubre de 2026 • Material comercial propuesto, sujeto a validación técnica y contractual.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":324,\"augmented\":0,\"text\":\"BAJA MONOPLAZA | Cuando el terreno exige una solución de movilidad individual, propone un vehículo Baja monoplaza desarrollado para desplazamientos en áreas de trabajo e inspección fuera del pavimento. Nuestro enfoque combina diseño mecánico, producción bajo pedido y una relación directa entre el comprador y el fabricante.\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":388,\"augmented\":0,\"text\":\"del fabricante. Responsables de compras, operaciones, Público principal mantenimiento y proyectos de campo. Una solución Baja monoplaza que debe demostrar Mensaje central desempeño, facilidad de servicio y viabilidad económica mediante pruebas. Técnico, directo, confiable y transparente; sin Tono de comunicación promesas no verificadas. Lema propuesto | Movilidad para llegar más lejos.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":420,\"augmented\":0,\"text\":\"Asunto: Propuesta de suministro de vehículos Baja monoplaza | Estimado equipo de compras de Outdoor Adventure Holdings: Nos gustaría presentar una propuesta de suministro de vehículos Baja monoplaza para las necesidades de movilidad individual del proyecto. Compartimos información técnica preliminar, el esquema de entregas propuesto y nuestra disposición a realizar pruebas de desempeño antes de la decisión de compra.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":583,\"augmented\":0,\"text\":\"Publicación corporativa para LinkedIn En desarrollamos una plataforma Baja monoplaza orientada a necesidades de movilidad individual en campo. Estamos trabajando en la validación de su desempeño y en la preparación de un proceso de fabricación por lotes para clientes empresariales. Nuestra prioridad: presentar especificaciones claras, pruebas verificables y una oferta de servicio que acompañe la operación. Texto corto para banner BAJA MONOPLAZA Movilidad individual para terrenos exigentes Diseño • Fabricación bajo pedido • Soporte técnico Solicite ficha técnica y demostración.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":257,\"augmented\":1,\"text\":\"Asunto: Propuesta de suministro de vehículos Baja monoplaza | Estimado equipo de compras de Outdoor Adventure Holdings: Nos gustaría presentar una propuesta de suministro de vehículos Baja monoplaza para las necesidades de movilidad individual del proyecto.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":584,\"augmented\":0,\"text\":\"Producto Baja monoplaza, versión de evaluación Traslado individual e inspección de campo; no se Aplicación prevista ofrece como vehículo de carga Motor Kohler CH440, 14 HP, gasolina Suspensión Independiente de doble horquilla Dimensiones 108 in largo × 64 in ancho Estado Prototipo en fabricación; pendiente de pruebas MXN $170,000 por unidad, antes de impuestos, Precio indicativo exportación y servicios adicionales 70 unidades: 20 en el mes 6 y 50 adicionales en el mes Pedido potencial 18, conforme al calendario del comprador Sujeto a acuerdo comercial; alcance por definir antes\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":276,\"augmented\":0,\"text\":\"Una plataforma UTV modular y durable, adaptada al terreno kárstico y al ambiente salino de Yucatán, con componentes comunes, mantenimiento en sitio y configuraciones intercambiables. La propuesta busca reducir tiempos de inactividad y simplificar el inventario de refacciones.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":263,\"augmented\":0,\"text\":\"La venta y el soporte de las 70 unidades se realizan bajo un esquema B2B. Se complementa con habilitación B2B2C: el proveedor entrega configuraciones, especificaciones de seguridad y protocolos para que el cliente opere recorridos turísticos con usuarios finales.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":584,\"augmented\":0,\"text\":\"Cliente objetivo: Outdoor Adventure Holdings, operador de un desarrollo turístico en Yucatán. El proyecto requiere una flota de 70 vehículos utilitarios todoterreno (UTV) para atender dos etapas: construcción del complejo y operación posterior. Etapa Flota Uso principal Condiciones del entorno Construcción 20 Traslado de cuadrillas, supervisión y Brechas, suelo kárstico y roca caliza (entrega mes 6) unidades carga ligera abrasiva. Operación (entrega 50 Mantenimiento, patrullaje ecológico, Salinidad, humedad, senderos estrechos, mes 18) unidades emergencias y tours arena y lodo.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":244,\"augmented\":1,\"text\":\"Cliente objetivo: Outdoor Adventure Holdings, operador de un desarrollo turístico en Yucatán. El proyecto requiere una flota de 70 vehículos utilitarios todoterreno (UTV) para atender dos etapas: construcción del complejo y operación posterior.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":466,\"augmented\":0,\"text\":\"Paquete Duración \\/ formato Tarifa estimada Enfoque Eco-Trail Discovery 1.5 h, recorrido guiado USD 95 por Observación de flora y fauna, velocidad en convoy persona controlada y paradas fotográficas. Technical Off-Road 2 h, circuito con USD 145 por Roca kárstica, grava y pendientes controladas; Challenge obstáculos persona instructor copiloto. Sunset Coastal 2.5 h, tour privado USD 210 por Recorrido escénico al atardecer para huéspedes Expedition persona premium.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":451,\"augmented\":0,\"text\":\"La solución utiliza una plataforma vehicular única que comparte aproximadamente el 85% de sus componentes. Esta estandarización reduce la complejidad del inventario y facilita el mantenimiento. Se ofrecen dos configuraciones: Xolotl Página 2 de 10 Configuración Cantidad Aplicación C-20 20 unidades Trabajo pesado y construcción: protección inferior y caja de servicio. O-50 50 unidades Operación del resort, mantenimiento, patrullaje y tours guiados.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":329,\"augmented\":1,\"text\":\"La solución utiliza una plataforma vehicular única que comparte aproximadamente el 85% de sus componentes. Esta estandarización reduce la complejidad del inventario y facilita el mantenimiento. Se ofrecen dos configuraciones: Xolotl Página 2 de 10 Configuración Cantidad Aplicación C-20 20 unidades Trabajo pesado y construcción:\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":121,\"augmented\":1,\"text\":\"Sunset Coastal 2.5 h, tour privado USD 210 por Recorrido escénico al atardecer para huéspedes Expedition persona premium.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":121,\"augmented\":1,\"text\":\"protección inferior y caja de servicio. O-50 50 unidades Operación del resort, mantenimiento, patrullaje y tours guiados.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":410,\"augmented\":0,\"text\":\"Frenos Discos hidráulicos en las cuatro ruedas. Frenado y control en condiciones de lodo. Seguridad Arnés de cinco puntos y jaula antivuelco. Retención de ocupantes y protección estructural. Sistema eléctrico Instalación a 12 V con conectores sellados. Protección frente a agua, lodo y ambiente salino. Acústica Silenciador secundario y supresor de Reducción del impacto acústico y control de riesgos. chispas.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":331,\"augmented\":1,\"text\":\"Tren motriz Motor monocilíndrico 4T de 10 HP, arranque Potencia para tareas utilitarias y uso de gasolina eléctrico. comercial. Transmisión CVT con caja reductora; High, Low, Neutro y Adaptación a traslado, arrastre y maniobras. Reversa. Suspensión Doble horquilla independiente; recorrido Absorción de irregularidades del terreno.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":107,\"augmented\":1,\"text\":\"Acústica Silenciador secundario y supresor de Reducción del impacto acústico y control de riesgos. chispas.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":647,\"augmented\":0,\"text\":\"Sistema Especificación resumida Justificación funcional Chasis Acero tubular ASTM A513 \\/ AISI 1018. Estructura primaria y soporte de jaula antivuelco. Protección Placa desmontable de aluminio 5052-H32, Protección contra impactos de roca caliza y acceso inferior 3\\/16”. para mantenimiento. Tren motriz Motor monocilíndrico 4T de 10 HP, arranque Potencia para tareas utilitarias y uso de gasolina eléctrico. comercial. Transmisión CVT con caja reductora; High, Low, Neutro y Adaptación a traslado, arrastre y maniobras. Reversa. Suspensión Doble horquilla independiente; recorrido Absorción de irregularidades del terreno. indicado de 9.5” (241 mm).\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":577,\"augmented\":0,\"text\":\"Riesgo Impacto Medida preventiva Aumento del precio Presión sobre el margen del Pactar precios y cobertura con proveedores desde la primera del acero y aluminio segundo lote. orden. Retraso del lote 1 El cliente tendría que prolongar Fabricar dos prototipos con anticipación y preensamblar la renta de vehículos. subchasis. Baja demanda Menor retorno esperado de los Retirar módulos turísticos y reasignar unidades a turística circuitos. mantenimiento. Desgaste por Más garantías y tiempo de paro. Inspecciones programadas y mantenimiento preventivo. salinidad y polvo kárstico\"},{\"section\":\"competencia\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":644,\"augmented\":0,\"text\":\"Criterio Propuesta modular Beneficio para el cliente TCO y Consumibles comerciales y piezas Menor dependencia de refacciones propietarias e refacciones estandarizadas. importadas. Modularidad Chasis común con módulos para carga, servicio Una flota adaptable a distintas etapas y actividades. o turismo. Mantenimiento Acceso directo a filtro, banda CVT y dirección. Reparaciones rápidas y menor tiempo fuera de servicio. Adaptación al Protección inferior y tratamiento anticorrosivo. Mayor resistencia a abrasión, humedad y salinidad. entorno Soporte Pruebas previas y asistencia técnica local. Validación antes de la compra y atención en sitio.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":253,\"augmented\":1,\"text\":\"Baja demanda Menor retorno esperado de los Retirar módulos turísticos y reasignar unidades a turística circuitos. mantenimiento. Desgaste por Más garantías y tiempo de paro. Inspecciones programadas y mantenimiento preventivo. salinidad y polvo kárstico\"},{\"section\":\"competencia\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":341,\"augmented\":1,\"text\":\"Mantenimiento Acceso directo a filtro, banda CVT y dirección. Reparaciones rápidas y menor tiempo fuera de servicio. Adaptación al Protección inferior y tratamiento anticorrosivo. Mayor resistencia a abrasión, humedad y salinidad. entorno Soporte Pruebas previas y asistencia técnica local. Validación antes de la compra y atención en sitio.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":191,\"augmented\":0,\"text\":\"La estrategia concentra internamente los procesos que definen la estructura y geometría del vehículo, y subcontrata operaciones especializadas o componentes comerciales. Xolotl Página 5 de 10\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":288,\"augmented\":0,\"text\":\"Se propone manufactura esbelta (Lean Manufacturing) con flujo celular para limitar inventarios ociosos y cumplir las fechas de entrega. Parámetro Lote 1 Lote 2 Unidades 20 50 Periodo de fabricación Meses 2 a 5 Meses 7 a 16 Entrega Mes 6 Mes 18 Ritmo base 5 unidades \\/ mes 5 unidades \\/ mes\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":333,\"augmented\":1,\"text\":\"Componente Decisión Proceso \\/ control Chasis tubular Fabricar Corte, perfilado y soldadura en plantillas; inspección internamente dimensional y líquidos penetrantes. Brazos de suspensión Fabricar Soldadura en dispositivos; control geométrico y alineación. internamente Placas skid plate Comprar \\/ maquila Corte por agua y doblez CNC;\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":560,\"augmented\":0,\"text\":\"Componente Decisión Proceso \\/ control Chasis tubular Fabricar Corte, perfilado y soldadura en plantillas; inspección internamente dimensional y líquidos penetrantes. Brazos de suspensión Fabricar Soldadura en dispositivos; control geométrico y alineación. internamente Placas skid plate Comprar \\/ maquila Corte por agua y doblez CNC; revisión de barrenos y fisuras. Motor y CVT Comprar comercial Adquisición de ensambles probados; prueba de arranque y ajuste. Caja reductora Fabricación y Maquinado, engranes comerciales y control de holguras y ensamble fugas.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":245,\"augmented\":0,\"text\":\"• Un turno de 8 horas al día, cinco días por semana; se consideran 160 horas efectivas al mes. • Tiempo directo de ensamble final: 32 horas por unidad. • La producción del primer lote inicia después de las compras y preparación de herramentales.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":407,\"augmented\":0,\"text\":\"Suspensión, Montaje de brazos, Racks, torquímetros, prensa para bujes y 6 h dirección y amortiguadores, dirección y purgado. frenos frenos. 4. Tren motriz y Instalación de motor, CVT, Polipasto, torquímetros y alineador de 7 h eléctrico reductora, tanque y arnés. poleas. 5. Inspección Alineación, revisión de torques, Alineador, banco de rodillos y checklist 5 h final electricidad y encendido. de calidad.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":612,\"augmented\":0,\"text\":\"Se plantea una nave de 450 m² con celda de manufactura en forma de U. La distribución busca reducir traslados internos y facilitar la supervisión visual mediante prácticas 5S y tableros Kanban. Estación Actividad principal Equipo \\/ control Tiempo por unidad 1. Corte y Corte, entallado y doblado de Sierra cinta, fresadora, dobladora y 5 h habilitado tubos. calibradores. 2. Fixturing y Montaje en jig, punteado y Plantilla maestra, soldadoras MIG\\/TIG y 9 h soldadura soldadura del chasis. mesas de trabajo. Proceso externo Pintura electrostática horneada. Proveedor regional especializado. 48 h por lote de 5 3.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":132,\"augmented\":1,\"text\":\"Inspección Alineación, revisión de torques, Alineador, banco de rodillos y checklist 5 h final electricidad y encendido. de calidad.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":412,\"augmented\":0,\"text\":\"Prueba Criterio indicado en el plan Estructural Sin deformación plástica de la jaula bajo carga vertical de 3G y lateral de 1.5G. Frenado Detención desde 40 km\\/h en menos de 6.5 m sobre superficie kárstica húmeda. Durabilidad 200 horas de ciclo continuo; sin holguras ni fisuras en soldaduras de horquillas. Ride & Drive 50 horas de operación en selva baja y costa, con evaluación de ergonomía y maniobrabilidad.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":307,\"augmented\":1,\"text\":\"Prueba Criterio indicado en el plan Estructural Sin deformación plástica de la jaula bajo carga vertical de 3G y lateral de 1.5G. Frenado Detención desde 40 km\\/h en menos de 6.5 m sobre superficie kárstica húmeda. Durabilidad 200 horas de ciclo continuo; sin holguras ni fisuras en soldaduras de horquillas.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":599,\"augmented\":0,\"text\":\"Concepto Componentes incluidos Costo por unidad Materiales directos Tubería, placas de aluminio, soldadura y soportería. USD 850 Tren motriz y Motor 10 HP, CVT, reductora, rodamientos y tanque. USD 1,650 mecánica Suspensión, dirección Amortiguadores, cálipers, discos, rines y llantas. USD 1,100 y frenos Seguridad, eléctrico y Arnés, cableado, iluminación e instrumentación. USD 450 módulos Procesos Corte CNC y pintura electrostática. USD 220 subcontratados Mano de obra directa 32 horas a USD 12.50 por hora. USD 400 Total COGS Costo directo estimado por vehículo. USD 4,670 Xolotl Página 7 de 10\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":539,\"augmented\":0,\"text\":\"Xolotl Página 8 de 10 Puesto Cantidad \\/ ubicación Responsabilidad Técnico soldador\\/fabricador 1 \\/ Planta Corte, perfilado, jigs y soldadura de chasis y horquillas. Técnico mecánico de 1 \\/ Planta Motor, CVT, reductora, suspensión, dirección y frenos. ensamble Técnico eléctrico y calidad 1 \\/ Planta Arnés, instrumentación, pruebas y liberación EOLT. Ingeniero residente de campo 1 \\/ Yucatán Relación con el cliente, garantías, refacciones y pruebas. Técnico mecánico de flotilla 1 \\/ Yucatán Mantenimiento preventivo y reparaciones en sitio.\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":293,\"augmented\":0,\"text\":\"Indicador Proyección Costo de manufactura de 70 unidades USD 326,900 Margen bruto acumulado USD 220,100 CapEx + OpEx de 18 meses USD 97,140 Utilidad neta proyectada USD 122,960 Punto de equilibrio reportado 31 unidades (aprox. mes 9) Ahorro estimado para el cliente en fase de obra USD 136,000\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":156,\"augmented\":0,\"text\":\"Lote Unidades Precio unitario Ingresos C-20 \\/ Construcción 20 USD 7,600 USD 152,000 O-50 \\/ Operación y tours 50 USD 7,900 USD 395,000 Total 70 — USD 547,000\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":385,\"augmented\":0,\"text\":\"Rubro Importe Jigs y herramentales USD 12,000 Equipamiento de taller USD 14,000 Equipo de inspección y calidad USD 6,000 Dos prototipos USD 9,340 Inversión inicial (CapEx) USD 41,340 Gasto fijo mensual USD 3,100 Gasto fijo acumulado en 18 meses USD 55,800 El gasto fijo mensual contempla renta y servicios de la nave, mantenimiento de herramentales, calibraciones, seguros y licencias.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":349,\"augmented\":1,\"text\":\"Rubro Importe Jigs y herramentales USD 12,000 Equipamiento de taller USD 14,000 Equipo de inspección y calidad USD 6,000 Dos prototipos USD 9,340 Inversión inicial (CapEx) USD 41,340 Gasto fijo mensual USD 3,100 Gasto fijo acumulado en 18 meses USD 55,800 El gasto fijo mensual contempla renta y servicios de la nave, mantenimiento de herramentales,\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":96,\"augmented\":1,\"text\":\"flora protegida. instructor copiloto. $210 USD \\/ pasajero $95 USD $145 USD \\/ pasajero \\/ pasajero\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":466,\"augmented\":0,\"text\":\"Monetización del Terreno: Circuitos y Tours UTV EXPERIENCIA PANORÁMICA EXPERIENCIA DINÁMICA EXPERIENCIA VIP Eco-Trail Discovery Off-Road Challenge Sunset Expedition Recorrido guiado de 1.5 horas Circuito técnico de 2.0 horas con Tour privado en dunas de 2.5 horas enfocado en avistamiento ecológico y pendientes, banco de grava e hacia la infraestructura frente al mar. flora protegida. instructor copiloto. $210 USD \\/ pasajero $95 USD $145 USD \\/ pasajero \\/ pasajero\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":600,\"augmented\":0,\"text\":\"Línea Celular en Flujo \\\"U\\\" (Nave de 450 m²) ESTACIÓN 1 • 5.0 H ESTACIÓN 2 • 9.0 H ESTACIÓN 3 • 6.0 H ESTACIÓN 4 • 7.0 H ESTACIÓN 5 • 5.0 H Corte y Habilitado Fixturing y Soldadura Suspensión y Frenos Powertrain y Blindaje Pruebas EOLT Corte en sierra cinta y Horquillas, Banco de rodillos doblado hidráulico Jig dimensional rígido. amortiguadores y Alineación láser de dinámico, alineación rotativo de tubos Cordones purgado neumático CVT, fijación de motor láser y checklist de ASTM A513. GMAW\\/GTAW con de frenos al vacío. 10 HP y skid plates de calidad digital. inspección de aluminio. penetración.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":82,\"augmented\":0,\"text\":\"Ride & Drive en Yucatán y Protocolo Ecológico Ride & Drive (Mes 2 en Terreno Real)\"},{\"section\":\"competencia\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":646,\"augmented\":0,\"text\":\"UTVs Comerciales Impacto en Outdoor Criterio Evaluado Nuestra Plataforma Baja SAE UTVs Comerciales (Polaris \\/ Can-Am) Adventure Consumibles estándar y piezas Refacciones de importación y software Ahorro neto >25% en la TCO a 3 Años nacionales cerrado flota Chasis base unificado al 85% Mismo stock de Modularidad de Flota Modelos distintos para carga y turismo común refacciones Reemplazo de desgaste <20 min en Días de espera en talleres Cero retrasos en fase de Tiempo de Inactividad campo concesionarios obra Ambiente Skid plates aluminio 5052 y sellado Corrosión y abrasión prematura de Mayor vida útil sin Marino\\/Karst 12V bajos depreciación\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":101,\"augmented\":0,\"text\":\"Validación y Soporte Garantías limitadas sujetas a agenda Compra validada sin riesgo técnicos in situ\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":9,\"char_len\":410,\"augmented\":0,\"text\":\"Estructura de Costos, Precios y Margen Operativo Fase \\/ Lote Unidades Costo Unitario (COGS) Precio Venta Unitario Margen Bruto Ingreso Total Lote 1 (Construcción C- 20 unidades $4,670 USD $7,600 USD 38.5% ($2,930) $152,000 USD Lote 2 (Operación O-50) 50 unidades $4,670 USD $7,900 USD 40.8% ($3,230) $395,000 USD Inversión CapEx: OpEx Fijo (18 m): Total Flota Contratada 70 unidades 40.1% promedio $547,000 USD\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":9,\"char_len\":347,\"augmented\":1,\"text\":\"Estructura de Costos, Precios y Margen Operativo Fase \\/ Lote Unidades Costo Unitario (COGS) Precio Venta Unitario Margen Bruto Ingreso Total Lote 1 (Construcción C- 20 unidades $4,670 USD $7,600 USD 38.5% ($2,930) $152,000 USD Lote 2 (Operación O-50) 50 unidades $4,670 USD $7,900 USD 40.8% ($3,230) $395,000 USD Inversión CapEx: OpEx Fijo (18 m):\"},{\"section\":\"mercado\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":124,\"augmented\":0,\"text\":\"individual en terrenos irregulares y dentro de áreas de trabajo. Outdoor Adventure Holdings; pedido potencial de 20 unidades\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":255,\"augmented\":0,\"text\":\"al mes 6 y 50 adicionales al mes 18. Personal autorizado de inspección, supervisión, vigilancia y Usuario final recorridos operativos. Producto Vehículo Baja monoplaza Fabricación propuesta en Saltillo Coahuila e importación a Cobertura geográfica Yucatán\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":193,\"augmented\":0,\"text\":\"a adjudicación. Posventa Garantía escrita, inventario mínimo, manuales y capacitación. Estimar combustible, refacciones, mantenimiento y horas Costo operativo improductivas durante varios años.\"},{\"section\":\"mercado\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":167,\"augmented\":0,\"text\":\"B2B: venta directa a una empresa compradora mediante Modelo de relación evaluación competitiva de ofertas. Operadores de instalaciones extensas que requieren movilidad\"},{\"section\":\"mercado\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":167,\"augmented\":0,\"text\":\"| Baja monoplaza | Oferta competitiva para Outdoor Adventure Holdings Fecha: octubre de 2026 • Documento complementario del plan de negocios y proyecciones financieras\"},{\"section\":\"segmentacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":616,\"augmented\":0,\"text\":\"Proyectos de construcción en predios Inspección visual, recorridos individuales Alta extensos y supervisión sin transporte voluminoso. Patrullaje y recorridos operativos de Complejos turísticos y recreativos Alta personal autorizado. Instalaciones industriales o de Inspección localizada en terreno no Media infraestructura pavimentado. Uso deportivo individual; exige otro canal Consumidor recreativo Baja \\/ fuera de alcance inicial de venta, servicio y cumplimiento. El Baja monoplaza no debe prometer transporte de pasajeros, carga pesada, remolque o uso en vías públicas sin pruebas y autorizaciones específicas.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":119,\"augmented\":0,\"text\":\"catálogo de refacciones. técnica probada. Mayor capacidad de llevar material o El Baja monoplaza no sustituye funciones\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":489,\"augmented\":0,\"text\":\"Atributo de ficha técnica Relevancia comercial \\/ limitación Dimensiones compactas en ciertos senderos; verificar radios 64 in de ancho y 108 in de largo de giro y acceso real. Sistema sencillo de atender, pero potencia y consumo deben Motor Kohler CH440 de 14 HP a gasolina probarse bajo carga real. Diseño apropiado para evaluar desplazamiento en terreno Suspensión independiente de doble horquilla irregular. Llantas 22 × 10-10 Adecuación al terreno y desgaste por determinar en pruebas.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":299,\"augmented\":1,\"text\":\"Atributo de ficha técnica Relevancia comercial \\/ limitación Dimensiones compactas en ciertos senderos; verificar radios 64 in de ancho y 108 in de largo de giro y acceso real. Sistema sencillo de atender, pero potencia y consumo deben Motor Kohler CH440 de 14 HP a gasolina probarse bajo carga real.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":303,\"augmented\":0,\"text\":\"pasajeros según modelo. de carga. Menor compromiso de compra y Costo recurrente y disponibilidad Renta de vehículos sustitución según contrato. dependiente del arrendador. Personalización y servicio comercial Prototipo no validado; red de servicio y Baja negociado para la flota. costo final pendientes.\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":620,\"augmented\":0,\"text\":\"Indicador Dato Interpretación Oportunidad del caso; pendiente de Demanda explícita 70 unidades adjudicación. Primer hito 20 unidades al mes 6 Necesidad de abastecimiento inicial. Segundo hito 50 unidades adicionales al mes 18 Expansión de flota de largo plazo. Escenario de precio propuesto; no Ingreso potencial base 70 × $170,000 = $11,900,000 MXN ingreso asegurado. Faltan universo de compradores y ventas Cuota de mercado No calculable con datos disponibles comparables. Escenarios comerciales ilustrativos (no probabilidades estadísticas): sin contrato = 0 unidades; adjudicación de 20 unidades = $3.4 millones MXN;\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":333,\"augmented\":0,\"text\":\"y tiempo medio de reparación. Comparar al menos 3 alternativas cotizadas Precio total entregado, garantía, repuestos, consumo y plazos. Compra + logística + operación + mantenimientos + tiempos de Análisis de costo total de propiedad a 3–5 años inactividad − valor residual. Lotes, tiempos, almacenamiento, aduana y abastecimiento de\"},{\"section\":\"mercado\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":299,\"augmented\":1,\"text\":\"Actividad Entregable verificable Horas de uso por día, pendientes, distancia, clima, carga, Entrevistar al responsable de operaciones del comprador número de usuarios y rutas. Identificar labores compatibles con monoplaza y labores que Mapeo de funciones de la flota requieren otro tipo de vehículo.\"},{\"section\":\"mercado\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":361,\"augmented\":0,\"text\":\"Actividad Entregable verificable Horas de uso por día, pendientes, distancia, clima, carga, Entrevistar al responsable de operaciones del comprador número de usuarios y rutas. Identificar labores compatibles con monoplaza y labores que Mapeo de funciones de la flota requieren otro tipo de vehículo. Frenado, temperatura, consumo, autonomía, estabilidad, fallas\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":105,\"augmented\":0,\"text\":\"adjudicación completa de 70 = $11.9 millones MXN. Se asume precio constante, sin impuestos ni transporte.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":336,\"augmented\":1,\"text\":\"Riesgo técnico: el prototipo todavía no ha completado pruebas, por lo que el rendimiento real es incierto. Riesgo de capacidad: el taller existe, pero la tasa de producción mensual no se ha medido. Riesgo comercial: el comprador adjudicará al mejor postor y puede preferir marcas con soporte establecido. Riesgo logístico y regulatorio:\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":387,\"augmented\":0,\"text\":\"Riesgo técnico: el prototipo todavía no ha completado pruebas, por lo que el rendimiento real es incierto. Riesgo de capacidad: el taller existe, pero la tasa de producción mensual no se ha medido. Riesgo comercial: el comprador adjudicará al mejor postor y puede preferir marcas con soporte establecido. Riesgo logístico y regulatorio: la importación de vehículos y motores todoterreno.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":575,\"augmented\":0,\"text\":\"La inversión contempla dos jigs físicos y sub-jigs para operar las células, con una geometría común de chasís para las tres variantes. El proceso considera fosfatado, primario epóxico y recubrimiento en polvo, además de tornillería protegida, rodamientos sellados y conectores sellados. La reducción del costo total de propiedad y del tiempo de inactividad es un objetivo por validar mediante pruebas de durabilidad. Oportunidad de senderos ecoturísticos (UTV Trails) y monetización: se propone evaluar circuitos controlados para recorridos guiados en el complejo de Yucatán.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":559,\"augmented\":0,\"text\":\"Para efectos de este caso de negocios, se define como una empresa ficticia de ingeniería aplicada y desarrollo vehicular de bajo volumen, vinculada al entorno de innovación del y al México. Se modela una sede operativa y taller en , con enlaces logísticos por definir para atender al cliente en Yucatán. La ubicación es provisional y se utiliza únicamente como supuesto de planeación. La empresa propuesta se enfoca en el diseño y manufactura de soluciones de movilidad todoterreno ligera para reservas ecológicas, operaciones de campo y complejos turísticos.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":647,\"augmented\":0,\"text\":\"el plan de producción contempla además personal directo escalonado y contrataciones temporales para la fase 1, por lo que las horas de nómina y la plantilla operativa deben entenderse como supuestos separados del perfil corporativo. La propuesta se apoya en el aprendizaje de diseño y manufactura asociado al entorno Baja SAE, pero el vehículo comercial descrito no se presenta como un producto previamente probado. La validación de suspensión, impacto, fatiga, frenado, estabilidad y operación en arena o ambientes salinos se realizará mediante simulaciones y pruebas físicas planificadas; sus resultados se incorporarán cuando estén disponibles.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":182,\"augmented\":1,\"text\":\"La propuesta se apoya en el aprendizaje de diseño y manufactura asociado al entorno Baja SAE, pero el vehículo comercial descrito no se presenta como un producto previamente probado.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":259,\"augmented\":1,\"text\":\"Estas cifras son antecedentes supuestos de la empresa simulada y no deben interpretarse como estados financieros reales de una compañía existente. Estrategia Tecnológica, Oportunidades y Propuesta de Valor Estrategia de materiales y tecnología de manufactura:\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":254,\"augmented\":1,\"text\":\"se propone utilizar tubería de acero estructural, corte de placas mediante láser CNC y ensamblaje en jigs de precisión. La inversión contempla dos jigs físicos y sub-jigs para operar las células, con una geometría común de chasís para las tres variantes.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":622,\"augmented\":0,\"text\":\"El escenario ilustrativo supone una tarifa de $34.55 USD ($650 MXN) por recorrido de 45 minutos, cuatro recorridos diarios por vehículo, 70% de ocupación y 25 días de operación al mes. Bajo esos supuestos, las 40 unidades Huésped de la flota completa podrían generar ingresos brutos aproximados de $96,732 USD ($1,820,000 MXN) mensuales para el cliente; con las 11 unidades Huésped contempladas al cierre de la fase 1, el escenario sería de aproximadamente $26,601 USD ($500,500 MXN) mensuales. Son ingresos brutos hipotéticos, sujetos a validar demanda, capacidad de operación, permisos, seguridad y viabilidad ambiental.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":650,\"augmented\":0,\"text\":\"Como parte del perfil ficticio requerido para el caso, se modela un capital semilla inicial de $26,575 USD ($500,000 MXN), ingresos acumulados de $63,780 USD ($1,200,000 MXN) por servicios de prototipado y consultoría y un margen promedio de 18% en proyectos anteriores. Estas cifras son antecedentes supuestos de la empresa simulada y no deben interpretarse como estados financieros reales de una compañía existente. Estrategia Tecnológica, Oportunidades y Propuesta de Valor Estrategia de materiales y tecnología de manufactura: se propone utilizar tubería de acero estructural, corte de placas mediante láser CNC y ensamblaje en jigs de precisión.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":589,\"augmented\":0,\"text\":\"Para completar el perfil solicitado por la competencia, se plantea como supuesto una participación de mercado del 15% en un nicho regional de movilidad todoterreno ligera; esta cifra no proviene de una medición de mercado y deberá presentarse como hipótesis del caso, no como dato comprobado. El perfil ficticio considera una plantilla base de 12 profesionales: 4 ingenieros mecánicos y mecatrónicos, 2 diseñadores industriales, 2 especialistas en manufactura, 2 analistas financieros y comerciales y 2 técnicos de validación. Esta plantilla representa la estructura corporativa propuesta;\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":642,\"augmented\":0,\"text\":\"No se afirma que la huella ambiental o el ruido ya hayan sido medidos. Propuesta de valor central y diferenciador clave: Amazonas propone una solución vehicular monoplaza modular para operaciones recreativas y de trabajo. Los accesorios se fijan a placas dedicadas para que una plataforma común pueda configurarse para huéspedes o para mantenimiento, con rack o tráiler, sin diseñar un chasís distinto para cada variante. La propuesta busca aportar flexibilidad, mantenimiento estandarizado y control de costos; su ventaja frente a alternativas comerciales deberá validarse con pruebas, cotizaciones y comparación de costo total de propiedad.\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":447,\"augmented\":0,\"text\":\"Esta herramienta se considera una iniciativa comercial por desarrollar, no un servicio actualmente operativo. Modelos de Comercialización Flexibles: Ofrecemos esquemas adaptados al flujo de inversión de proyectos inmobiliarios y turísticos: ● Venta Directa Escalonada: 20% de anticipo para asignación de manufactura y adquisición de materiales, 80% liquidable contra entrega y validación en sitio de cada lote. ● Arrendamiento operativo (leasing):\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":576,\"augmented\":0,\"text\":\"se propone evaluar esquemas de renta a mediano y largo plazo para convertir parte del gasto de capital (CAPEX) en gasto operativo (OPEX), sujetos a la viabilidad financiera, contractual y fiscal aplicable. ● Pólizas de mantenimiento preventivo: se proponen paquetes de servicio técnico periódico, capacitación a operadores y suministro programado de refacciones críticas, sujeto a inventario, acuerdos con proveedores y tiempos logísticos. Factores diferenciadores: la propuesta combina una plataforma modular, opciones recreativas y operativas, y un plan de soporte posventa.\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":593,\"augmented\":0,\"text\":\"La ergonomía, accesibilidad, durabilidad en ambientes severos y disponibilidad de refacciones se tratarán como objetivos de diseño y operación sujetos a evaluación, no como resultados ya garantizados. Fuentes de Ingreso y Modelo de Negocio de Amazonas El modelo comercial y los flujos de ingreso de se estructuran a través de cuatro líneas estratégicas: ● Comercialización directa de la flota todoterreno: Venta B2B de unidades en sus configuraciones modulares Huésped, Trabajo con rack de carga y Trabajo con tráiler de arrastre ligero. ● Venta de paquetes de opciones y accesorios modulares:\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":598,\"augmented\":0,\"text\":\"Canales de Venta Directa e Indirecta: La comercialización principal se realiza mediante venta consultiva B2B directa con Outdoor Adventure Holdings y empresas de hospitalidad al aire libre, complementada con alianzas estratégicas con distribuidores y talleres regionales de equipo todoterreno en el sureste mexicano para soporte posventa inmediato. Plataforma digital B2B: se propone desarrollar un portal para clientes corporativos que permita configurar especificaciones de flota, gestionar solicitudes de cotización, dar seguimiento a órdenes de producción y solicitar refacciones y consumibles.\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":268,\"augmented\":1,\"text\":\"● Venta de paquetes de opciones y accesorios modulares: Comercialización de aditamentos de personalización visual corporativa, kits para arena y ambientes marinos, y accesorios específicos para recreación o mantenimiento. ● Kits de refacciones y consumibles iniciales:\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":348,\"augmented\":1,\"text\":\"Canales de Venta Directa e Indirecta: La comercialización principal se realiza mediante venta consultiva B2B directa con Outdoor Adventure Holdings y empresas de hospitalidad al aire libre, complementada con alianzas estratégicas con distribuidores y talleres regionales de equipo todoterreno en el sureste mexicano para soporte posventa inmediato.\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":178,\"augmented\":1,\"text\":\"20% de anticipo para asignación de manufactura y adquisición de materiales, 80% liquidable contra entrega y validación en sitio de cada lote. ● Arrendamiento operativo (leasing):\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":336,\"augmented\":1,\"text\":\"Fuentes de Ingreso y Modelo de Negocio de Amazonas El modelo comercial y los flujos de ingreso de se estructuran a través de cuatro líneas estratégicas: ● Comercialización directa de la flota todoterreno: Venta B2B de unidades en sus configuraciones modulares Huésped, Trabajo con rack de carga y Trabajo con tráiler de arrastre ligero.\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":218,\"augmented\":1,\"text\":\"Suministro programado de paquetes de piezas críticas, bandas CVT, rodamientos, filtros y componentes de desgaste para garantizar la disponibilidad operativa de la flota. ● Servicios posventa y pólizas de mantenimiento:\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":607,\"augmented\":0,\"text\":\"Comercialización de aditamentos de personalización visual corporativa, kits para arena y ambientes marinos, y accesorios específicos para recreación o mantenimiento. ● Kits de refacciones y consumibles iniciales: Suministro programado de paquetes de piezas críticas, bandas CVT, rodamientos, filtros y componentes de desgaste para garantizar la disponibilidad operativa de la flota. ● Servicios posventa y pólizas de mantenimiento: Contratos de servicio preventivo y correctivo periódico, con la opción de incorporar un técnico mecánico residente directamente en las instalaciones del complejo ecoturístico.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":623,\"augmented\":0,\"text\":\"Para arena y ambientes cercanos al mar se proponen llantas y rines adecuados, precarga trasera ajustable y un paquete anticorrosión opcional (fosfatado, primario epóxico y recubrimiento en polvo en chasís, rack y tráiler; tornillería con recubrimiento zinc-níquel o inoxidable; rodamientos sellados con grasa marina; conectores sellados), con costo estimado de USD$6,580 por unidad y precio sugerido de USD$8,800. Agrega aproximadamente seis horas de mano de obra propia y dos días de proceso externo por lote. El desempeño en arena, agua salina y condiciones severas se considera pendiente de validación física. Autonomía.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":632,\"augmented\":0,\"text\":\"La propuesta conserva la jaula tubular de la plataforma monoplaza e incorpora asiento anclado a la estructura, arnés de cuatro o cinco puntos con certificación SFI según la especificación aplicable, redes laterales, dos interruptores de corte, extintor y luz de freno. Se adopta como requisito de diseño un sistema de dos circuitos independientes que permita frenar las cuatro ruedas; la selección final de componentes y el cumplimiento de frenado deben comprobarse mediante simulación y pruebas. Se propone casco obligatorio y una charla de seguridad de diez minutos antes del uso. La versión Huésped utilizará limitador con llave.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":576,\"augmented\":0,\"text\":\"La versión comercial considera un tanque de 3 galones (≈11.4 litros) bajo el asiento y entre los largueros, con cuatro puntos de soporte con bandas y cojines, escudo contra impactos, mamparo de separación del ocupante, venteo accesible desde el exterior y línea de combustible de especificación automotriz alejada del escape. Con un consumo supuesto de 1.3 L\\/h, la autonomía teórica sería de 8.8 horas; en un escenario conservador de 2.0 L\\/h, de 5.7 horas. Ambas cifras son supuestos sin validación en banco y no se presentan como desempeño comprobado. Seguridad del ocupante.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":582,\"augmented\":0,\"text\":\"Con tráiler se propone un límite inicial de 15–20 km\\/h. Edad y estatura mínimas de los conductores se definirán en el protocolo operativo del cliente; no se fija un límite sin evaluación ergonómica y de seguridad. Diseño para manufactura. Se propone migrar del ajuste manual a jigs y plantillas. Las tolerancias objetivo son ±1 mm en puntos de suspensión y dirección y ±2 mm en los demás puntos de referencia. La fabricación utilizará un procedimiento de soldadura calificado, inspección visual del 100% de las juntas de la jaula y líquidos penetrantes en una de cada diez unidades.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":515,\"augmented\":0,\"text\":\"La propuesta consiste en un vehículo todoterreno monoplaza dirigido a Outdoor Adventure Holdings para el traslado de huéspedes y el apoyo a actividades de construcción, mantenimiento y operación de senderos, playas y zonas naturales. Se plantea una geometría de chasís tubular común para las configuraciones Huésped, Trabajo con rack y Trabajo con tráiler. Para cubrir la capacidad de producción se presupuestan dos jigs físicos con sub-jigs sobre la misma geometría; no se desarrolla una variante de dos ocupantes.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":534,\"augmented\":0,\"text\":\"El rack será de aluminio para limitar la masa añadida. El paquete de refuerzo para Trabajo se estima en $7,080 e incluye resortes traseros más rígidos o con precarga ajustable, placas de montaje, frenos con mayor capacidad térmica y reducción final más corta. En costo, una geometría común de chasís y dos jigs físicos con sub-jigs para atender la capacidad de producción para ambas versiones, tubo y placa en pocas medidas y el soporte del caliper cortado con láser (en vez de maquinado) buscan reducir inventario, horas y variación.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":605,\"augmented\":0,\"text\":\"Las opciones de personalización (pintura o vinil con la identidad del cliente, butaca drenable, rines y llantas de arena, volante antideslizante) cuestan entre USD$900 y USD$4,900 por unidad y se montan sin alterar el chasís ni el jig. Terreno y protección. La plataforma se plantea para senderos, terracería, rocas, troncos, lodo y pendientes, lo que cubre el traslado de huéspedes y equipo a las rutas de ciclismo, pesca y senderismo, las actividades acuáticas en la costa (tabla o bodyboard, hielera sellada) y el apoyo de la versión Trabajo a las labores de la reserva, incluido el manejo de animales.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":337,\"augmented\":1,\"text\":\"Paquete (kit de refuerzo + Traslado de materiales voluminosos en terracería firme. Tráiler ligero de un eje; carga nominal de diseño de Trabajo con tráiler tráiler): costo $37,680; precio sugerido La mezcla de 40 unidades Huésped, 20 Trabajo con rack y 10 Trabajo con tráiler es una hipótesis de planeación basada en el modelo comercial.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":268,\"augmented\":1,\"text\":\"La propuesta conserva la jaula tubular de la plataforma monoplaza e incorpora asiento anclado a la estructura, arnés de cuatro o cinco puntos con certificación SFI según la especificación aplicable, redes laterales, dos interruptores de corte, extintor y luz de freno.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":272,\"augmented\":1,\"text\":\"precio Huésped con llave. sugerido $21,400. Construcción, mantenimiento y operación diaria. Rack modular de aluminio; capacidad nominal de diseño 20 unidades. Paquete (kit de refuerzo + rack): Trabajo con rack de 50 kg. costo $12,620; precio sugerido $16,900. 10 unidades.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":332,\"augmented\":1,\"text\":\"Para reducir operaciones internas se propone subcontratar el dog clutch mediante CNC y tratamiento térmico; cortar los pedales con láser y soldarlos internamente bajo un diseño ajustable común; fabricar el soporte del caliper como placa cortada con láser; y comprar una butaca comercial o subcontratar su fabricación por rotomoldeo.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":340,\"augmented\":1,\"text\":\"Todas las unidades incluyen ganchos de remolque y recuperación, soporte para extintor, kit de herramienta y placa de identificación de flota. Configuración Uso y equipamiento Volumen y paquete (USD) Turismo, traslado de huéspedes y apoyo en playa y senderos. Accesorios de playa, iluminación y limitador 40 unidades. Paquete: costo $16,040;\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":281,\"augmented\":1,\"text\":\"Se plantea una geometría de chasís tubular común para las configuraciones Huésped, Trabajo con rack y Trabajo con tráiler. Para cubrir la capacidad de producción se presupuestan dos jigs físicos con sub-jigs sobre la misma geometría; no se desarrolla una variante de dos ocupantes.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":234,\"augmented\":1,\"text\":\"La configuración Huésped incorpora soporte para tabla o bodyboard, hielera sellada, bolsa impermeable, soporte para celular o cámara, guardalodos y defensas para arena, bandera, luces LED y limitador de velocidad con llave (Figura 1).\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":293,\"augmented\":1,\"text\":\"Para organizar la producción se propone repartirla proporcionalmente entre las fases: fase 1, de 20 unidades, con 11 Huésped, 6 con rack y 3 con tráiler; fase 2, de 50 unidades, con 29 Huésped, 14 con rack y 7 con tráiler. Esta distribución deberá ajustarse si el cliente confirma otra mezcla.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":259,\"augmented\":1,\"text\":\"El rack será de aluminio para limitar la masa añadida. El paquete de refuerzo para Trabajo se estima en $7,080 e incluye resortes traseros más rígidos o con precarga ajustable, placas de montaje, frenos con mayor capacidad térmica y reducción final más corta.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":325,\"augmented\":1,\"text\":\"La versión comercial considera un tanque de 3 galones (≈11.4 litros) bajo el asiento y entre los largueros, con cuatro puntos de soporte con bandas y cojines, escudo contra impactos, mamparo de separación del ocupante, venteo accesible desde el exterior y línea de combustible de especificación automotriz alejada del escape.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":637,\"augmented\":0,\"text\":\"Configuración Uso y equipamiento Volumen y paquete (USD) Turismo, traslado de huéspedes y apoyo en playa y senderos. Accesorios de playa, iluminación y limitador 40 unidades. Paquete: costo $16,040; precio Huésped con llave. sugerido $21,400. Construcción, mantenimiento y operación diaria. Rack modular de aluminio; capacidad nominal de diseño 20 unidades. Paquete (kit de refuerzo + rack): Trabajo con rack de 50 kg. costo $12,620; precio sugerido $16,900. 10 unidades. Paquete (kit de refuerzo + Traslado de materiales voluminosos en terracería firme. Tráiler ligero de un eje; carga nominal de diseño de Trabajo con tráiler tráiler):\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":616,\"augmented\":0,\"text\":\"Se estandarizarán tubos y placas secundarias, tornillería en cuatro o cinco medidas, arneses prearmados y subensambles modulares para chasís, tren delantero y tren trasero. El congelamiento del diseño comercial se planea en ocho semanas, en paralelo con la compra del motor y la CVT. Para reducir operaciones internas se propone subcontratar el dog clutch mediante CNC y tratamiento térmico; cortar los pedales con láser y soldarlos internamente bajo un diseño ajustable común; fabricar el soporte del caliper como placa cortada con láser; y comprar una butaca comercial o subcontratar su fabricación por rotomoldeo.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":465,\"augmented\":0,\"text\":\"costo $37,680; precio sugerido La mezcla de 40 unidades Huésped, 20 Trabajo con rack y 10 Trabajo con tráiler es una hipótesis de planeación basada en el modelo comercial. Para organizar la producción se propone repartirla proporcionalmente entre las fases: fase 1, de 20 unidades, con 11 Huésped, 6 con rack y 3 con tráiler; fase 2, de 50 unidades, con 29 Huésped, 14 con rack y 7 con tráiler. Esta distribución deberá ajustarse si el cliente confirma otra mezcla.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":610,\"augmented\":0,\"text\":\"La configuración Huésped incorpora soporte para tabla o bodyboard, hielera sellada, bolsa impermeable, soporte para celular o cámara, guardalodos y defensas para arena, bandera, luces LED y limitador de velocidad con llave (Figura 1). La configuración Trabajo permite montar un rack trasero modular de aluminio (Figura 2) o un tráiler ligero de un eje (Figura 3). Los accesorios se sujetan a placas dedicadas y se retiran sin modificar los tubos primarios de la jaula. Todas las unidades incluyen ganchos de remolque y recuperación, soporte para extintor, kit de herramienta y placa de identificación de flota.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":373,\"augmented\":0,\"text\":\"En costo, se adopta US$7.50\\/m² como referencia de mercado a partir de la mediana de listados de naves grandes, varias de perfil clase A. Las entregas se realizan en Yucatán; el transporte desde Monterrey se contabiliza por lote como costo logístico independiente de la nave (CBRE, 2026; LAAR, 2026; MXoffices, 2026; Spot2, s. f.). Distribución por área de la nave (600 m²):\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":580,\"augmented\":0,\"text\":\"La fase 2 contempla 50 unidades entre los meses 7–18, a un ritmo de referencia de 4.2 unidades mensuales. El plan considera dos células de soldadura con jig, dos estaciones de ensamble y una estación de pruebas. La secuencia de fabricación es: recepción e inspección de materiales; preparación de tubos y placas mediante corte y doblado; armado y soldadura del chasís y jaula en jig; inspección dimensional y de soldadura; montaje de suspensión y dirección; integración de tren motriz, frenos y electrónica; ensamble final con torque documentado; y pruebas con liberación firmada.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":407,\"augmented\":0,\"text\":\"lotes de 20 Frenos Wilwood o equivalente 3–5 semanas Importado; lotes de 10 Arnés de seguridad Fabricante con certificación SFI 3–5 semanas Importado; lotes de 20 Tubería, placas, corte y doblado Distribuidores y talleres nacionales ≈1 semana Nacional; compra por tonelada CNC y recubrimiento Talleres externos nacionales ≈1 semana por lote; recubrimiento: 2 días por lote Nacional; subcontratación por lote\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":645,\"augmented\":0,\"text\":\"Área Operativa \\/ Célula Superficie (m²) % del Espacio Total Recepción y Almacén General de Materiales 120 m² 20.0% Corte y Preparación de Perfiles y Placas 60 m² 10.0% Células de Soldadura de Precisión con Jig (2 estaciones) 90 m² 15.0% Células de Ensamble Mecánico y Eléctrico (2 estaciones) 90 m² 15.0% Estación de Pruebas Dinámicas y Banco de Potencia 60 m² 10.0% Almacén de Producto Terminado y Listo para Embarque 60 m² 10.0% Oficinas Técnicas, Comedor y Servicios Administrativos 60 m² 10.0% Inspección de Calidad y Metrología \\/ Área de Retoque (30 m² c\\/u) 60 m² 10.0% Superficie Total Operativa 600 m² 100.0% Metodología de cálculo de las\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":601,\"augmented\":0,\"text\":\"de 3 a 10 semanas para componentes importados y alrededor de 1 semana para insumos y servicios nacionales. Los lotes de compra son de 10 piezas (motor, CVT y frenos) y de 20 piezas (amortiguadores y arneses de seguridad). Los plazos y cantidades son supuestos internos pendientes de cotización ( , 2026). Componente\\/proceso Fuente de referencia Plazo de referencia Origen y lote de compra Motor Rehlko CH440 Distribuidor autorizado 6–8 semanas Importado; lotes de 10 CVT compatible; Gaged o Polaris como 6–8 semanas Importado; lotes de 10 Amortiguadores Fox, Elka o equivalente 6–10 semanas Importado;\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":623,\"augmented\":0,\"text\":\"Todos los importes de las secciones se expresan en dólares estadounidenses (USD), convertidos al tipo de cambio FIX de 18.81 MXN por dólar (18.8148 sin redondear), determinado por el Banco de México y publicado en el DOF el 30 de octubre de 2026. Comparativo de renta industrial en Monterrey (USD por m² al mes): Referencia de renta industrial US$\\/m² al mes Fuente Centro de Monterrey (clase C) 2.50–4.00 MXoffices (sep. 2026) Guadalupe \\/ Juárez (clase B) 3.80–5.20 MXoffices (sep. 2026) Escobedo \\/ Salinas Victoria (clase B+) 4.20–5.80 MXoffices (sep. 2026) Santa Catarina \\/ García (clase A \\/ B+) 5.00–6.80 MXoffices (sep.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":620,\"augmented\":0,\"text\":\"ocho semanas de congelamiento de diseño y preparación de equipo (en paralelo con el plazo de entrega del motor y la CVT), más dos semanas de ensamble y pruebas de la primera unidad. Programa de demostración. Tres unidades de preserie —una Huésped, una Trabajo con rack y una Trabajo con tráiler— se destinan a pruebas de manejo antes del contrato de compra definitivo, al amparo de una carta de intención que autoriza las actividades, el seguro y las responsabilidades de ambas partes. Las unidades se contabilizan dentro del lote inicial de 20. El presupuesto del programa, de US$8,772, se integra con cuatro conceptos:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":584,\"augmented\":0,\"text\":\"unidades 21 a 70) F 47 h chasís y jaula + 37 h 4 Desglose de la fase 2 por etapa Las 166 h promedio por unidad se reparten entre las cuatro etapas de fabricación. suspensión y dirección + 42 h tren motriz, frenos y electrónica + 40 h ensamble final y pruebas = 166 h 5 Utilización de la plantilla en fase 2 ri d li z sá b ≈ 6 unidades al mes 3,359 h de arranque, capacitación 6 Horas pagadas en fase 1 ti l m es: 6 h por persona) y curva Plan de producción. La fase 1 contempla 20 unidades durante los meses 1–6, con tres unidades en el mes 3 y las 17 restantes entre los meses 4 y 6.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":627,\"augmented\":0,\"text\":\"2026) Listados activos en Nuevo León: mediana (rango) 5.64 (4.52–6.75) MXoffices (sep. 2026) Clase A, área metropolitana: precio solicitado 2T26 6.98 CBRE MarketView Industrial 2T26, vía LAAR Mediana de naves lis tadas, corredor Monterrey Renta modelada para Amazonas (600 m²) US$7.50\\/m² → (600m2) US$4,500 al mes Mediana. ¿Por qué Monterrey? Se elige Monterrey por su concentración de proveedores industriales y talleres especializados que pueden atender corte láser, doblado, maquinado CNC y recubrimiento; por la disponibilidad regional de talento técnico y de manufactura; y por su conexión con el ecosistema de innovación.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":635,\"augmented\":0,\"text\":\"Fase Volumen y calendario Horas de trabajo Personal directo propuesto 4,961 h totales; promedio aproximado de Escalamiento de 4 personas en mes 1, 8 en mes 2 y 10 en meses 20 unidades; meses 1–6. 3 unidades en mes 3 y Fase 1 248 h\\/unidad. Unidad 1: 400 h; unidad 20: 3–6: 3 soldadores, 4 ensambladores, 1 técnico eléctrico\\/arnés y 2 17 en meses 4–6. 198 h. personas de calidad\\/pruebas. 50 unidades; meses 7–18. Promedio de 4.2 8,306 h totales; promedio de 166.1 5 personas: 2 soldadores, 2 ensambladores y 1 persona de Fase 2 unidades\\/mes; capacidad cercana a 6 con h\\/unidad. Unidad 70: aproximadamente calidad\\/pruebas. sábados. 148 h.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":552,\"augmented\":0,\"text\":\"transporte (US$1,914), seguro (US$1,276), refacciones y desgaste (US$2,392) y personal de acompañamiento (US$3,190); no incluye una segunda construcción. Las unidades están disponibles desde la semana 11, una vez concluida la revisión de seguridad de la primera unidad (semanas 9–10). Cadena de suministro. Los componentes críticos se identificaron a partir de la plataforma Baja SAE del equipo (motor Rehlko CH440, CVT, suspensión, frenos y arnés SFI) y a cada uno se asignó una fuente de referencia por categoría. Los plazos son rangos de planeación:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":237,\"augmented\":1,\"text\":\"El equipo de producción queda listo para la semana 8 y el ensamble y las pruebas de la primera unidad se programan para las semanas 9–10. La referencia de diez semanas se cuenta desde la autorización de ingeniería y el inicio de compras:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":337,\"augmented\":1,\"text\":\"Tres unidades de preserie —una Huésped, una Trabajo con rack y una Trabajo con tráiler— se destinan a pruebas de manejo antes del contrato de compra definitivo, al amparo de una carta de intención que autoriza las actividades, el seguro y las responsabilidades de ambas partes. Las unidades se contabilizan dentro del lote inicial de 20.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":330,\"augmented\":1,\"text\":\"En costo, se adopta US$7.50\\/m² como referencia de mercado a partir de la mediana de listados de naves grandes, varias de perfil clase A. Las entregas se realizan en Yucatán; el transporte desde Monterrey se contabiliza por lote como costo logístico independiente de la nave (CBRE, 2026; LAAR, 2026; MXoffices, 2026; Spot2, s. f.).\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":236,\"augmented\":1,\"text\":\"148 h 3 Horas totales y promedio por fase (cid:0) nidades 1 a 20; fase 2: unidades 21 a 70) F 47 h chasís y jaula + 37 h 4 Desglose de la fase 2 por etapa Las 166 h promedio por unidad se reparten entre las cuatro etapas de fabricación.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":245,\"augmented\":1,\"text\":\"Los componentes críticos se identificaron a partir de la plataforma Baja SAE del equipo (motor Rehlko CH440, CVT, suspensión, frenos y arnés SFI) y a cada uno se asignó una fuente de referencia por categoría. Los plazos son rangos de planeación:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":334,\"augmented\":1,\"text\":\"La secuencia de fabricación es: recepción e inspección de materiales; preparación de tubos y placas mediante corte y doblado; armado y soldadura del chasís y jaula en jig; inspección dimensional y de soldadura; montaje de suspensión y dirección; integración de tren motriz, frenos y electrónica; ensamble final con torque documentado;\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":228,\"augmented\":1,\"text\":\"El presupuesto del programa, de US$8,772, se integra con cuatro conceptos: transporte (US$1,914), seguro (US$1,276), refacciones y desgaste (US$2,392) y personal de acompañamiento (US$3,190); no incluye una segunda construcción.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":345,\"augmented\":1,\"text\":\"Área Operativa \\/ Célula Superficie (m²) % del Espacio Total Recepción y Almacén General de Materiales 120 m² 20.0% Corte y Preparación de Perfiles y Placas 60 m² 10.0% Células de Soldadura de Precisión con Jig (2 estaciones) 90 m² 15.0% Células de Ensamble Mecánico y Eléctrico (2 estaciones) 90 m² 15.0% Estación de Pruebas Dinámicas y Banco de\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":348,\"augmented\":1,\"text\":\"Referencia de renta industrial US$\\/m² al mes Fuente Centro de Monterrey (clase C) 2.50–4.00 MXoffices (sep. 2026) Guadalupe \\/ Juárez (clase B) 3.80–5.20 MXoffices (sep. 2026) Escobedo \\/ Salinas Victoria (clase B+) 4.20–5.80 MXoffices (sep. 2026) Santa Catarina \\/ García (clase A \\/ B+) 5.00–6.80 MXoffices (sep. 2026) Listados activos en Nuevo León:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":207,\"augmented\":1,\"text\":\"ocho semanas de congelamiento de diseño y preparación de equipo (en paralelo con el plazo de entrega del motor y la CVT), más dos semanas de ensamble y pruebas de la primera unidad. Programa de demostración.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":621,\"augmented\":0,\"text\":\"La operación se modela para una nave de 600 m² en un parque industrial de Monterrey, N.L. como ubicación provisional y no confirmada. Para estimar la renta se adopta la mediana en listados de naves grandes del corredor Monterrey, de aproximadamente 5,500 a 23,600 m² y varias de perfil clase A; aplicada a 600 m². Esta mediana corresponde a inmuebles industriales de gran superficie y puede ubicarse por encima de la renta esperable para una nave pequeña. El valor se contrasta con las rentas por corredor publicadas por MXoffices y con el precio solicitado de clase A reportado por CBRE para 2T26 y reproducido por LAAR.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":627,\"augmented\":0,\"text\":\"Paso Qué se calcula Cómo se calcula Resultado Se parte de 400 h, por debajo de las 500–1,000 h de un prototipo artesanal, porque la 1 Horas de la unidad 1 primera unidad ya se arma con jigs, tubo y placa cortados con láser, y arneses T₁ = 400 h prearmados. Modelo de Wright con curva del 85%, referencia para ensamble y soldadura manual con Unidad 20: 198 h 2 Curva de aprendizaje herramental: cada vez que se duplica el número acumulado de unidades, las horas por unidad bajan 15%. T(cid:0) = 400 × n^b, con b = log(0.85) ÷ log(2) = −0.2345. Unidad 70: 148 h 3 Horas totales y promedio por fase (cid:0) nidades 1 a 20; fase 2:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":494,\"augmented\":0,\"text\":\"Calendario de arranque. Las órdenes del motor y la CVT se emiten en la semana 1, con plazos de entrega de seis a ocho semanas. El diseño, los planos y la lista de materiales se congelan entre las semanas 1 y 8; los jigs se fabrican entre las semanas 4 y 8. El equipo de producción queda listo para la semana 8 y el ensamble y las pruebas de la primera unidad se programan para las semanas 9–10. La referencia de diez semanas se cuenta desde la autorización de ingeniería y el inicio de compras:\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":605,\"augmented\":0,\"text\":\"1. Un atraso en motor o CVT desplaza la primera unidad; el margen del calendario es de aproximadamente dos semanas. Mitigación: emitir las órdenes de motor y CVT en la semana 1, calificar proveedores alternos para piezas críticas y escalonar compras según la demanda. 2. Una variación cambiaria modifica el costo de materiales importados y el margen. Mitigación: solicitar cotizaciones con vigencia definida y establecer una cláusula de revisión cuando el dólar varíe más de 5%. 3. Con una curva del 90%, las horas subirán a aproximadamente 17,148 (+29%) y aumentaría el costo de mano de obra. Mitigación:\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":547,\"augmented\":0,\"text\":\"medir las horas reales de las primeras cinco unidades, recalcular la curva y ajustar la capacidad. 4. Retraso en la entrega de las 20 unidades del mes 6. Mitigación: utilizar sub-jigs, balancear cargas entre estaciones y revisar semanalmente el avance y la capacidad disponible. 5. Capacidad de la fase 2. Origen: cinco personas aportan 800 horas al mes frente a 692 horas requeridas (aproximadamente 87% de utilización). Efecto: un aumento de 20% en el volumen exigiría cerca de 830 horas al mes, por encima de la capacidad ordinaria. Mitigación:\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":482,\"augmented\":0,\"text\":\"El vehículo comercial se valida en dos etapas: simulación antes de congelar el diseño y pruebas físicas con la primera unidad y con cada lote de producción. Los criterios de aceptación se derivan de los requisitos de inspección técnica de Baja SAE —salida del conductor en 5 segundos, bloqueo de las cuatro ruedas con dos circuitos de freno independientes, interruptores de corte, extintor y luz de freno— y de las cargas nominales de diseño del rack (50 kg) y del tráiler (150 kg).\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":438,\"augmented\":0,\"text\":\"Con anticipo de 20%, la caja mínima estimada es −US$2,660 en el mes 5; sin anticipo, −US$189,374 en el mes 4. Mitigación: condicionar el inicio de fabricación al anticipo de 20% y escalonar las compras por lote. 9. La renta puede variar respecto de la mediana de mercado y el flete por lote incrementa el costo de entrega. Mitigación: negociar un arrendamiento a plazo fijo por los 18 meses de producción y consolidar los fletes por lote.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":445,\"augmented\":0,\"text\":\"Salida del conductor con Salida con arnés abrochado en un máximo de 5 segundos, sin Primera unidad accesorios interferencia de accesorios. Control de producción: en cada unidad se realiza inspección dimensional del chasís, inspección visual del 100% de las soldaduras de la jaula, verificación de torque, comprobación de frenado, fugas de combustible, luces e interruptor de corte y prueba de pista de 15 minutos, con hoja de liberación firmada.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":545,\"augmented\":0,\"text\":\"congelamiento Bloqueo de las cuatro ruedas con carga nominal y distancia ≤5 m a 20 Simulación y prueba Frenado con carga km\\/h, sujeto a validar método y condiciones de prueba. física Antes del Suspensión multicuerpo Sin llegar al tope en un bache de 15 cm a 20 km\\/h. congelamiento Temperatura de CVT y Temperaturas dentro de límites del fabricante después de una hora de Simulación y banco motor con tráiler carga continua. Carga estática de rack y Aplicar 1.5 veces la carga nominal sin deformación permanente ni Primera unidad tráiler grietas.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":595,\"augmented\":0,\"text\":\"utilizar sábados como holgura, habilitar un segundo turno parcial o ampliar temporalmente la capacidad mediante autorización de producción. 6. Retrabajos, rechazos o fallas en las pruebas de liberación. Mitigación: procedimiento de soldadura calificado, inspección visual del 100% de las juntas, líquidos penetrantes en una de cada diez unidades y hoja de liberación firmada. 7. Retrabajo de jigs y retraso de la primera unidad. Mitigación: completar las simulaciones de jaula antes de liberar los planos de jig y conservar un chasís común con sub-jigs para limitar el alcance de los cambios. 8.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":168,\"augmented\":0,\"text\":\"Los límites de edad y estatura, la capacidad nominal de carga y los requisitos reglamentarios se establecen en el protocolo de operación antes de entregar las unidades.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":283,\"augmented\":1,\"text\":\"en cada unidad se realiza inspección dimensional del chasís, inspección visual del 100% de las soldaduras de la jaula, verificación de torque, comprobación de frenado, fugas de combustible, luces e interruptor de corte y prueba de pista de 15 minutos, con hoja de liberación firmada.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":266,\"augmented\":1,\"text\":\"congelamiento Bloqueo de las cuatro ruedas con carga nominal y distancia ≤5 m a 20 Simulación y prueba Frenado con carga km\\/h, sujeto a validar método y condiciones de prueba. física Antes del Suspensión multicuerpo Sin llegar al tope en un bache de 15 cm a 20 km\\/h.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":316,\"augmented\":1,\"text\":\"condicionar el inicio de fabricación al anticipo de 20% y escalonar las compras por lote. 9. La renta puede variar respecto de la mediana de mercado y el flete por lote incrementa el costo de entrega. Mitigación: negociar un arrendamiento a plazo fijo por los 18 meses de producción y consolidar los fletes por lote.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":161,\"augmented\":1,\"text\":\"Salida del conductor con Salida con arnés abrochado en un máximo de 5 segundos, sin Primera unidad accesorios interferencia de accesorios. Control de producción:\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":214,\"augmented\":1,\"text\":\"utilizar sábados como holgura, habilitar un segundo turno parcial o ampliar temporalmente la capacidad mediante autorización de producción. 6. Retrabajos, rechazos o fallas en las pruebas de liberación. Mitigación:\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":203,\"augmented\":1,\"text\":\"con y sin tráiler congelamiento Estabilidad estática y Pérdida de estabilidad no mayor a 10% frente al monoplaza sin carga, Antes del lateral con carga según método de evaluación definido por Ingeniería.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":551,\"augmented\":0,\"text\":\"Por lote de diez unidades se realiza una prueba de resistencia de dos horas, medición de peso y distribución de masa, re-torque, líquidos penetrantes y carga estática de 1.5 veces la carga nominal del rack y tráiler. La primera unidad tiene inspección de primer artículo y verificación de salida del conductor en cinco segundos. El consumo de combustible se mide en banco y en operación representativa para determinar la autonomía real. Las pruebas en arena y ambientes salinos verifican tracción, corrosión, sellado eléctrico, frenado y temperaturas.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":537,\"augmented\":0,\"text\":\"Prueba Criterio de aceptación propuesto Etapa Elementos finitos de jaula Antes del Factor de seguridad ≥1.5 con carga vertical de 3 g sobre la carga y chasís con congelamiento de nominal; sin deformación plástica en tubos primarios. rack\\/enganche diseño Dinámica longitudinal Antes del Arranque en pendiente de 15% con carga nominal. con y sin tráiler congelamiento Estabilidad estática y Pérdida de estabilidad no mayor a 10% frente al monoplaza sin carga, Antes del lateral con carga según método de evaluación definido por Ingeniería.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":626,\"augmented\":0,\"text\":\"Para apoyar la operación de la flota de 70 vehículos en Yucatán, se propone infraestructura en las instalaciones del cliente. Estas superficies corresponden al sitio del cliente y son independientes de la nave de manufactura de 600 m² modelada para Amazonas: ● Almacenamiento y Taller: Se requiere un área de estacionamiento techado o con sombra de aproximadamente 350 m² (70 unidades × 5 m² por unidad). Además, sugerimos la adecuación de un taller de 60 m² equipado con caballetes, banco de trabajo y toma de aire, un almacén seguro de refacciones de 20 m², y un área de lavado con agua dulce de 30 m². ● Personal Requerido:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":252,\"augmented\":0,\"text\":\"El tanque de 2,000 L de doble pared se dimensiona para cubrir 2.7 días del escenario base con jornada de 8 h y 1.8 días del escenario de alta exigencia; su instalación queda sujeta a la normativa aplicable, permisos y evaluación de seguridad del sitio.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":340,\"augmented\":1,\"text\":\"Se requiere un área de estacionamiento techado o con sombra de aproximadamente 350 m² (70 unidades × 5 m² por unidad). Además, sugerimos la adecuación de un taller de 60 m² equipado con caballetes, banco de trabajo y toma de aire, un almacén seguro de refacciones de 20 m², y un área de lavado con agua dulce de 30 m². ● Personal Requerido:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":319,\"augmented\":1,\"text\":\"Para la gestión de la flota, el cliente necesitará 2 técnicos mecánicos y 1 almacenista. ● Abastecimiento de combustible: la demanda diaria se calcula como flota × consumo por hora × horas de jornada. Con 1.3 L\\/h (escenario base), los 70 vehículos requieren entre 546 L (70 × 1.3 × 6 h) y 728 L (70 × 1.3 × 8 h) al día;\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":508,\"augmented\":0,\"text\":\"Para la gestión de la flota, el cliente necesitará 2 técnicos mecánicos y 1 almacenista. ● Abastecimiento de combustible: la demanda diaria se calcula como flota × consumo por hora × horas de jornada. Con 1.3 L\\/h (escenario base), los 70 vehículos requieren entre 546 L (70 × 1.3 × 6 h) y 728 L (70 × 1.3 × 8 h) al día; con 2.0 L\\/h (escenario de alta exigencia), entre 840 y 1,120 L. Ambos consumos son estimaciones internas de Ingeniería y Manufactura para el motor Rehlko CH440, no mediciones de operación.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":529,\"augmented\":0,\"text\":\"Si se incorpora la reserva de 25,049USD ($471,175), el resultado estimado sería de 29,634USD ($557,425), equivalente a un margen aproximado de 3.0%. En términos de flujo de efectivo (que incluye la retención de $16,768 USD de inventario de seguridad sin liquidar inmediatamente todas las contingencias proyectadas), el modelo prevé requerir un anticipo del 20% equivalente a 200,393USD (3,769,400 MXN), lo que permite que el requerimiento de caja mínima de la compañía baje a -2,660USD (-50,037 MXN) para el quinto mes operativo.\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":183,\"augmented\":0,\"text\":\"Concepto (USD) Fase 1 (20 unidades) Fase 2 (50 unidades) Materiales $5,157 $5,157 Mano de obra $2,875 $2,875 Overhead $2,376 $2,376 Desperdicio, retrabajo y $1,111 $1,111 contingencia\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":569,\"augmented\":0,\"text\":\"Flujos de efectivo por unidad a cinco años (MXN) Concepto (por Año 0 Año 1 Año 2 Año 3 Año 4 Año 5 VPN unidad, USD) (12%) Flujo A – Rentar Renta mensual $0 -$19,139 -$19,139 -$19,139 -$19,139 -$19,139 -$68,991 (incluye mantenimiento) Flujo B – Comprar Compra (precio -$14,314 $0 $0 $0 $0 $0 -$14,314 promedio) Mantenimiento propio $0 -$2,233 -$2,233 -$2,233 -$2,233 -$2,233 -$8,049 Total Flujo B -$14,314 -$2,233 -$2,233 -$2,233 -$2,233 -$2,233 -$22,363 Comparación (B – A) Ahorro neto de -$14,314 $16,906 $16,906 $16,906 $16,906 $16,906 $46,628 comprar vs rentar Nota:\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":571,\"augmented\":0,\"text\":\"En conclusión, la adquisición directa de la flota representa la alternativa financieramente óptima para asegurar la rentabilidad a largo plazo de las operaciones turísticas del complejo. 8. Conclusión y Cierre Estratégico Para Outdoor Adventure Holdings, la adquisición directa de la flota de representa una decisión estratégica superior frente al arrendamiento externo. Comprar asegura el control total de los activos operativos, elimina la dependencia de intermediarios o arrendadores externos y permite personalizar cada vehículo con la identidad gráfica del complejo.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":362,\"augmented\":0,\"text\":\"Bajo este modelo, el Costo Total de Propiedad (TCO) por unidad a 5 años para Outdoor Adventure Holdings se estima en 42,201USD (793,813 MXN) al sumar el precio promedio de compra (14,314USD), cinco años operativos (28,121 USD) y dos overhauls de mantenimiento ($1,914 USD en conjunto), deduciendo un valor de recuperación (residual) estimado en $2,147 USD (15%).\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":372,\"augmented\":0,\"text\":\"Costo Total de Propiedad (TCO a 5 años) Considerando una vida útil estándar de cinco años en operaciones exigentes, el costo operativo anual que absorberá el cliente por cada vehículo se proyecta en 5,624USD (105,791 MXN), que abarca $3,105 USD de combustible (estimado a 1.3 L\\/h y $1.28 USD\\/L), $2,233 USD en mantenimiento y consumibles, y $286 USD de seguro corporativo.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":216,\"augmented\":0,\"text\":\"Al comprar la unidad de Amazonas, el cliente eroga la inversión inicial promedio de $14,314 USD, pero sustituye el pago de alquiler mensual por un costo directo de mantenimiento de tan solo 186USD (3,500 MXN) al mes.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":523,\"augmented\":0,\"text\":\"Asimismo, ofrece una solución vehicular modular 2 en 1 especialmente diseñada y adaptada a la severidad del terreno de Yucatán, logrando optimizar significativamente el Costo Total de Propiedad (TCO) a lo largo de los cinco años de horizonte operativo. Los flujos de efectivo detallados, las tasas de retorno y la fundamentación financiera específica del proyecto se desarrollan y presentan a detalle en la sección financiera correspondiente. La propuesta de valor de Amazonas se fundamenta en diferenciadores contundentes:\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":312,\"augmented\":1,\"text\":\"El flujo de combustible se omite en la tabla comparativa porque representa el mismo gasto en ambos escenarios para el cliente. Al evaluar los flujos a 5 años, el proyecto evidencia que rentar generaría un costo a valor presente de $68,991 USD por unidad, mientras que comprar reduce ese costo a solo $22,363 USD.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":200,\"augmented\":1,\"text\":\"Comprar asegura el control total de los activos operativos, elimina la dependencia de intermediarios o arrendadores externos y permite personalizar cada vehículo con la identidad gráfica del complejo.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":245,\"augmented\":1,\"text\":\"El cliente obtiene un VPN favorable (ahorro real) de $46,628 USD por cada vehículo adquirido a Amazonas. Esta dinámica genera un ROI para Outdoor Adventure Holdings del 325.8% y un periodo de recuperación de capital acelerado de solo 10.2 meses.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":349,\"augmented\":1,\"text\":\"Retorno de Inversión (ROI) y Beneficio Financiero para el Cliente Para que Outdoor Aventura Holdings decida determinar la viabilidad de la adquisición frente a un modelo tradicional de alquiler, proyectamos los flujos de caja operativos descontados a valor presente neto (VPN) a 5 años (con una tasa de descuento del 12%), basados en la hipótesis de\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":235,\"augmented\":1,\"text\":\"el costo unitario base se mantiene en $11,519 USD ($216,680 MXN) en ambas fases y, al pasar de 20 a 50 unidades, el costo total crece en proporción 230,388 USD ($4,333,600 MXN en la Fase 1) y 575,970 USD ($10,834,000 MXN en la Fase 2).\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":378,\"augmented\":0,\"text\":\"una plataforma modular monoplaza única con capacidad de transformación entre huésped y trabajo en menos de 15 minutos sin intervenir la estructura primaria, soporte técnico local directo en sitio, garantía integral de 12 meses (o 1,500 horas) y la validación en campo mediante tres unidades de preserie disponibles para pruebas antes de la formalización del contrato definitivo.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":558,\"augmented\":0,\"text\":\"El flujo de combustible se omite en la tabla comparativa porque representa el mismo gasto en ambos escenarios para el cliente. Al evaluar los flujos a 5 años, el proyecto evidencia que rentar generaría un costo a valor presente de $68,991 USD por unidad, mientras que comprar reduce ese costo a solo $22,363 USD. El cliente obtiene un VPN favorable (ahorro real) de $46,628 USD por cada vehículo adquirido a Amazonas. Esta dinámica genera un ROI para Outdoor Adventure Holdings del 325.8% y un periodo de recuperación de capital acelerado de solo 10.2 meses.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":632,\"augmented\":0,\"text\":\"Costo total de la fase $230,388 $230,388 Precio base propuesto $13,025 $13,025 Con un precio base de 13,025 USD ($245,000 MXN) por unidad y variantes con margen objetivo cercano al 25%, el precio promedio ponderado estimado es de 14,314 USD (269,243 MXN); el contrato de 70 unidades genera ingresos estimados de 1,001,967USD ($18,847,000). Como el costo unitario base se mantiene en $216,680 en ambas fases, el costo base de los 70 vehículos es de 54,683USD(1,028,600 MXN), equivalente al 5.5% se calcula después de costos base, variantes, contingencia e inversión inicial, pero antes de provisionar la reserva completa de garantía.\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":586,\"augmented\":0,\"text\":\"Análisis de Costos Unitarios y Economía de Escala. El proyecto exige una inversión inicial de 73,365 USD (1,380,00 MXN) para el acondicionamiento de nuestra planta, adquisición de herramental especializado, ingeniería y programa de unidades demo. Con un criterio conservador, no se aplican reducciones por curva de aprendizaje ni economías de escala: el costo unitario base se mantiene en $11,519 USD ($216,680 MXN) en ambas fases y, al pasar de 20 a 50 unidades, el costo total crece en proporción 230,388 USD ($4,333,600 MXN en la Fase 1) y 575,970 USD ($10,834,000 MXN en la Fase 2).\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":10,\"char_len\":494,\"augmented\":0,\"text\":\"Retorno de Inversión (ROI) y Beneficio Financiero para el Cliente Para que Outdoor Aventura Holdings decida determinar la viabilidad de la adquisición frente a un modelo tradicional de alquiler, proyectamos los flujos de caja operativos descontados a valor presente neto (VPN) a 5 años (con una tasa de descuento del 12%), basados en la hipótesis de que Outdoor Adventure Holdings pagaría una renta de 1,595USD (30,000 MXN) por una unidad UTV similar (renta que cubriría su mantenimiento base).\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":265,\"augmented\":0,\"text\":\"Nuestra plataforma permite desarrollar vehículos con diseños temáticos adaptados a cada atracción, temporada o concepto. Desde recorridos inspirados en los cenotes y expediciones por la selva, hasta rutas culturales, aventuras al atardecer y experiencias nocturnas.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":393,\"augmented\":0,\"text\":\"En un complejo que combina construcción y futura operación turística, una unidad detenida puede retrasar inspecciones, mantenimiento o el traslado de herramientas. El costo no se limita a reparar: también implica reorganizar personal y tareas. STORMS parte de esa realidad operativa; nosotros adaptamos los puntos de riesgo del entorno y demostramos el resultado antes de fabricar masivamente.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":415,\"augmented\":0,\"text\":\"Unidades de movilidad capaces de responder a las Solución integral de movilidad adaptada a las exigencias de operaciones en Yucatán, donde las condiciones regionales de Yucatán, combinando condiciones ambientales y el uso constante pueden protección especializada de las unidades, provocar desgaste, fallas y tiempos de inactividad. mantenimiento preventivo, trazabilidad mediante Fleet ID y acompañamiento técnico.\"},{\"section\":\"competencia\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":561,\"augmented\":0,\"text\":\"Alternativas Funciones a rastrear Lectura comercial para OAH • Alta potencia y desempeño Mayor capacidad de trabajo Can-Am Defender HD10\\/HD11 para trabajo pesado. $553,900 MXN • Gran capacidad de carga y Referencia oficial México: Defender HD11 2027, remolque. precio desde, sin transporte ni preparación. • Referente en capacidad Referente de alta potencia, trabajo pesado y mecánica y equipamiento equipamiento. premium. Es el competidor más difícil de superar en capacidad mecánica. Una solución operativa que se demuestra en campo · CASO FICTICIO · PÁGINA 3\"},{\"section\":\"competencia\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":255,\"augmented\":1,\"text\":\"• Referente en capacidad Referente de alta potencia, trabajo pesado y mecánica y equipamiento equipamiento. premium. Es el competidor más difícil de superar en capacidad mecánica. Una solución operativa que se demuestra en campo · CASO FICTICIO · PÁGINA 3\"},{\"section\":\"competencia\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":305,\"augmented\":1,\"text\":\"Alternativas Funciones a rastrear Lectura comercial para OAH • Alta potencia y desempeño Mayor capacidad de trabajo Can-Am Defender HD10\\/HD11 para trabajo pesado. $553,900 MXN • Gran capacidad de carga y Referencia oficial México: Defender HD11 2027, remolque. precio desde, sin transporte ni preparación.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":82,\"augmented\":1,\"text\":\"mantenimiento preventivo, trazabilidad mediante Fleet ID y acompañamiento técnico.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":109,\"augmented\":1,\"text\":\"nosotros adaptamos los puntos de riesgo del entorno y demostramos el resultado antes de fabricar masivamente.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":332,\"augmented\":1,\"text\":\"Unidades de movilidad capaces de responder a las Solución integral de movilidad adaptada a las exigencias de operaciones en Yucatán, donde las condiciones regionales de Yucatán, combinando condiciones ambientales y el uso constante pueden protección especializada de las unidades, provocar desgaste, fallas y tiempos de inactividad.\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":534,\"augmented\":0,\"text\":\"Cruce Acción prioritaria Convertir OAH en plataforma de validación y expansión comercial. Demostrar el desempeño de las FO unidades, documentar resultados e integrar Fleet ID y soporte como diferenciadores para abrir mercado B2B y turístico. Construir un ecosistema de proveedores y aliados estratégicos. Asegurar componentes críticos, capacidad DO de servicio técnico y acuerdos comerciales que permitan crecer sin inversiones prematuras en infraestructura. Una solución operativa que se demuestra en campo · CASO FICTICIO · PÁGINA 4\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":304,\"augmented\":1,\"text\":\"Cruce Acción prioritaria Convertir OAH en plataforma de validación y expansión comercial. Demostrar el desempeño de las FO unidades, documentar resultados e integrar Fleet ID y soporte como diferenciadores para abrir mercado B2B y turístico. Construir un ecosistema de proveedores y aliados estratégicos.\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":229,\"augmented\":1,\"text\":\"Asegurar componentes críticos, capacidad DO de servicio técnico y acuerdos comerciales que permitan crecer sin inversiones prematuras en infraestructura. Una solución operativa que se demuestra en campo · CASO FICTICIO · PÁGINA 4\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":251,\"augmented\":0,\"text\":\"Revisar frenos, dirección, estructura, retención del Inspeccionar admisión por polvo, humedad, temperatura operador y respuesta en las tareas acordadas. de operación, corrosión, conexiones y desgaste de Una falla crítica detiene la prueba. neumáticos.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":383,\"augmented\":0,\"text\":\"STORMS propone una venta directa B2B basada en diagnóstico, configuración y evidencia. La conversación con Outdoor Adventure Holdings no comienza con una ficha técnica: comienza por entender qué tareas son críticas, qué sucede cuando una unidad se detiene y cómo se reconocerá una mejora. Con esa información se define una prueba en campo que permita decidir con menor incertidumbre.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":288,\"augmented\":1,\"text\":\"STORMS propone una venta directa B2B basada en diagnóstico, configuración y evidencia. La conversación con Outdoor Adventure Holdings no comienza con una ficha técnica: comienza por entender qué tareas son críticas, qué sucede cuando una unidad se detiene y cómo se reconocerá una mejora.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":105,\"augmented\":0,\"text\":\"Dato reportado Estimación incremental Subtotal preliminar Escenario académico 30.9% antes de otros gastos\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":279,\"augmented\":0,\"text\":\"Escenario ficticio de planeación en MXN. El eBOM reporta $23,750 MXN como costo histórico del prototipo. El kit, el precio de venta, los costos de repetición y el ROI son supuestos para evaluar la viabilidad; no son cotizaciones confirmadas ni resultados comerciales demostrados.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":190,\"augmented\":0,\"text\":\"Lote1(5u) Gas ¢ ' Lote2(5u) Gun ¢ H Lote 3(5u) Gun ¢ ! Lote 4 (5u) GED e : Semanas desde el inicio de construcción Una solución operativa que se demuestra en campo - CASO FICTICIO - PÁGINA 8\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":563,\"augmented\":0,\"text\":\"depreciación INcluve filtros, lubricantes, inspecciones y Depreciación Yi A > okies el consumibles menores; excluye ees lineal de $8,320\\/año por unidad. Mantenimiento: $2900\\/año por : le mantenimiento combustible, neumáticos mayores y unidad; $58,000 para 20. ° * reparaciones extraordinarias. No fabricar ni registrar ingreso hasta que Expansión opcional: hasta Al costo adaptado actual: $1,797,500 de costo directo indicativo, antes OAH ejerza la opción y exista orden 50 unidades de economías de escala. aprobada, cotizaciones actualizadas y anticipo recibido.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":585,\"augmented\":0,\"text\":\"Resultado y RoIsrorms — $221,000 - $45,000 - $71,900 = $204.00; ROI ilustrativo de 24.4% de gastos fijos, nómina no incluida, sobre $835,900. administración, impuestos, — seguros, garantía y financiamiento. : . - RO! simple a 5 años =181%, incluyendo Retorno y recuperación de Pent? evitada supuesta: $2,500\\/mes; menos mantenimiento de cidual supuesto de $10,400. Excluye $2900\\/año = $27,100 netos\\/año. Recuperación simple: cerca de 23 : OAH combustible, operadores, — seguros, meses ‘ , . impuestos, fallas y tiempo detenido. oe Vida contable supuesta: 5 años; residual 20% ($10,400);\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":293,\"augmented\":1,\"text\":\"depreciación INcluve filtros, lubricantes, inspecciones y Depreciación Yi A > okies el consumibles menores; excluye ees lineal de $8,320\\/año por unidad. Mantenimiento: $2900\\/año por : le mantenimiento combustible, neumáticos mayores y unidad; $58,000 para 20. ° * reparaciones extraordinarias.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":272,\"augmented\":1,\"text\":\"menos mantenimiento de cidual supuesto de $10,400. Excluye $2900\\/año = $27,100 netos\\/año. Recuperación simple: cerca de 23 : OAH combustible, operadores, — seguros, meses ‘ , . impuestos, fallas y tiempo detenido. oe Vida contable supuesta: 5 años; residual 20% ($10,400);\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":269,\"augmented\":1,\"text\":\"No fabricar ni registrar ingreso hasta que Expansión opcional: hasta Al costo adaptado actual: $1,797,500 de costo directo indicativo, antes OAH ejerza la opción y exista orden 50 unidades de economías de escala. aprobada, cotizaciones actualizadas y anticipo recibido.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":590,\"augmented\":0,\"text\":\"We present a strategic mobility solution and an on-site support ecosystem engineered to shield your operations against downtime and transform a traditional operating expense into a high-yield asset. Our proposal addresses your operational needs through four strategic pillars: 1. Two-Phase Strategic Deployment: We supply a single standardized platform delivered in two structured stages: 20 units with the Work & Utility Kit in Month 6 for construction logistics, and 50 units with the Eco-Tour Comfort Kit in Month 18 calibrated specifically for guest experience 2. Downtime Minimization:\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":525,\"augmented\":0,\"text\":\"For the attention of the Procurement Committee and Project Directorate of Outdoor Adventure Holdings: Developing a luxury resort complex spanning 250,000 hectares of Yucatán jungle presents a monumental logistical challenge. In a project of this scale, a broken-down vehicle is not merely a mechanical failure; it is a direct operational bottleneck that stalls civil construction, incurs hidden costs, and jeopardizes the resort's official opening date. At Racing Team, we are not here to sell you a simple fleet of vehicles.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":427,\"augmented\":0,\"text\":\"We provide a dedicated 24\\/7 technical hotline to handle inquiries, coordinate preventive service, and dispatch our on-site team for immediate field resolution 3. Tourism Profit Center (ROI): We transform the fleet into the flagship \\\"Yucatán Off-Road Eco-Expedition\\\" tour. Under conservative, baseline, and peak occupancy models, guided tours fully amortize the initial capital investment 4. Minimal Ecological Impact Operation:\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":214,\"augmented\":1,\"text\":\"it is a direct operational bottleneck that stalls civil construction, incurs hidden costs, and jeopardizes the resort's official opening date. At Racing Team, we are not here to sell you a simple fleet of vehicles.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":155,\"augmented\":1,\"text\":\"Under conservative, baseline, and peak occupancy models, guided tours fully amortize the initial capital investment 4. Minimal Ecological Impact Operation:\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":313,\"augmented\":1,\"text\":\"Therefore, we conclude our technical proposal with a direct operational challenge: we offer to transport two prototypes directly to Yucatán for an exhaustive 14-day Test Drive. Let us prove on your terrain, under your load and climate conditions, that is the ultimate logistical partner for your resort's success.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":136,\"augmented\":0,\"text\":\"Let us prove on your terrain, under your load and climate conditions, that is the ultimate logistical partner for your resort's success.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":526,\"augmented\":0,\"text\":\"We protect the natural reserve through a lightweight single-seater architecture (~270 kg) that prevents soil compaction, equipped with a certified spark arrestor, low-decibel exhaust (<75 dB), and spill containment trays ensuring zero fluid leaks in the workshop We understand an investment of this magnitude requires absolute operational certainty. Therefore, we conclude our technical proposal with a direct operational challenge: we offer to transport two prototypes directly to Yucatán for an exhaustive 14-day Test Drive.\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":640,\"augmented\":0,\"text\":\"Technical Differentiation and Cost Allocation Our proposal is built upon a Single Standardized Platform (85% to 90% identical components across chassis, engine, and braking systems), significantly simplifying spare parts inventory in the reserve. To optimize Outdoor Adventure Holdings' budget, we tailor the CVT transmission and secondary components according to each phase's operational demands: ● Phase 1: Work & Utility ($18,500 USD — 20 Units): Engineered for the physical demands of civil construction. Features structural reinforcement for heavy cargo\\/towing, high-torque CVT calibration, and heavy-duty utility accessories. Phase 2:\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":218,\"augmented\":0,\"text\":\"Eco-Tour Comfort ($15,000 USD — 50 Units): Units optimized for guest comfort. The unit price reflects a 24% corporate volume discount (50 units) and focuses on smooth acceleration, rider safety, and terrain protection.\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":350,\"augmented\":1,\"text\":\"Features structural reinforcement for heavy cargo\\/towing, high-torque CVT calibration, and heavy-duty utility accessories. Phase 2: Eco-Tour Comfort ($15,000 USD — 50 Units): Units optimized for guest comfort. The unit price reflects a 24% corporate volume discount (50 units) and focuses on smooth acceleration, rider safety, and terrain protection.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":525,\"augmented\":0,\"text\":\"Staged Delivery Plan (20 + 50 Units) We structure production into two strategic logistical stages to guarantee a continuous supply to Outdoor Adventure Holdings, ensuring resources are deployed optimally to support both civil infrastructure and guest experience: ● Phase 1 (Month 6 — 20 Units): Delivery of the initial 20 units equipped for civil construction support, providing agile single-seat mobility for field engineers, site technicians, and light tool logistics across rugged terrain. ● Phase 2 (Month 18 — 50 Units):\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":252,\"augmented\":0,\"text\":\"Supply Chain and Local Suppliers in Torreón, Coahuila By operating from Torreón, Coahuila, leverages the industrial strength and automotive manufacturing ecosystem of northern Mexico, backed by strategic dual-sourcing agreements and local buffer stock:\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":579,\"augmented\":0,\"text\":\"Production line scaling to deliver the remaining 50 units configured with the Eco-Tour Comfort Kit, calibrated specifically for commercial resort operations and guest comfort along guided expedition trails. Shipment from our manufacturing facility in Torreón, Coahuila, to Yucatán is executed via consolidated overland freight (car-hauler \\/ flatbed trailer). The initial Phase 1 batch (20 units × 270 kg = ~5,400 kg net load \\/ 5.4 metric tons) is transported in a single consolidated shipment, optimizing shipping costs and minimizing the overall carbon footprint of the project.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":206,\"augmented\":1,\"text\":\"Production line scaling to deliver the remaining 50 units configured with the Eco-Tour Comfort Kit, calibrated specifically for commercial resort operations and guest comfort along guided expedition trails.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":294,\"augmented\":1,\"text\":\"Staged Delivery Plan (20 + 50 Units) We structure production into two strategic logistical stages to guarantee a continuous supply to Outdoor Adventure Holdings, ensuring resources are deployed optimally to support both civil infrastructure and guest experience: ● Phase 1 (Month 6 — 20 Units):\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":220,\"augmented\":1,\"text\":\"The initial Phase 1 batch (20 units × 270 kg = ~5,400 kg net load \\/ 5.4 metric tons) is transported in a single consolidated shipment, optimizing shipping costs and minimizing the overall carbon footprint of the project.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":151,\"augmented\":1,\"text\":\"Shipment from our manufacturing facility in Torreón, Coahuila, to Yucatán is executed via consolidated overland freight (car-hauler \\/ flatbed trailer).\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":392,\"augmented\":0,\"text\":\"Acquiring an operating fleet for a 250,000-hectare development represents a critical capital expenditure (CAPEX). We have structured our pricing based on a Modular Platform philosophy: a standardized central chassis, but with rigorously separated technical specifications for heavy construction and ecotourism, ensuring you pay solely for the attributes required in each phase of the project.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":386,\"augmented\":0,\"text\":\"The unit price of $15,000 USD for Phase 2 includes a 24% corporate volume discount (50 units) and reflects hardware optimization by not requiring a hitch or heavy-duty metal cargo bed. Suggested Financing Plans To align the hotel project's cash flow with the physical receipt of assets, proposes two direct payment schemes, eliminating financial intermediaries and excessive bank rates.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":10,\"char_len\":267,\"augmented\":0,\"text\":\"Outdoor Adventure Holdings - Yucatán Project Sign Date: _____ \\/ _____ \\/ _____ ___________________________ Corporate Representation and General Management: Racing Team – Universidad Tecmilenio Campus Laguna Sign Date: _____ \\/ _____ \\/ _____ ____________________________\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":194,\"augmented\":0,\"text\":\"Reserva de garantía y posventa $420,000 (supuesto) Administración comercial del proyecto $180,000 (18 meses, supuesto) Resultado operativo indicativo $1,727,710 antes de impuestos y financiación\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":1,\"char_len\":544,\"augmented\":0,\"text\":\"· Propuesta de suministro de 70 vehículos Baja monoplaza RELACIÓN COMERCIAL: Outdoor Adventure Holdings es el cliente comprador, responsable de evaluar las propuestas recibidas y seleccionar la mejor oferta. es un proveedor independiente que compite por obtener el contrato de suministro de 70 vehículos Baja monoplaza. Horizonte: 18 meses desde el inicio previsto de la construcción del proyecto del comprador. Moneda: pesos mexicanos (MXN). Documento de planeación, elaborado con datos de ficha técnica y supuestos expresamente identificados.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":1,\"char_len\":341,\"augmented\":1,\"text\":\"Concepto 20 unidades 50 unidades 70 unidades Ingresos $3,400,000 $8,500,000 $11,900,000 Costo de fabricación $2,494,940 $6,237,350 $8,732,290 Margen bruto $905,060 $2,262,650 $3,167,710 Logística y exportación $240,000 $600,000 $840,000 Reserva de garantía $120,000 $300,000 $420,000 Contribución después de servicio y entrega Administración\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":1,\"char_len\":355,\"augmented\":0,\"text\":\"Excedente de caja proyectado al mes 18 $1,227,710 antes de impuestos\\/financiación Interpretación: el excedente de caja es el resultado de cobros menos desembolsos modelados, incluida la inversión inicial y la reserva de garantía. NO equivale a utilidad neta contable. No incorpora impuestos, intereses, fluctuación cambiaria ni contingencias regulatorias.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":1,\"char_len\":612,\"augmented\":0,\"text\":\"Concepto 20 unidades 50 unidades 70 unidades Ingresos $3,400,000 $8,500,000 $11,900,000 Costo de fabricación $2,494,940 $6,237,350 $8,732,290 Margen bruto $905,060 $2,262,650 $3,167,710 Logística y exportación $240,000 $600,000 $840,000 Reserva de garantía $120,000 $300,000 $420,000 Contribución después de servicio y entrega Administración adicional del periodo: $180,000. Resultado operativo indicativo de los 18 meses: $1,727,710. La inversión fija de $500,000 se trata como salida de efectivo de inversión, NO como gasto operativo; la utilidad contable futura dependerá de depreciación y criterios fiscales.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":124,\"augmented\":0,\"text\":\"La propuesta también requiere demostrar conformidad técnica, plazos de producción y condiciones de garantía antes de firmar.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":312,\"augmented\":1,\"text\":\"Primer lote: 20 vehículos, con fecha objetivo de disponibilidad en mes 6. Segundo lote: 50 vehículos adicionales, con fecha objetivo en mes 18. Los anticipos se consideran pagos por compraventa, nunca aportaciones patrimoniales del comprador. Hito Mes Cobro estimado Lote 1: anticipo del 40% 1 $1,360,000 Lote 1:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":596,\"augmented\":0,\"text\":\"Primer lote: 20 vehículos, con fecha objetivo de disponibilidad en mes 6. Segundo lote: 50 vehículos adicionales, con fecha objetivo en mes 18. Los anticipos se consideran pagos por compraventa, nunca aportaciones patrimoniales del comprador. Hito Mes Cobro estimado Lote 1: anticipo del 40% 1 $1,360,000 Lote 1: inspección del 40% 5 $1,360,000 Lote 1: entrega del 20% 6 $680,000 Lote 2: anticipo del 40% 7 $3,400,000 Lote 2: inspección del 40% 17 $3,400,000 Lote 2: aceptación del 20% 18 $1,700,000 Total $11,900,000 La cobranza por hitos es negociable y podría no ser aceptada por el comprador.\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":3,\"char_len\":629,\"augmented\":0,\"text\":\"Contribución unitaria después de fabricación, logística y garantía: $27,253. Para evaluar recuperación del capital inicial se consideran costos comprometidos de $680,000 ($500,000 de inversión más $180,000 de administración adicional). Punto de recuperación simple: 25 vehículos = redondeo hacia arriba de $680,000 ÷ $27,253. Esta aproximación no sustituye un punto de equilibrio contable: incluye inversión de capital y no contempla depreciación ni impuestos. Escenario Precio\\/u Costo fab.\\/u Contribución 70 u. Caja final* Conservador: precio - 10%; costo +10% Base $170,000 $124,747 $1,907,710 $1,227,710 Favorable: precio +5%;\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":3,\"char_len\":111,\"augmented\":0,\"text\":\"la reserva de garantía de $420,000 es la provisión comercial interna de . Estos conceptos no deben confundirse.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":3,\"char_len\":614,\"augmented\":0,\"text\":\"Perspectiva del comprador Estimación o situación Compra de 70 unidades $11,900,000 antes de IVA\\/importación aplicable Mantenimiento anual de la flota $840,000 (supuesto de $12,000 por unidad\\/año) Combustible, almacenamiento, personal y pérdida por paros Por determinar con horas reales de uso Pruebas de manejo y confiabilidad Pendientes; prototipo aún no concluido No cuantificable sin tarifa, duración y alcance del arrendamiento Ahorro frente a renta comparable El mantenimiento anual proyectado de $840,000 pertenece a los costos operativos futuros del COMPRADOR y no se descuenta de las ventas del FABRICANTE;\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":3,\"char_len\":556,\"augmented\":0,\"text\":\"• Validar prototipo, durabilidad, seguridad, calidad de soldaduras y capacidad de suministro antes de oferta vinculante. • Completar lista de materiales y revisar compatibilidad entre diferencial, eje trasero, suspensión independiente y transmisión. • Obtener cotizaciones reales de exportación, transporte a Montana, seguros, impuestos y cumplimiento normativo. • Medir horas de fabricación y demostrar capacidad de 4–5 unidades mensuales, con plan de contingencia. • Negociar pagos por hitos; anticipos de compra no representan asociación con el cliente.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":3,\"char_len\":219,\"augmented\":0,\"text\":\"• Definir riesgo cambiario si el contrato se cotiza o cobra en dólares y los costos se pagan en pesos. • Incorporar impuestos, depreciación y financiamiento en una versión posterior antes de aprobar precios definitivos.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":3,\"char_len\":306,\"augmented\":1,\"text\":\"• Obtener cotizaciones reales de exportación, transporte a Montana, seguros, impuestos y cumplimiento normativo. • Medir horas de fabricación y demostrar capacidad de 4–5 unidades mensuales, con plan de contingencia. • Negociar pagos por hitos; anticipos de compra no representan asociación con el cliente.\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":3,\"char_len\":303,\"augmented\":1,\"text\":\"Esta aproximación no sustituye un punto de equilibrio contable: incluye inversión de capital y no contempla depreciación ni impuestos. Escenario Precio\\/u Costo fab.\\/u Contribución 70 u. Caja final* Conservador: precio - 10%; costo +10% Base $170,000 $124,747 $1,907,710 $1,227,710 Favorable: precio +5%;\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":3,\"char_len\":249,\"augmented\":1,\"text\":\"• Validar prototipo, durabilidad, seguridad, calidad de soldaduras y capacidad de suministro antes de oferta vinculante. • Completar lista de materiales y revisar compatibilidad entre diferencial, eje trasero, suspensión independiente y transmisión.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":3,\"char_len\":332,\"augmented\":0,\"text\":\"Outdoor Adventure Holdings busca la mejor oferta global. La comparación debe considerar el precio de adquisición, costos de mantenimiento, combustible, refacciones, formación de operadores, transporte, tiempo de inactividad y capacidad comprobada de entrega. Una propuesta de precio menor no necesariamente ofrece menor costo total.\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":3,\"char_len\":269,\"augmented\":0,\"text\":\"costo -5% * Caja final simplificada después de $500,000 de inversión y $180,000 de administración, bajo cobro total de las 70 unidades. Sin impuesto, interés, diferencias cambiarias o otras contingencias. Un escenario desfavorable puede volver no rentable la propuesta.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":190,\"augmented\":0,\"text\":\"The MR26 is the starting point. The first prototype for OAH is built on the current design; after the client test drives, modifications are made to personalize the vehicle to OAH operations.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":161,\"augmented\":1,\"text\":\"capacity Approximately 350 kg Dimensions (L x W x H) 1.95m x 0.77m x 1.32m Center of gravity X=30.85mm Y=297.80mm Safety Roll cage compliant with Baja SAE rules;\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":580,\"augmented\":0,\"text\":\"Parameter MR26 baseline Engine Kohler Command Pro CH440, 14 hp Drivetrain 4x4 with reduction gearbox Mass Approximately 213 kg (without driver) Top speed Approximately 55 km\\/h Endurance Approximately 4 hours in extreme-terrain endurance racing Occupants 1 Towing 2.7 Tons Demonstrated pulling a John Deere tractor (in neutral) and a Ford Lobo Cargo capacity Approximately 350 kg Dimensions (L x W x H) 1.95m x 0.77m x 1.32m Center of gravity X=30.85mm Y=297.80mm Safety Roll cage compliant with Baja SAE rules; 4-point harness; kill switch Figure 2. CAD and general specifications\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":498,\"augmented\":0,\"text\":\"review of CVT and engine cooling ducting for high ambient temperature Intensive use, minimal Maintenance burden Reduced service points, accessible layout, downtime simple procedures, spare-parts kit Safety Guest and staff protection Baja SAE roll cage, 4-point harness Note on cargo capacity: the baseline MR26 was not designed as a cargo vehicle. We propose a rated payload for the cargo platform of 350kg, to be confirmed by FEA and a loaded field test before it appears in any commitment to OAH.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":458,\"augmented\":0,\"text\":\"Sea-front, humidity, Corrosion, electrical faults Stainless-steel or coated sheet for firewall and water panels; full anti-corrosion paint; sealed connectors; stainless fasteners Sand Intake and bearing wear Improved air pre-filtration; sealed bearings and boots; filtered and sealed CVT intake and ducting against sand and water High temperature Driver heat from engine, cooling Seat materials and heat barrier that insulate the load driver from the engine;\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":263,\"augmented\":1,\"text\":\"Sea-front, humidity, Corrosion, electrical faults Stainless-steel or coated sheet for firewall and water panels; full anti-corrosion paint; sealed connectors; stainless fasteners Sand Intake and bearing wear Improved air pre-filtration; sealed bearings and boots;\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":194,\"augmented\":1,\"text\":\"filtered and sealed CVT intake and ducting against sand and water High temperature Driver heat from engine, cooling Seat materials and heat barrier that insulate the load driver from the engine;\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":150,\"augmented\":1,\"text\":\"We propose a rated payload for the cargo platform of 350kg, to be confirmed by FEA and a loaded field test before it appears in any commitment to OAH.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":533,\"augmented\":0,\"text\":\"High-Density Storage & Service Center To support the operational fleet efficiently, we propose a purpose-built facility designed to minimize the building footprint while maximizing capacity. This design implements a three- level industrial racking system allows us to house the fleet within a compact 240 m² structure, preserving valuable land for resort amenities. Facility Specifications: • Main Warehouse (10m x 24m ): Houses the three-level vehicle rack system, optimizing horizontal space. • Integrated Service Center (5m x 8m :\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":627,\"augmented\":0,\"text\":\"Item Batch 1: 20 units Batch 2: 50 units Required by Month 6 after construction start Month 18 after construction start Production window Months 2 to 6 (about 5 units per month) Months 12 to 18 (about 8 units per month) Main stations Frame welding on fixtures; sub- Same cell, with a second shift or a assembly (suspension, drivetrain); final duplicated fixture set assembly; test and quality control Labor per unit 140 labor-hours 108 labor-hours, reduced by learning curve Purchased parts Engine, kill switch, suspension, screws Order at month 9 to 10; and bolts, Gas Tank, etc; order at month 0 to 1 Infrastructure Proposal:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":353,\"augmented\":0,\"text\":\"Madrams manufactures nearly everything in-house, which lets us control quality, cost and schedule. Only highly specific components, or components required by regulation, are purchased: engine, kill switch, suspension and similar items . Repeatability is achieved with fixtures and defined work instructions at every station. Figure 5. Example of fixture\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":125,\"augmented\":0,\"text\":\"An attached dedicated workspace for the maintenance team, equipped for the service tasks outlined in the maintenance program.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":126,\"augmented\":0,\"text\":\"The total contract value is USD 835,700 for 70 vehicles, with an estimated gross margin of about USD 100,700 (13.7 % on cost).\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":639,\"augmented\":0,\"text\":\"Assembly, testing and quality-control labor is 50 hours per unit in Batch 1 and 38.6 hours in Batch 2 (90 % learning curve), at a fully loaded rate of USD 16 per hour. Fabrication labor (about 90 hours per unit, USD 1,440) is already inside the USD 2,163.53 of manufacturing processes in the cost report. Overhead covers facilities, tooling upkeep, training and logistics. The adaptation line is an estimate (corrosion-resistant sheet and coatings, sealing, seat heat barrier, hitch, cargo rack and standard mounts) to be confirmed with supplier quotations. To reward the volume commitment, the 50-vehicle lot is offered at a 3 % discount.\"},{\"section\":\"competencia\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":321,\"augmented\":0,\"text\":\"It also comes with operator and maintenance training, an on-site service center, accessories for each OAH activity and customization after the test drives, which catalog vehicles do not include. Competitor prices are 2026 US MSRP from manufacturer and dealer listings, for single units, excluding freight, taxes and fees.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":304,\"augmented\":1,\"text\":\"Assembly, testing and quality-control labor is 50 hours per unit in Batch 1 and 38.6 hours in Batch 2 (90 % learning curve), at a fully loaded rate of USD 16 per hour. Fabrication labor (about 90 hours per unit, USD 1,440) is already inside the USD 2,163.53 of manufacturing processes in the cost report.\"},{\"section\":\"competencia\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":167,\"augmented\":1,\"text\":\"In exchange, the MR26 carries about twice the cargo of either competitor and has demonstrated towing of 2.7 tons, the capability that OAH’s construction phase demands.\"},{\"section\":\"competencia\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":589,\"augmented\":0,\"text\":\"Vehicle Seats Engine Cargo \\/ towing Price (USD) Madrams MR26, OAH 1 Kohler CH440, 350 kg tested \\/ 2.7 t 12,200 (11,834 in version 14 hp demonstrated the 50-unit lot) Kawasaki Mule SX 4x4 2 401 cc 181 kg \\/ 499 kg 9,999 Kawasaki Mule SX 4x4 XC 2 401 cc 181 kg \\/ 499 kg 11,549 Our price is 22 % above the Mule SX 4x4, 11 % above the Pioneer 520 and 6 % above the Mule SX 4x4 XC LE (18 %, 8 % and 2 % for the discounted lot). In exchange, the MR26 carries about twice the cargo of either competitor and has demonstrated towing of 2.7 tons, the capability that OAH’s construction phase demands.\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":9,\"char_len\":534,\"augmented\":0,\"text\":\"Buying the fleet removes vehicle rental during construction and lowers long-term operating cost. With an assumed rental saving of USD 1,200 and an operating cost of USD 150 per vehicle per month, the net monthly saving is USD 1,050 and the return is: For the discounted lot (USD 11,834) the payback is 11.3 months and the 3-year ROI is 219 %. The rental and operating figures are assumptions and will be replaced by OAH’s own data in the final quotation. For Madrams, the contract yields a gross margin of 13.7 % on cost. Sensitivity:\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":9,\"char_len\":226,\"augmented\":0,\"text\":\"a 10 % increase in the cost of purchased parts and materials raises the unit cost by about USD 866 and lowers the Batch 1 margin from 14.5 % to 5.9 % if the price is held. Passing it through would raise the price by about 8 %.\"},{\"section\":\"mercado\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":626,\"augmented\":0,\"text\":\"Este tipo de proyectos también genera nuevas necesidades de movilidad, mantenimiento y operación en terrenos de gran extensión. Según un estudio de Next Move Strategy Consulting (2026), el mercado de vehículos ATV y UTV en México presenta una proyección de crecimiento anual del 4.5% entre 2024 y 2030. Esta tendencia refleja oportunidades para vehículos todoterreno en actividades recreativas y operativas. Para Outdoor Adventure Holdings, su incorporación puede facilitar el desplazamiento en terrenos difíciles, apoyar las operaciones del complejo y abrir nuevas oportunidades comerciales mediante experiencias de aventura.\"},{\"section\":\"mercado\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":617,\"augmented\":0,\"text\":\"El turismo representa uno de los sectores estratégicos para la economía mexicana. De acuerdo con el INEGI (2025), durante 2024 este sector aportó el 8.7% del Producto Interno Bruto nacional, favoreciendo el desarrollo de nuevas experiencias enfocadas en la naturaleza y la aventura. En este panorama, Yucatán destaca por su riqueza natural, su oferta hotelera y su infraestructura de transporte, que facilitan la llegada de visitantes nacionales e internacionales. Estas características generan oportunidades para el desarrollo de complejos turísticos que combinan naturaleza, recreación y experiencias diferenciadas.\"},{\"section\":\"mercado\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":323,\"augmented\":1,\"text\":\"Esta tendencia refleja oportunidades para vehículos todoterreno en actividades recreativas y operativas. Para Outdoor Adventure Holdings, su incorporación puede facilitar el desplazamiento en terrenos difíciles, apoyar las operaciones del complejo y abrir nuevas oportunidades comerciales mediante experiencias de aventura.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":116,\"augmented\":0,\"text\":\"Control Preciso En Senderos Desde equipo de ciclismo y pesca hasta herramientas y Facilita el recorrido por senderos\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":579,\"augmented\":0,\"text\":\"BALTEK está diseñado para facilitar la movilidad de los huéspedes y del personal de Outdoor Su sistema híbrido busca combinar autonomía, eficiencia Adventure Holdings, con un manejo intuitivo y capacidad para desplazarse por una amplia energética y menores emisiones, ofreciendo una solución variedad de terrenos. versátil tanto para las experiencias turísticas como para las operaciones diarias del complejo. BALTEK mantiene una configuración única, con un área de carga trasera diseñada para transportar equipo y suministros destinados a las distintas actividades del complejo.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":327,\"augmented\":1,\"text\":\"naturales y caminos irregulares, Su capacidad de carga y remolque suministros para senderismo, su diseño permite atender favoreciendo una experiencia facilita el traslado de materiales y cómoda e inmersiva. diferentes necesidades turísticas y operativas sin requerir suministros para labores de construcción y operación diaria.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":265,\"augmented\":1,\"text\":\"versátil tanto para las experiencias turísticas como para las operaciones diarias del complejo. BALTEK mantiene una configuración única, con un área de carga trasera diseñada para transportar equipo y suministros destinados a las distintas actividades del complejo.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":313,\"augmented\":1,\"text\":\"BALTEK está diseñado para facilitar la movilidad de los huéspedes y del personal de Outdoor Su sistema híbrido busca combinar autonomía, eficiencia Adventure Holdings, con un manejo intuitivo y capacidad para desplazarse por una amplia energética y menores emisiones, ofreciendo una solución variedad de terrenos.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":376,\"augmented\":0,\"text\":\"naturales y caminos irregulares, Su capacidad de carga y remolque suministros para senderismo, su diseño permite atender favoreciendo una experiencia facilita el traslado de materiales y cómoda e inmersiva. diferentes necesidades turísticas y operativas sin requerir suministros para labores de construcción y operación diaria. modificaciones en la configuración del vehículo.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":6,\"char_len\":186,\"augmented\":0,\"text\":\"Población: Tasa de Empleo: Automotrices: Económicamente Activa: Directa: Calificados: $33,4 B USD 2,859,767 $3 ,000 M USD 45.3% 6 PROCESO DE Eficiencia de Producción y Tiempos de Entrega\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":639,\"augmented\":0,\"text\":\"la documentación técnica exigida por el sector calidad. industrial y automotriz. La planta opera bajo un Almacenamiento y Control de Pruebas en Banco y Validación sistema de flujo continuo que permite la Inventario Evaluaciones funcionales de arranque, fabricación simultánea de hasta tres vehículos en Gestión de inventario por componentes, transmisión y frenado. Validación de sistemas diferentes etapas. Bajo condiciones estándar, la condiciones ambientales controladas para de comunicación, sensores, componentes planta alcanza una capacidad mensual de 12 a 15 piezas sensibles como sensores, arneses y eléctricos y desempeño dinámico.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":635,\"augmented\":0,\"text\":\"Desde inventario: Tiempo de entrega = 1 a 2 semanas Frecuencia: 1 vehículo cada 1 a 3 días. Nuestro proceso de producción integra Recepción e Inspección de 5 Ensamble Final metodologías formales de calidad como AMEF, 1 Materia Prima Línea modular con puntos de control en control estadístico de procesos, certificaciones Verificación de lotes de entrada, controles proceso: torque controlado, pruebas de fuga, (como IATF 16949, ISO 14001 y la ISO 9001) y dimensionales, revisión de documentación integración electrónica y verificación trazabilidad completa de lotes, respaldadas por técnica y validación de certificados de dimensional.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":313,\"augmented\":0,\"text\":\"proveedores certificados Tier-1 y Tier-2 en la región del Bajío (incluyendo a Martinrea, Nissin, Honda, Yamaha y Dassault Systèmes), lo que Nuestra planta cuenta con un equipo multidisciplinario (técnico, asegura el suministro constante de componentes industrial y administrativo) diseñado para escalarse según el\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":478,\"augmented\":0,\"text\":\"mientras que la distribución de las instalaciones y Varianza Coordinación de recepción con el cliente, el modelo de personal garantizan un crecimiento Producción y control de calidad de procedimientos de conformidad e elástico sin comprometer la calidad. Finalmente, subsistemas (suspensión, tren motriz, frenos integración en su cadena logística. trabajamos con una red consolidada de y arneses). Aquí se introduce la varianza del modelo base vs. variantes sin frenar la línea.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":313,\"augmented\":1,\"text\":\"Transporte en su totalidad desde la dureza y resistencia del Manufactura de chasis tubular (doblado, Aseguramiento del vehículo, gestión de chasis y sus componentes estructurales hasta la soldadura estructural y fluidos, preparación para transporte validación del producto final. Asimismo, pintura\\/recubrimiento).\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":347,\"augmented\":1,\"text\":\"mientras que la distribución de las instalaciones y Varianza Coordinación de recepción con el cliente, el modelo de personal garantizan un crecimiento Producción y control de calidad de procedimientos de conformidad e elástico sin comprometer la calidad. Finalmente, subsistemas (suspensión, tren motriz, frenos integración en su cadena logística.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":246,\"augmented\":1,\"text\":\"Aquí mantenemos el terrestre\\/aéreo y compilación de kits de 80% de la plataforma estandarizada. contamos con un protocolo de producción repuestos, manuales y documentación. acelerada para responder a picos de demanda, Subensamble y Posposición de\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":325,\"augmented\":1,\"text\":\"La planta opera bajo un Almacenamiento y Control de Pruebas en Banco y Validación sistema de flujo continuo que permite la Inventario Evaluaciones funcionales de arranque, fabricación simultánea de hasta tres vehículos en Gestión de inventario por componentes, transmisión y frenado. Validación de sistemas diferentes etapas.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":91,\"augmented\":1,\"text\":\"Desde inventario: Tiempo de entrega = 1 a 2 semanas Frecuencia: 1 vehículo cada 1 a 3 días.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":123,\"augmented\":0,\"text\":\"unidades completas, respaldada por un baterías. protocolo de pruebas que evalúan cada vehículo 3 Empaque y Preparación para\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":560,\"augmented\":0,\"text\":\"Transporte en su totalidad desde la dureza y resistencia del Manufactura de chasis tubular (doblado, Aseguramiento del vehículo, gestión de chasis y sus componentes estructurales hasta la soldadura estructural y fluidos, preparación para transporte validación del producto final. Asimismo, pintura\\/recubrimiento). Aquí mantenemos el terrestre\\/aéreo y compilación de kits de 80% de la plataforma estandarizada. contamos con un protocolo de producción repuestos, manuales y documentación. acelerada para responder a picos de demanda, Subensamble y Posposición de\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":291,\"augmented\":0,\"text\":\"Empresa emergente de la región Saltillo–Arteaga, Coahuila, con taller y herramientas disponibles, equipo de 10 integrantes y primer prototipo en construcción. Proveedor independiente Cliente: Outdoor Adventure Holdings Propuesta competitiva de suministro, no asociación comercial ni sociedad\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":4,\"char_len\":122,\"augmented\":0,\"text\":\"El desempeño financiero histórico no está disponible. Todas las cifras mostradas son estimaciones, no ingresos realizados.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":209,\"augmented\":0,\"text\":\"Conservar ingeniería, montaje y control de calidad internos; recurrir a proveedores de corte CNC y rolado especializado. Validar 1–2 unidades\\/mes y escalar a 4–5 unidades\\/mes. Es meta, no capacidad comprobada.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":187,\"augmented\":0,\"text\":\"Transmisión y asociados Chasis y estructura Dirección y diferencial Rines, neumáticos y mazas Cotizar y reservar materiales críticos antes de producir cada lote; usar pedidos programados.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":8,\"char_len\":148,\"augmented\":0,\"text\":\"Ciclos repetitivos, inspección de soldaduras, suspensión y fijaciones. Consumo real de combustible, horas entre servicios y facilidad de reparación.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":222,\"augmented\":0,\"text\":\"Comparar aceros estructurales certificados, espesores y recubrimientos anticorrosión según resistencia, disponibilidad y costo. Estandarizar piezas repetidas, fijaciones y plantillas para reducir retrabajos y variabilidad.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":10,\"char_len\":235,\"augmented\":0,\"text\":\"Ficha de especificaciones, muestras iniciales, trazabilidad de lote, verificación dimensional y evaluación de entregas. Inspección de soldaduras, tolerancias, torque de ensamble, frenos y pruebas funcionales antes de liberar la unidad.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":461,\"augmented\":0,\"text\":\"Ocelot Racing Company emplea a 200 profesionales calificados en diversos campos como: diseño, ingeniería, manufactura, logística y operaciones. -Diversidad e inclusión: Un 38% de estos expertos está conformado por mujeres. -Crecimiento profesional: Programas de capacitación continua, certificaciones, planes de retiro y seguro médico. -Excelencia colaborativa: Equipos multidisciplinarios que garantizan soluciones robustas, desde el concepto hasta la entrega.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":335,\"augmented\":1,\"text\":\"Ocelot Racing Company emplea a 200 profesionales calificados en diversos campos como: diseño, ingeniería, manufactura, logística y operaciones. -Diversidad e inclusión: Un 38% de estos expertos está conformado por mujeres. -Crecimiento profesional: Programas de capacitación continua, certificaciones, planes de retiro y seguro médico.\"},{\"section\":\"mercado\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":406,\"augmented\":0,\"text\":\"Respaldados por el crecimiento financiero constante y la experiencia obtenida con nuestros modelos anteriores como es IKTAN II, estimamos posicionar al IKTAN III con una cuota de mercado del 15% al 25% dentro del sector de vehículos utilitarios y recreativos especializados. Esta cifra representa nuestro conocimiento anual y responde a la demanda de vehículos nacionales duraderos y de bajo mantenimiento.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":397,\"augmented\":0,\"text\":\"Presentamos IKTAN III, un UTV\\/BAJA de alto rendimiento propulsado por motor V-Twin 1000cc EFI a gasolina, diseñado para operar con confiabilidad en las condiciones extremas de la Península de Yucatán. Se construye sobre una plataforma única que se adapta en dos configuraciones, que reducen el mantenimiento, simplifican el inventario de refacciones y hace más eficiente el despliegue de la flota.\"},{\"section\":\"mercado\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":131,\"augmented\":1,\"text\":\"Esta cifra representa nuestro conocimiento anual y responde a la demanda de vehículos nacionales duraderos y de bajo mantenimiento.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":196,\"augmented\":1,\"text\":\"Se construye sobre una plataforma única que se adapta en dos configuraciones, que reducen el mantenimiento, simplifican el inventario de refacciones y hace más eficiente el despliegue de la flota.\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":143,\"augmented\":0,\"text\":\"IKTAN III no solo es un medio de ayuda o transporte, sino toda una experiencia y aventura exclusiva para los visitantes del complejo turístico.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":576,\"augmented\":0,\"text\":\"Para garantizar la operación continua del grupo de vehículos dentro del complejo sin salir a estaciones públicas, proponemos un esquema de soporte en dos puntos estratégicos: • Planta Regional de Apoyo (Ocelot Racing Company): Ubicada cerca de la zona turística, esta planta funciona como el centro principal de resguardo técnico, reparaciones pesadas, revisiones mecánicas complejas y suministro continuo de refacciones. • Módulo de Mantenimiento Interno (En el Resort): Instalación adaptada dentro del complejo para dar atención preventiva inmediata y revisiones cotidianas.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":180,\"augmented\":0,\"text\":\"Este espacio es atendido directamente por el propio personal del resort previamente capacitado, lo que genera empleos locales y evita tiempos de espera ante eventualidades menores.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":349,\"augmented\":1,\"text\":\"Ubicada cerca de la zona turística, esta planta funciona como el centro principal de resguardo técnico, reparaciones pesadas, revisiones mecánicas complejas y suministro continuo de refacciones. • Módulo de Mantenimiento Interno (En el Resort): Instalación adaptada dentro del complejo para dar atención preventiva inmediata y revisiones cotidianas.\"},{\"section\":\"suministro\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":261,\"augmented\":0,\"text\":\"Módulo estacionario de autoconsumo (base operativa): tanque contenedor anti fugas de doble pared (UL-142\\/NOM-005), de 1,000 L a 2,000 L, con cero evaporaciones, trampa de humedad, válvula de venteo con trampa de presión\\/vacío y bomba dispensadora de 12 V\\/110 V.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":509,\"augmented\":0,\"text\":\"Nuestra estrategia de producción se basa en un enfoque balanceado de dos pilares: • Control de calidad centralizado en Guanajuato: Mantenemos en nuestra planta el control total de la calidad y la flexibilidad técnica en los sistemas más críticos del vehículo (como el chasis, la soldadura, el ensamble final y la integración). • Eficiencia mediante alianzas estratégicas: Optimizamos los costos de fabricación y los tiempos de respuesta colaborando con proveedores especializados para componentes secundarios.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":182,\"augmented\":1,\"text\":\"• Eficiencia mediante alianzas estratégicas: Optimizamos los costos de fabricación y los tiempos de respuesta colaborando con proveedores especializados para componentes secundarios.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":208,\"augmented\":0,\"text\":\"Operamos bajo la filosofía Lean Manufacturing e integración Just in Time (JIT) para minimizar desperdicios, mantener un producto robusto y de costo competitivo, sosteniendo nuestro margen de utilidad del 30%.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":646,\"augmented\":0,\"text\":\"Nivel 1: Laboratorio Nivel 2: Componentes Nivel 3: Escenarios Nivel 4: Campo Simulación FEA \\/ CAD Banco de frenos Aceleración y cuesta Terrenos reales (Yucatán) Certificaciones normativas • MIL-STD-810H: resistencia ambiental (humedad, vibración, impacto, polvo y salitre). • ISO 9001:2015: gestión de calidad en fabricación y ensamble. • IATF 16949:2016: estándar de calidad automotriz internacional. • SAE J1194: certificación de pruebas dinámicas de chasis, fatiga de materiales e ingeniería en tren motriz. • ISO 3471: certificación internacional de estabilidad dinámica, seguridad de ocupantes y resistencia estructural ROPS ante volcaduras.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":6,\"char_len\":255,\"augmented\":0,\"text\":\"Nuestra estructura de costos se apoya en principios de manufactura modular y en acuerdos de volumen con proveedores, que reducen el costo unitario a medida que crece el lote. El modelo proyecta un margen de utilidad del 30%. Todas las cifras están en USD.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":309,\"augmented\":1,\"text\":\"• IATF 16949:2016: estándar de calidad automotriz internacional. • SAE J1194: certificación de pruebas dinámicas de chasis, fatiga de materiales e ingeniería en tren motriz. • ISO 3471: certificación internacional de estabilidad dinámica, seguridad de ocupantes y resistencia estructural ROPS ante volcaduras.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":155,\"augmented\":0,\"text\":\"Mano de obra directa de corte, soldadura y ensamble, con un modelo de eficiencia progresiva: las horas por vehículo disminuyen conforme aumenta el volumen.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":604,\"augmented\":0,\"text\":\"• Diseño de monoplaza Baja orientado a uso recreativo\\/off-road, sujeto a validación mediante pruebas del prototipo. • Configuración de referencia: motor gasolina Kohler CH440 de 14 HP, suspensión independiente de doble horquilla y neumáticos 22×10-10. • Acompañamiento técnico para documentar inspección, mantenimiento preventivo, refacciones críticas y operación segura. • Plan de suministro por lotes para reducir el riesgo de producción y permitir retroalimentación después de la primera entrega. Las características indicadas son de diseño\\/prototipo y no deben presentarse como desempeño certificado.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":488,\"augmented\":0,\"text\":\"participa como proveedor independiente en un proceso competitivo para suministrar monoplazas Baja a Outdoor Adventure Holdings (OAH). OAH es el cliente potencial y evaluará ofertas de distintos proveedores. Esta propuesta no contempla sociedad, inversión conjunta ni participación accionaria. El objetivo comercial es presentar una oferta técnicamente responsable, con precio competitivo, entregas por etapas y un plan de validación del prototipo antes de comprometer producción a escala.\"},{\"section\":\"cliente\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":466,\"augmented\":0,\"text\":\"• Primera entrega solicitada: 20 unidades, disponibles seis meses después del inicio de construcción del proyecto. • Segunda entrega solicitada: 50 unidades adicionales, 18 meses después del inicio de construcción; total previsto de 70 unidades. • Prioridades declaradas: alta disponibilidad operativa, mantenimiento reducido, pruebas previas a la compra y claridad sobre almacenamiento, refacciones, combustible, infraestructura y costos de operación a largo plazo.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":195,\"augmented\":1,\"text\":\"El objetivo comercial es presentar una oferta técnicamente responsable, con precio competitivo, entregas por etapas y un plan de validación del prototipo antes de comprometer producción a escala.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":247,\"augmented\":1,\"text\":\"• Acompañamiento técnico para documentar inspección, mantenimiento preventivo, refacciones críticas y operación segura. • Plan de suministro por lotes para reducir el riesgo de producción y permitir retroalimentación después de la primera entrega.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":649,\"augmented\":0,\"text\":\"• Reunión de levantamiento: confirmar terreno, ciclo de uso, operadores, almacenamiento, combustible, transporte y criterios de aceptación de OAH. • Demostración del prototipo: inspección estática y pruebas controladas, con protocolo acordado, registro de resultados y lista de pendientes. • Entrega de expediente técnico: ficha de configuración, plan de mantenimiento preliminar, lista inicial de refacciones, recomendaciones de almacenamiento y requerimientos de infraestructura. • Oferta formal: precio, alcance, calendario, garantías, condiciones de pago, exclusiones y criterios de aceptación claramente definidos. • Lote piloto de 20 unidades:\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":262,\"augmented\":0,\"text\":\"fabricación y entrega sujetas a aprobación del prototipo, disponibilidad de componentes y validación de capacidad productiva. • Seguimiento posterior al lote piloto: recopilar incidencias y retroalimentación para ajustar el proceso antes del lote de 50 unidades.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":120,\"augmented\":0,\"text\":\"Consumo, confiabilidad, carga útil, velocidad, autonomía y vida útil deberán medirse y documentarse durante las pruebas.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":2,\"char_len\":478,\"augmented\":0,\"text\":\"• Prototipo aún en construcción: mitigar mediante pruebas documentadas y criterios de aceptación previos a producción en serie. • Capacidad productiva no confirmada: realizar planificación de materiales y capacidad antes de comprometer entregas. • Costos sujetos a cotización: establecer vigencia de oferta y mecanismos de ajuste ante cambios relevantes de insumos. • Desempeño operativo aún no medido: evitar promesas no verificadas y entregar resultados de pruebas al cliente.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":494,\"augmented\":0,\"text\":\"Rendimiento contable simple 74.5 % acumulado sobre el capital hipotético de $1.75 millones; no es una tasa anual ni efectivo distribuible. Las cifras financieras son supuestos de planeación, no cotizaciones, pedidos ganados ni aportaciones realizadas. El aporte real de los fundadores no fue informado. Al mes 18 se proyectan $203,400 de caja, $2.70 millones por cobrar y $270,000 de garantía pendiente; por ello, la utilidad contable no demuestra recuperación de efectivo dentro del horizonte.\"},{\"section\":\"solucion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":1,\"char_len\":581,\"augmented\":0,\"text\":\"Indicador Resultado del escenario de planeación Oferta al cliente Vehículo configurado, entrega, manuales e inducción; refacciones y servicio definidos por contrato. Precio de trabajo $180,000 por unidad, antes de IVA; sujeto a costos cotizados y alcance final. Ingresos \\/ utilidad neta $12.60 millones \\/ $1.303 millones durante el proyecto; margen neto de 10.3 %. Equilibrio \\/ capacidad objetivo 33 vehículos \\/ 2 unidades conformes por semana en campañas de fabricación. Capital inicial propuesto $1.75 millones: $350,000 propios propuestos y $1.40 millones externos por levantar.\"},{\"section\":\"segmentacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":602,\"augmented\":0,\"text\":\"Economía de operación Horas, kilómetros, vehículos actuales, gasto comprobado de renta y mantenimiento. Decisión y servicio Responsables de compras y operación, presupuesto, calendario y recursos del taller. Tamaño y tendencia: contexto medido, demanda por validar El Compendio Estadístico del Turismo 2024 de SECTUR–DATATUR reporta para Yucatán 710 establecimientos de alojamiento y 17,100 cuartos, frente a 649 y 16,483 en 2023. La oferta reportada creció 9.4 % y 3.7 %, respectivamente. Incluye distintas modalidades de alojamiento: no representa 710 resorts ni compradores confirmados de vehículos.\"},{\"section\":\"segmentacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":381,\"augmented\":0,\"text\":\"[1] La demanda total y el mercado atendible de UTV no están cuantificados en la fuente. Se estimarán desde un directorio de sitios elegibles, entrevistas y necesidades de reposición verificadas. La oportunidad inicial del RFP es de 70 unidades, condicionada a selección y contratación. Las proyecciones excluyen ventas a otros clientes e ingresos turísticos todavía no contratados.\"},{\"section\":\"segmentacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":227,\"augmented\":1,\"text\":\"Economía de operación Horas, kilómetros, vehículos actuales, gasto comprobado de renta y mantenimiento. Decisión y servicio Responsables de compras y operación, presupuesto, calendario y recursos del taller. Tamaño y tendencia:\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":332,\"augmented\":1,\"text\":\"Compras y logística Proveedores, inventario, preservación y entrega en sitio. Finanzas y administración Costeo, cobranza, caja, obligaciones y control documental. Servicio Manuales, formación del cliente, incidencias y garantías. Los líderes nominales y su dedicación están pendientes de confirmar. Las funciones pueden compartirse;\"},{\"section\":\"segmentacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":576,\"augmented\":0,\"text\":\"El cliente objetivo es un operador de complejo turístico de aventura en Yucatán con predio extenso, caminos internos y traslados recurrentes de herramientas, suministros y equipo. Para ser elegible deberá tener tareas y terreno compatibles, presupuesto identificable y capacidad de mantenimiento o servicio contratado. No se presupone que un vehículo utilitario sustituya maquinaria pesada ni que todo hotel necesite esta flota. Criterio de segmentación Dato que se verificará con el comprador Uso y terreno Rutas, pendientes, firme, carga, ocupantes y frecuencia de traslado.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":287,\"augmented\":0,\"text\":\"no equivalen a seis puestos de tiempo completo. La célula de producción presupuestada contempla ocho recursos equivalentes por campaña, con mano de obra pagada dentro del costo unitario. La experiencia se respaldará con proyectos, planos, prototipos y reportes de pruebas identificables.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":620,\"augmented\":0,\"text\":\"Función Responsabilidad Dirección y comercial Contrato, coordinación, presupuesto, decisiones y relación con OAH. Ingeniería de producto Diseño, integración, cargas, validación y control de cambios. Producción y calidad Métodos, capacitación, tiempos, inspección y liberación por unidad. Compras y logística Proveedores, inventario, preservación y entrega en sitio. Finanzas y administración Costeo, cobranza, caja, obligaciones y control documental. Servicio Manuales, formación del cliente, incidencias y garantías. Los líderes nominales y su dedicación están pendientes de confirmar. Las funciones pueden compartirse;\"},{\"section\":\"competencia\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":611,\"augmented\":0,\"text\":\"No permiten afirmar un ahorro porcentual frente al precio propuesto de Blue Buffalo sin homologar impuestos, entrega, plazas, carga, tracción y soporte. La diferenciación se probará en una ruta común, con la misma tarea, jornada y método de costo total. [1] SECTUR–DATATUR: Compendio 2024; base 10_1.xlsx, oferta por entidad. [2] Polaris México: RANGER 500. Precios y configuración publicados. [3] Polaris México: RANGER 600. Precios y configuración publicados. [4] Can-Am México: Defender 2026, precio; ficha mexicana HD7 vinculada abajo. Ficha técnica mexicana Can-Am Defender HD7 2026: capacidad y ocupantes.\"},{\"section\":\"marketing\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":311,\"augmented\":0,\"text\":\"Posicionamiento: una flota configurada para las tareas del complejo, con mantenimiento accesible y desempeño verificable. La venta se sustentará en la demostración, el costo de servicio y la capacidad de cumplir el contrato. Las ventajas de precio, consumo o durabilidad quedarán sujetas a evidencia comparable.\"},{\"section\":\"competencia\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":265,\"augmented\":1,\"text\":\"RANGER 500. Precios y configuración publicados. [3] Polaris México: RANGER 600. Precios y configuración publicados. [4] Can-Am México: Defender 2026, precio; ficha mexicana HD7 vinculada abajo. Ficha técnica mexicana Can-Am Defender HD7 2026: capacidad y ocupantes.\"},{\"section\":\"competencia\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":3,\"char_len\":557,\"augmented\":0,\"text\":\"Alternativa Referencia pública de precio Capacidad o condición relevante Polaris RANGER 500 Desde $234,900; IVA incluido. [2] Dos plazas y AWD; capacidad de carga por confirmar en cotización. Polaris RANGER 600 Desde $299,900; IVA incluido. [3] Dos plazas; carga útil total de 420 kg y PREMIUM caja de 227 kg. Can-Am Defender HD7 2026 Desde $273,900; transporte y preparación Tres plazas; carga útil total de 680 kg y aparte; IVA por confirmar. [4] caja de 454 kg. Consulta: 30\\/10\\/2026. Son precios publicados, no cotizaciones de flota entregada en Yucatán.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":242,\"augmented\":0,\"text\":\"El material ASTM A500 Grado B considerado para la estructura requiere verificación del conjunto y protección anticorrosiva. La transición de uso exigirá inspección y liberación; la limitación de velocidad deberá funcionar también en descenso.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":199,\"augmented\":0,\"text\":\"Un piloto de recorridos UTV requerirá configuración, ocupantes, rutas y economía propios; los 70 vehículos de apoyo no se contabilizan automáticamente como una flota turística generadora de ingresos.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":568,\"augmented\":0,\"text\":\"La venta será directa a compras, operaciones y mantenimiento. El proceso contempla diagnóstico de tareas, demostración con criterios de aprobación, propuesta técnica y económica, y cierre con evidencia de fondos. El seguimiento medirá cuentas calificadas, demostraciones aprobadas, propuestas y contratos; las metas comerciales se fijarán después de validar el directorio. La entrega se coordinará desde la planta mediante transporte consolidado y recepción en sitio. Tras la aceptación del primer lote, los datos de campo podrán apoyar referencias y nuevos contratos.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":342,\"augmented\":1,\"text\":\"Transmisión con reversa y frenos de servicio y estacionamiento están por seleccionar y dimensionar. El material ASTM A500 Grado B considerado para la estructura requiere verificación del conjunto y protección anticorrosiva. La transición de uso exigirá inspección y liberación; la limitación de velocidad deberá funcionar también en descenso.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":128,\"augmented\":1,\"text\":\"El escenario base supone venta directa; no incluye cartera de leasing, intereses ni ingresos de arrendamiento para Blue Buffalo.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":262,\"augmented\":1,\"text\":\"las metas comerciales se fijarán después de validar el directorio. La entrega se coordinará desde la planta mediante transporte consolidado y recepción en sitio. Tras la aceptación del primer lote, los datos de campo podrán apoyar referencias y nuevos contratos.\"},{\"section\":\"ventas\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":450,\"augmented\":0,\"text\":\"Se propone canalizar un arrendamiento de 36 meses mediante una entidad financiadora, sujeto a aprobación crediticia. El contrato deberá fijar anticipo o depósito, renta, mantenimiento, seguro, uso permitido, devolución y opción de compra cuando corresponda. La mensualidad no se ofrece sin una cotización del financiador. El escenario base supone venta directa; no incluye cartera de leasing, intereses ni ingresos de arrendamiento para Blue Buffalo.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":648,\"augmented\":0,\"text\":\"Producto y límites de la configuración inicial La base de diseño contempla 2.20 m de largo, 1.80 m de ancho exterior, masa objetivo de 280 kg y carga útil objetivo de 350 kg. Debe definirse si la carga incluye ocupantes y accesorios antes de fijar la masa máxima operativa. Las plazas autorizadas, la pendiente admisible y la velocidad segura se establecerán mediante pruebas; estas cifras no son capacidades certificadas. Se propone motor CH440 de 429 cm³, tracción trasera, dirección de piñón y cremallera, doble horquilla y neumáticos 25 × 8-10. Transmisión con reversa y frenos de servicio y estacionamiento están por seleccionar y dimensionar.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":569,\"augmented\":0,\"text\":\"M0 es el inicio de construcción del complejo. El desarrollo y la demostración previa se presupuestan antes de ese hito; los demostradores no forman parte de las 70 entregas. Contratación y recursos tardíos obligarían a recalcular el plazo restante sin reiniciar las fechas del RFP. Periodo Plan y condición de salida Aceptadas acumuladas Antes de M0 Demostración, alcance, presupuesto y compromisos de financiación. 0 M1–M3 Industrialización, herramental, proveedores y validación de serie. 0 M4 \\/ M5 \\/ M6 Fabricar 8 \\/ 8 \\/ 4 unidades; trasladar y aceptar el lote en M6.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":287,\"augmented\":1,\"text\":\"Periodo Plan y condición de salida Aceptadas acumuladas Antes de M0 Demostración, alcance, presupuesto y compromisos de financiación. 0 M1–M3 Industrialización, herramental, proveedores y validación de serie. 0 M4 \\/ M5 \\/ M6 Fabricar 8 \\/ 8 \\/ 4 unidades; trasladar y aceptar el lote en M6.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":244,\"augmented\":1,\"text\":\"Las reservas de dos y tres semanas se comparten entre logística y contingencias. El modelo supone anticipo a proveedores dos meses antes del ensamble, recepción un mes antes y saldo en el mes de fabricación; estos plazos requieren confirmación.\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":568,\"augmented\":0,\"text\":\"20 en M6 M7–M11 Servicio, seguimiento, mejoras aprobadas y compras de la siguiente 20 campaña. M12–M17 Fabricar 8 unidades por mes: 48 en total; preservar inventario. 20 M18 Fabricar 2 adicionales, trasladar y aceptar el segundo lote. 70 Con cuatro semanas productivas por mes, los lotes ocupan 10 de 12 y 25 de 28 semanas. Las reservas de dos y tres semanas se comparten entre logística y contingencias. El modelo supone anticipo a proveedores dos meses antes del ensamble, recepción un mes antes y saldo en el mes de fabricación; estos plazos requieren confirmación.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":6,\"char_len\":621,\"augmented\":0,\"text\":\"corregir y Ingeniería No cumple carga, frenado, repetir los ensayos pertinentes. No habilitar transporte estabilidad o temperatura. turístico con una configuración sin validar. Costo de compra mayor Actualizar cotizaciones y alcance antes de cerrar precio; el Compras y finanzas Fabricación y entrega suben 15 escenario requiere $2.70 millones de capital. Sustituciones y %. ajustes de precio necesitan aprobación técnica y contractual. Cobros tardíos Condicionar órdenes y avance a fondos disponibles; Finanzas y Anticipos o avances se retrasan negociar hitos o financiación comprometida. El capital comercial dos meses.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":6,\"char_len\":253,\"augmented\":0,\"text\":\"La decisión de producir se apoyará en riesgos con responsables, señales de activación y acciones financiables. Los proveedores alternativos y recursos adicionales descritos son requisitos de preparación; todavía no se presentan como acuerdos existentes.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":6,\"char_len\":630,\"augmented\":0,\"text\":\"Proveedor crítico incumple Calificar dos alternativas para amortiguadores y piezas Compras e Su fecha confirmada consume la críticas, con muestras y pruebas de equivalencia. Mantener ingeniería holgura disponible. la compra principal y activar otra solo después de aprobarla; ajustar secuencia y stock. Capacidad insuficiente Medir la estación limitante, balancear cargas y contratar Producción y El piloto no sostiene 2 unidades capacidad externa calificada. Reprogramar antes de calidad conformes por semana. comprometer entrega; conservar controles y pruebas. Diseño o seguridad Detener liberación de la configuración afectada;\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":6,\"char_len\":564,\"augmented\":0,\"text\":\"requerido sube a $2.85 millones en la simulación. Concentración en OAH Reformular desarrollo, capacidad y precio antes de iniciar. Dirección Solo se formalizan 20 unidades. Mantener la estructura de 18 meses produciría una pérdida de $618,000; no compensarla con ventas futuras sin evidencia. Fallas en campo Analizar causa, ajustar mantenimiento y piezas críticas; Servicio y calidad Disponibilidad inferior al revisar corrosión y uso. Reestimar la provisión de garantía objetivo o garantía por encima de con datos reales y ampliar fondos si corresponde. reserva.\"},{\"section\":\"riesgos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":6,\"char_len\":327,\"augmented\":1,\"text\":\"Capacidad insuficiente Medir la estación limitante, balancear cargas y contratar Producción y El piloto no sostiene 2 unidades capacidad externa calificada. Reprogramar antes de calidad conformes por semana. comprometer entrega; conservar controles y pruebas. Diseño o seguridad Detener liberación de la configuración afectada;\"},{\"section\":\"financiera\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":284,\"augmented\":0,\"text\":\"Escenario gerencial ilustrativo en MXN sin IVA, desde la preparación registrada en M0 hasta M18. Todos los costos y condiciones son hipótesis por cotizar. Supone 70 unidades contratadas y aceptadas; no incorpora otros clientes, leasing, ingresos turísticos, deuda ni trabajo gratuito.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":250,\"augmented\":0,\"text\":\"Balance general y flujo de caja pro forma Miles de MXN; escenario ilustrativo. El balance presenta saldos al cierre de cada mes indicado. Balance M0 M6 M12 M18 Efectivo 800.0 460.0 2,226.0 203.4 Cuentas por cobrar 0.0 1,080.0 0.0 2,700.0 Inventarios:\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":640,\"augmented\":0,\"text\":\"materiales y terminados 0.0 0.0 1,610.0 0.0 Anticipos a proveedores 0.0 0.0 320.0 0.0 Activo fijo neto 600.0 540.0 480.0 420.0 Activo total 1,400.0 2,080.0 4,636.0 3,323.4 Proveedores por pagar 0.0 0.0 320.0 0.0 Anticipos del cliente 0.0 0.0 2,700.0 0.0 Provisión de garantía pendiente 0.0 108.0 54.0 270.0 Provisión fiscal por pagar 0.0 66.6 0.0 0.0 Pasivo total 0.0 174.6 3,074.0 270.0 Capital aportado hipotético 1,750.0 1,750.0 1,750.0 1,750.0 Resultados acumulados -350.0 155.4 -188.0 1,303.4 Patrimonio total 1,400.0 1,905.4 1,562.0 3,053.4 Pasivo + patrimonio 1,400.0 2,080.0 4,636.0 3,323.4 M0 incluye el gasto de desarrollo previo.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":620,\"augmented\":0,\"text\":\"El inventario de M12 contiene $10,000 de depreciación capitalizada. Los anticipos del cliente son pasivo hasta aceptación. La garantía concilia: gasto de $378,000 menos pagos de $108,000 = pasivo final de Flujo de caja por periodo · miles de MXN Concepto M0 M1–M6 M7–M12 M13–M18 Caja inicial 0.0 800.0 460.0 2,226.0 Cobros del cliente 0.0 2,520.0 3,780.0 3,600.0 Pagos operativos, incluido impuesto -350.0 -2,860.0 -2,014.0 -5,622.6 Flujo operativo neto -350.0 -340.0 1,766.0 -2,022.6 Inversión en activo fijo -600.0 0.0 0.0 0.0 Aportaciones propuestas de capital 1,750.0 0.0 0.0 0.0 Caja final 800.0 460.0 2,226.0 203.4\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":289,\"augmented\":1,\"text\":\"Pasivo total 0.0 174.6 3,074.0 270.0 Capital aportado hipotético 1,750.0 1,750.0 1,750.0 1,750.0 Resultados acumulados -350.0 155.4 -188.0 1,303.4 Patrimonio total 1,400.0 1,905.4 1,562.0 3,053.4 Pasivo + patrimonio 1,400.0 2,080.0 4,636.0 3,323.4 M0 incluye el gasto de desarrollo previo.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":144,\"augmented\":1,\"text\":\"El inventario de M12 contiene $10,000 de depreciación capitalizada. Los anticipos del cliente son pasivo hasta aceptación. La garantía concilia:\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":8,\"char_len\":349,\"augmented\":1,\"text\":\"gasto de $378,000 menos pagos de $108,000 = pasivo final de Flujo de caja por periodo · miles de MXN Concepto M0 M1–M6 M7–M12 M13–M18 Caja inicial 0.0 800.0 460.0 2,226.0 Cobros del cliente 0.0 2,520.0 3,780.0 3,600.0 Pagos operativos, incluido impuesto -350.0 -2,860.0 -2,014.0 -5,622.6 Flujo operativo neto -350.0 -340.0 1,766.0 -2,022.6 Inversión\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":536,\"augmented\":0,\"text\":\"ilustración a 60 meses Horizonte desde inicio de obra: el primer lote opera 54 meses y el segundo 42. Supuestos: 1,000 horas programadas por vehículo\\/año, 95 % de disponibilidad, 10 km por hora efectiva, consumo de 2.5 L\\/h a $25\\/L, mantenimiento de $22\\/h y seguro de $5,000 por vehículo\\/año. Son hipótesis, no mediciones ni precios cotizados. Componente o indicador Resultado ilustrativo Compra \\/ infraestructura incremental $12,600,000 \\/ $400,000 Combustible \\/ mantenimiento \\/ seguros $15,734,375 \\/ $5,538,500 \\/ $1,325,000 Costo total;\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":616,\"augmented\":0,\"text\":\"Sensibilidades · importes en miles de MXN Escenario Resultado neto Capital con reserva Base: 70 unidades 1,303.4 1,750.0 Fabricación y entrega +15 % 384.6 2,700.0 Anticipos y avances +2 meses 1,303.4 2,850.0 Solo 20 unidades; fijos completos -618.0 1,500.0 Costos +15 % y cobros +2 meses 384.6 3,900.0 El capital requerido no equivale al capital ya disponible. La demora de cobros no modifica el resultado en este escenario sin deuda, pero sí la caja. El caso de 20 unidades evita desde el inicio las compras del segundo lote; una cancelación tardía podría generar una pérdida mayor. Costo total de operación de OAH:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":621,\"augmented\":0,\"text\":\"Economía por vehículo y rendimiento del capital Costo variable entregado con garantía: $130,400; contribución: $49,600 por unidad. Los costos fijos del horizonte, desarrollo y depreciación suman $1,610,000. A 70 unidades, el costo total contable antes de impuestos es $153,400 por vehículo. Rendimiento contable simple estimado: $1,303,400 \\/ $1,750,000 = 74.5 %, acumulado en el horizonte e incluyendo el desarrollo previo. No es TIR, tasa anual, dividendo ni retorno del cliente. Depende del financiamiento mediante anticipos y de cobrar las ventas pendientes; no demuestra recuperación de capital en efectivo al mes 18.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":300,\"augmented\":1,\"text\":\"fijos completos -618.0 1,500.0 Costos +15 % y cobros +2 meses 384.6 3,900.0 El capital requerido no equivale al capital ya disponible. La demora de cobros no modifica el resultado en este escenario sin deuda, pero sí la caja. El caso de 20 unidades evita desde el inicio las compras del segundo lote;\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":202,\"augmented\":1,\"text\":\"una cancelación tardía podría generar una pérdida mayor. Costo total de operación de OAH: ilustración a 60 meses Horizonte desde inicio de obra: el primer lote opera 54 meses y el segundo 42. Supuestos:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":193,\"augmented\":1,\"text\":\"Componente o indicador Resultado ilustrativo Compra \\/ infraestructura incremental $12,600,000 \\/ $400,000 Combustible \\/ mantenimiento \\/ seguros $15,734,375 \\/ $5,538,500 \\/ $1,325,000 Costo total;\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":229,\"augmented\":1,\"text\":\"1,000 horas programadas por vehículo\\/año, 95 % de disponibilidad, 10 km por hora efectiva, consumo de 2.5 L\\/h a $25\\/L, mantenimiento de $22\\/h y seguro de $5,000 por vehículo\\/año. Son hipótesis, no mediciones ni precios cotizados.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":579,\"augmented\":0,\"text\":\"valor residual supuesto cero $35,597,875 Uso efectivo acumulado 251,750 horas \\/ 2,517,500 km Costo por hora \\/ por kilómetro $141.40\\/h \\/ $14.14\\/km El umbral de renta equivalente es $6,246 por vehículo\\/mes, con mantenimiento y seguro incluidos, combustible aparte y sin cargos iniciales adicionales. Es un cálculo comparativo, no una oferta. ROI y recuperación del cliente requieren renta cotizada con servicio equivalente. Se excluyen IVA, financiación, descuento, inflación y costo monetizado de indisponibilidad; el consumo y la utilización deben medirse. Criterio de inversión:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":9,\"char_len\":245,\"augmented\":0,\"text\":\"avanzar al suministro cuando existan cotizaciones, pedidos de ambos lotes, capital acreditado, calendario fiscal y diseño\\/capacidad validados. Si una condición cambia, actualizar precio, alcance y financiación antes de comprometer la producción.\"},{\"section\":\"organizacion\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":2,\"char_len\":272,\"augmented\":0,\"text\":\"We cultivate a skilled, diverse, and collaborative workforce that drives innovation across every stage of our operations. By investing in continuous training and cross-functional expertise, we strengthen our capabilities and maintain high standards of quality, efficiency.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":440,\"augmented\":0,\"text\":\"Leveraging initial field data, our R&D pipeline transitions the 50-unit fleet into the KINAM Venture configuration for eco-tourism, featuring 100% backward compatibility to seamlessly integrate early construction units with fully interchangeable modular attachments and zero structural modifications. Engineered to maximize asset lifecycle and ROI, the Venture package completes the fleet's transition into high-margin eco-resort operations\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":300,\"augmented\":1,\"text\":\"Leveraging initial field data, our R&D pipeline transitions the 50-unit fleet into the KINAM Venture configuration for eco-tourism, featuring 100% backward compatibility to seamlessly integrate early construction units with fully interchangeable modular attachments and zero structural modifications.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":4,\"char_len\":139,\"augmented\":1,\"text\":\"Engineered to maximize asset lifecycle and ROI, the Venture package completes the fleet's transition into high-margin eco-resort operations\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":333,\"augmented\":0,\"text\":\"We implement Lean Six Sigma and Kaizen methodologies to drive high-quality, cost-efficient production. This rigorous framework guarantees continuous process consistency and strict defect minimization across the KINAM assembly line, ultimately ensuring on-time fleet delivery and a lower Total Cost of Ownership (TCO) for our clients.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":5,\"char_len\":212,\"augmented\":0,\"text\":\"Continuous Serpentine Flow: Engineered for high throughput and zero process backtracking, routing units seamlessly from raw storage through powertrain, assembly, and testing directly to the vehicle dispatch yard.\"},{\"section\":\"ingresos\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":5,\"char_len\":242,\"augmented\":0,\"text\":\"approach that combines product validation, global visibility, and differentiated guest experiences. “Performance Meets Experience” Beyond operational utility, the KINAM Venture fleet unlocks high-margin revenue streams to accelerate your ROI:\"},{\"section\":\"vehiculo\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":117,\"augmented\":0,\"text\":\"To guarantee zero-defect deployment and maximize uptime, every KINAM platform undergoes a strict validation protocol:\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":6,\"char_len\":580,\"augmented\":0,\"text\":\"To ensure scalable, cost-efficient production, our manufacturing model executes a strictly calculated strategy: • In-House Core Engineering: We maintain strict internal control over all mission- critical systems, ensuring absolute quality oversight. • Certified Tier-1 Partnerships: We optimize capital costs and guarantee supply chain by partnering exclusively with ISO and IATF-certified institutions for specialized components. • Phased Internalization: As fleet production scales to meet long-term resort demands, we will progressively internalize key manufacturing processes.\"},{\"section\":\"inversion\",\"clase\":\"finanzas_riesgo\",\"label\":1,\"page\":7,\"char_len\":84,\"augmented\":0,\"text\":\"The KINAM Base Vehicle integrates all non- negotiable structural and safety systems.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":374,\"augmented\":0,\"text\":\"Our cost structure is anchored in modular automotive manufacturing, operating out of our assembly facility in Puebla. Direct production expenses are divided into standardized Bill of Materials (BOM) and certified technical labor. By transitioning from a pilot run to commercial volume Ocelot Racing captures a discount from tier-1 suppliers and reduces assembly cycle times.\"},{\"section\":\"operaciones\",\"clase\":\"mercado_propuesta\",\"label\":0,\"page\":7,\"char_len\":168,\"augmented\":0,\"text\":\"Managing a 250,000-hectare property requires a highly efficient logistical footprint to eliminate operational downtime. We propose a decentralized infrastructure model:\"}]"
train = pd.DataFrame(json.loads(TRAIN_JSON))
print(f'Filas de entrenamiento: {len(train)}')
print(train.clase.value_counts().to_string())
print(f"Extra de train: {int(train.augmented.sum())}")
train.groupby('section').size().sort_values(ascending=False)


Filas de entrenamiento: 500
clase
mercado_propuesta    385
finanzas_riesgo      115
Extra de train: 166
section
operaciones     114
vehiculo         84
inversion        42
solucion         38
suministro       36
riesgos          31
ventas           27
marketing        24
financiera       21
ingresos         21
organizacion     18
cliente          13
competencia      12
mercado          11
segmentacion      8


In [1]:
train


section,clase,label,page,char_len,augmented,text
solucion,mercado_propuesta,0,2,590,0,"Ofrecemos dos variantes sobre la misma plataforma mecánica monoplaza 2WD para simplificar el inventario de refacciones. La versión utilitaria incorpora un módulo trasero de carga para 65 kg de herramientas, punto de remolque para arrastrar un carro ligero con 150 kg adicionales de piezas o insumos, recubrimiento anticorrosivo marino y faros de trabajo, mientras que la versión base se orienta a supervisión ágil y rutas recreativas guiadas. La inversión total por las 70 unidades es de $871,653.70 USD en flota con un paquete de soporte inicial y refacciones de $30,826.80 USD en soporte."
solucion,mercado_propuesta,0,2,282,0,"Tan solo en los primeros 12 meses de obra, la compra del primer lote genera un ahorro neto de $51,023.66 USD en el primer año frente al arrendamiento comercial de unidades equivalentes, alcanzando un periodo de recuperación de 10.4 meses y una disponibilidad operativa meta del 96%."
organizacion,mercado_propuesta,0,3,362,0,"Nuestra plantilla está integrada por 20 profesionales de diversas disciplinas de ingeniería y administración, con una composición multidisciplinaria e inclusiva enfocada en la colaboración técnica y operativa. La estructura opera mediante áreas de dirección, gestión de proyectos, diseño mecánico, manufactura, chasis, suspensión, tren motriz, compras y calidad."
vehiculo,mercado_propuesta,0,3,479,0,"Acorde a las necesidades de Outdoor Adventure Holdings, la configuración comercial soporta a un operador de hasta 110 kg e incorpora un módulo trasero desmontable con capacidad de 65 kg para herramientas, punto de remolque trasero para un carro utilitario ligero de hasta 150 kg de piezas e insumos, faros auxiliares LED, salpicaderas ampliadas contra lodo y silenciador adicional para zonas de reserva ecológica, manteniendo intacta la base del reporte de costos de competencia."
vehiculo,mercado_propuesta,0,3,545,0,"Configuración elegida Nuestra plataforma mantiene una arquitectura de tracción trasera 2WD optimizada bajo el reglamento Baja SAE México 2026. La arquitectura 4WD se incorpora como una mejora conceptual modular para requerimientos de terreno extremo en zonas de arena suelta y lodo en Yucatán. Para las condiciones de suelo calizo, arena costera, lodo y vegetación densa de Yucatán, el vehículo presenta dimensiones generales de 2.20 m de largo, 0.88 m de ancho y 1.26 m de alto, con una altura libre al suelo de 305 mm y peso en seco de 215 kg."
vehiculo,mercado_propuesta,0,3,397,0,"El sistema de dirección utiliza una cremallera reforzada con columna de aluminio 7075 de 1 pulgada y juntas universales de 36 estrías con fuelles guardapolvo. El frenado opera mediante doble circuito hidráulico con cilindros maestros Tilton Serie 74 de 7/8 de pulgada, barra de balance Tilton 600 y calipers con discos Kawasaki Brute Force, garantizando el bloqueo simultáneo de las cuatro ruedas."
vehiculo,mercado_propuesta,0,3,293,0,"El habitáculo protege al operador con asiento ergonómico, arnés de cinco puntos SFI 16.1, sujeciones de brazos SFI 3.3, soporte de cuello, doble interruptor de paro Polaris, luz de freno LED y extintor Kidde con soporte rápido, permitiendo la salida por ambos lados en menos de cinco segundos."
vehiculo,mercado_propuesta,0,3,622,0,"Requisitos de seguridad y costeo El chasis tubular se fabrica en acero tubular AISI 1020 DOM con corte láser rotativo y soldadura controlada en dispositivos de sujeción, verificado mediante análisis de elementos finitos con factor de seguridad superior a 1.8 en casos de impacto frontal, lateral y vuelco. Toda la estructura recibe recubrimiento anticorrosivo por aspersión líquida para resistir la intemperie costera. La suspensión independiente en las cuatro ruedas integra amortiguadores ajustables de largo recorrido y ocho rótulas de acero aleado de 5/8 de pulgada Rod End Supply que absorben irregularidades severas."
organizacion,mercado_propuesta,0,3,501,0,"Capacidad insti

## 3. Modelo y decisión

Modelo entregado: TF-IDF de palabras (1–2 gramas) + MLP de una capa oculta con 12 neuronas ReLU, entrenada desde cero con scikit-learn (`lbfgs`). La PWA reimplementa la inferencia en JavaScript. No hay pesos de un modelo de lenguaje.

Costos: falso positivo = 1, falso negativo = 5. El umbral se elige en validación minimizando el costo esperado; quedó en **0.26**.

Baselines de la corrida provisional:

- Mayoría: 742.6 por 1000.
- Keyword: 465.3 por 1000.
- Logistic: 505.0 por 1000.
- MLP calibrada: 514.9 por 1000.

La MLP sigue por detrás del baseline keyword, pero mejora frente a la corrida de 334 ejemplos (534.7 → 514.9). Se reporta el baseline y no se usa el test para elegir el umbral.

Calibración: temperature scaling. ECE baja de 0.151 a 0.084. El costo no baja con la calibración en esta corrida. La probabilidad ordena mejor de lo que estima una frecuencia exacta.

### Código de inferencia de la app

`practica1/pwa/inference.js` es el archivo que corre en el navegador. Incluye tokenización, TF-IDF, MLP, calibración y reglas OOD:

```javascript
/**
 * Inferencia pura de la PWA: TF-IDF + MLP de una capa oculta.
 * No usa DOM, para poder probarla con Node.
 */

const TOKEN_RE = /[a-z0-9_]{2,}/g;

export function stripAccents(text) {
  return String(text || "")
    .normalize("NFKD")
    .replace(/\p{M}/gu, "");
}

export function tokenize(text) {
  const normalized = stripAccents(String(text || "").toLowerCase());
  return normalized.match(TOKEN_RE) || [];
}

export function createModel(raw) {
  if (!raw || !raw.vocab || !raw.idf || !raw.w1 || !raw.w2) {
    throw new Error("Modelo inválido");
  }
  const hidden = raw.w1.length ? raw.w1[0].length : 0;
  return {
    ...raw,
    hiddenSize: hidden,
    stopwords: new Set(raw.ood?.spanish_stopwords || []),
  };
}

function featureCounts(tokens) {
  const counts = new Map();
  for (let i = 0; i < tokens.length; i += 1) {
    const uni = tokens[i];
    counts.set(uni, (counts.get(uni) || 0) + 1);
    if (i + 1 < tokens.length) {
      const bi = `${uni} ${tokens[i + 1]}`;
      counts.set(bi, (counts.get(bi) || 0) + 1);
    }
  }
  return counts;
}

function vectorize(model, text) {
  const tokens = tokenize(text);
  const counts = featureCounts(tokens);
  const sparse = [];
  let norm = 0;
  for (const [token, count] of counts.entries()) {
    if (!Object.prototype.hasOwnProperty.call(model.vocab, token)) continue;
    const idx = model.vocab[token];
    const tf = model.preprocess?.sublinear_tf ? 1 + Math.log(count) : count;
    const value = tf * model.idf[idx];
    if (!Number.isFinite(value) || value === 0) continue;
    sparse.push([idx, value]);
    norm += value * value;
  }
  const denom = Math.sqrt(norm) || 1;
  return { tokens, sparse: sparse.map(([idx, value]) => [idx, value / denom]) };
}

function relu(value) {
  return value > 0 ? value : 0;
}

function sigmoid(value) {
  const z = Math.max(-60, Math.min(60, value));
  return 1 / (1 + Math.exp(-z));
}

export function oodDecision(model, text, probability) {
  const trimmed = String(text || "").trim();
  const ood = model.ood || {};
  if (trimmed.length < (ood.min_chars ?? 80)) return "texto_vacio_o_muy_corto";
  const tokens = tokenize(text);
  if (tokens.length < (ood.min_tokens ?? 8)) return "muy_pocas_palabras";
  if (tokens.length >= (ood.spanish_min_tokens ?? 20)) {
    let hits = 0;
    for (const token of tokens) if (model.stopwords.has(token)) hits += 1;
    if (hits < (ood.spanish_min_hits ?? 2)) return "idioma_no_espanol";
  }
  let known = 0;
  for (const token of tokens) {
    if (Object.prototype.hasOwnProperty.call(model.vocab, token)) known += 1;
  }
  const oovRate = 1 - known / Math.max(1, tokens.length);
  if (oovRate > (ood.max_oov_rate ?? 0.6)) return "fuera_de_vocabulario";
  const margin = ood.review_margin ?? 0.08;
  if (Math.abs(probability - Number(model.threshold || 0.5)) <= margin) {
    return "cerca_del_umbral";
  }
  return null;
}

export function predict(model, text) {
  const { tokens, sparse } = vectorize(model, text);
  const hidden = new Array(model.hiddenSize).fill(0);
  for (const [idx, value] of sparse) {
    const row = model.w1[idx];
    if (!row) continue;
    for (let h = 0; h < model.hiddenSize; h += 1) {
      hidden[h] += value * row[h];
    }
  }
  for (let h = 0; h < model.hiddenSize; h += 1) {
    hidden[h] = relu(hidden[h] + (model.b1?.[h] || 0));
  }
  let logit = Number(model.b2 || 0);
  for (let h = 0; h < model.hiddenSize; h += 1) {
    logit += hidden[h] * (model.w2?.[h] || 0);
  }
  const temperature = Number(model.temperature || 1);
  const probability = sigmoid(logit / temperature);
  const threshold = Number(model.threshold || 0.5);
  const reason = oodDecision(model, text, probability);
  const rejected = Boolean(reason && reason !== "cerca_del_umbral");
  return {
    probability,
    threshold,
    label: probability >= threshold ? 1 : 0,
    labelName: probability >= threshold ? "Finanzas / riesgo" : "Mercado / propuesta",
    rejectReason: rejected ? reason : null,
    reviewReason: reason === "cerca_del_umbral" ? reason : null,
    tokenCount: tokens.length,
    featureCount: sparse.length,
  };
}

export function spanishReason(reason) {
  const map = {
    texto_vacio_o_muy_corto: "El texto está vacío o tiene muy poco contenido.",
    muy_pocas_palabras: "Hay muy pocas palabras para clasificar.",
    idioma_no_espanol: "El texto no parece estar en español.",
    fuera_de_vocabulario: "El texto está fuera del dominio del modelo.",
    cerca_del_umbral: "La probabilidad está cerca del umbral; conviene revisarlo.",
  };
  return map[reason] || "Entrada no clasificable.";
}
```

### Carga del modelo y clasificación en la app

El siguiente es `practica1/pwa/app.js` completo: carga `model.json`, conecta el botón, clasifica y abre la ventana de análisis. Es JavaScript de navegador; el notebook no lo ejecuta.

```javascript
import { createModel, predict, spanishReason } from "./inference.js";

const els = {
  input: document.querySelector("#case-text"),
  button: document.querySelector("#classify-btn"),
  clear: document.querySelector("#clear-btn"),
  status: document.querySelector("#model-status"),
  result: document.querySelector("#result"),
  empty: document.querySelector("#empty-state"),
  badge: document.querySelector("#result-badge"),
  title: document.querySelector("#result-title"),
  probability: document.querySelector("#probability"),
  probabilityBar: document.querySelector("#probability-bar"),
  threshold: document.querySelector("#threshold"),
  thresholdMarker: document.querySelector("#threshold-marker"),
  explanation: document.querySelector("#explanation"),
  meta: document.querySelector("#result-meta"),
  examples: document.querySelectorAll("[data-example]"),
  oodNote: document.querySelector("#ood-note"),
};

let model = null;

const EXAMPLES = {
  finance:
    "La inversión inicial del proyecto es de USD 1,250,000 e incluye la compra de maquinaria, el capital de trabajo y los costos de instalación. El flujo de efectivo proyectado a cinco años tiene un margen bruto del 32%, un VAN positivo de USD 410,000 y una TIR del 18.4%. El punto de equilibrio se alcanza en el mes 14. Los principales riesgos son el aumento del costo de los materiales y la demora en la cadena de suministro; se propone una contingencia del 8% del presupuesto.",
  market:
    "El cliente objetivo son resorts de aventura en la península de Yucatán que necesitan una flota de vehículos todo terreno para tours guiados. El mercado de turismo de aventura crece 7% anual y los competidores principales son Polaris, Can-Am y proveedores locales de renta. La propuesta se dirige a gerentes de operaciones y dueños de hospedajes boutique. El canal principal será la venta directa con demostraciones en sitio, apoyada por marketing digital y alianzas con operadores turísticos.",
  oodEnglish:
    "Hello, I would like to book a flight to New York next week. Please confirm the price and the baggage allowance for two passengers.",
  oodRecipe:
    "Preheat the oven to 180 degrees. Mix flour, sugar and butter, then bake the cake for forty minutes until golden. Let it cool before serving.",
};

function setStatus(text, kind = "loading") {
  els.status.textContent = text;
  els.status.dataset.kind = kind;
}

async function loadModel() {
  try {
    setStatus("Cargando modelo…", "loading");
    const response = await fetch("./model.json?v=1", { cache: "no-cache" });
    if (!response.ok) throw new Error(`HTTP ${response.status}`);
    const raw = await response.json();
    model = createModel(raw);
    const summary = model.metrics_summary || {};
    const suffix = model.provisional ? " · evaluación provisional" : "";
    const trained = summary.train_size ?? "?";
    setStatus(`Entrenado con ${trained} fragmentos${suffix}`, "ready");
    const trainNote = document.querySelector("#train-size");
    if (trainNote) {
      const total = summary.fragment_count ?? "?";
      const pos = summary.train_positive ?? "?";
      const neg = summary.train_negative ?? "?";
      const augmented = summary.train_augmented ?? 0;
      trainNote.textContent =
        `El modelo se entrenó con ${trained} fragmentos (${pos} finanzas/riesgo y ${neg} mercado/propuesta). ` +
        `Incluye ${augmented} fragmentos extra de los mismos equipos de entrenamiento. ` +
        `Validación: ${summary.val_size ?? "?"}. Test no usado para entrenar: ${summary.test_size ?? "?"}. ` +
        `Dataset base: ${total} fragmentos.`;
    }
    els.button.disabled = false;
    if (els.input.value.trim()) classify();
  } catch (error) {
    console.error(error);
    setStatus("No se pudo cargar el modelo. Recarga la página.", "error");
    els.button.disabled = true;
  }
}

function formatPct(value) {
  return `${(value * 100).toFixed(1)}%`;
}

function render(result) {
  els.empty.hidden = true;
  els.result.hidden = false;
  els.oodNote.hidden = true;

  const p = result.probability;
  const t = result.threshold;
  els.probability.textContent = formatPct(p);
  els.threshold.textContent = formatPct(t);
  els.probabilityBar.style.width = `${Math.max(0, Math.min(100, p * 100))}%`;
  els.thresholdMarker.style.left = `${Math.max(0, Math.min(100, t * 100))}%`;
  els.meta.textContent = `${result.tokenCount} tokens · ${result.featureCount} features activas`;

  if (result.rejectReason) {
    els.badge.textContent = "No clasificable";
    els.badge.dataset.kind = "reject";
    els.title.textContent = "Entrada fuera de dominio";
    els.explanation.textContent = spanishReason(result.rejectReason);
    els.oodNote.hidden = false;
    els.oodNote.textContent =
      "La app prefiere rechazar antes que forzar una clase. Prueba con un fragmento de un business case.";
    return;
  }

  const isFinance = result.label === 1;
  els.badge.textContent = isFinance ? "Finanzas / riesgo" : "Mercado / propuesta";
  els.badge.dataset.kind = isFinance ? "finance" : "market";
  els.title.textContent = isFinance
    ? "El fragmento parece financiero o de riesgo"
    : "El fragmento parece de mercado o propuesta";
  els.explanation.textContent = isFinance
    ? "El modelo encontró vocabulario asociado con inversión, ingresos, costos, rentabilidad o riesgos."
    : "El modelo encontró vocabulario asociado con cliente, mercado, competencia, propuesta, operación o marketing.";

  if (result.reviewReason) {
    els.oodNote.hidden = false;
    els.oodNote.textContent =
      "La probabilidad está cerca del umbral. La clase es tentativa; revisa el fragmento antes de decidir.";
  }
}

function classify() {
  if (!model) return;
  const text = els.input.value;
  const result = predict(model, text);
  render(result);
}

function clearAll() {
  els.input.value = "";
  els.result.hidden = true;
  els.empty.hidden = false;
  els.input.focus();
}

els.button.addEventListener("click", classify);
els.clear.addEventListener("click", clearAll);
els.input.addEventListener("keydown", (event) => {
  if ((event.metaKey || event.ctrlKey) && event.key === "Enter") classify();
});

for (const button of els.examples) {
  button.addEventListener("click", () => {
    const key = button.dataset.example;
    els.input.value = EXAMPLES[key] || "";
    els.input.focus();
    classify();
  });
}

if ("serviceWorker" in navigator) {
  window.addEventListener("load", () => {
    navigator.serviceWorker.register("./sw.js").catch((error) => console.warn("SW", error));
  });
}

document.querySelector("#open-eval")?.addEventListener("click", () => {
  const popup = window.open(
    "./evaluacion.html",
    "practica1-evaluacion",
    "popup=yes,width=1180,height=820,resizable=yes,scrollbars=yes"
  );
  if (!popup) window.location.href = "./evaluacion.html";
});

window.addEventListener("message", (event) => {
  if (event.origin !== window.location.origin) return;
  if (event.data?.type !== "try-fragment") return;
  els.input.value = event.data.text || "";
  classify();
  els.input.focus();
});

const pending = sessionStorage.getItem("try-fragment");
if (pending) {
  sessionStorage.removeItem("try-fragment");
  els.input.value = pending;
}

loadModel();
```


## 4. Errores y fuera de dominio

Los 20 errores se agrupan en sección mixta (7), negación o contraste (7), clase confundida (5) y texto corto (1). Cada error lleva hipótesis. La app abre una ventana aparte con dos paneles de scroll independiente: errores y OOD.

De 10 entradas fuera de dominio, **8 se rechazan**. Deporte y texto legal en español no se rechazan: la app los clasifica como mercado con probabilidad baja. No se cae. Esa diferencia entre respuesta esperada y real es parte del análisis.

### Código de la ventana de análisis

`practica1/pwa/evaluacion.js` renderiza los 20 errores, sus filtros y los 10 casos OOD:

```javascript
const CATEGORIES = {
  seccion_mixta: "Sección mixta",
  negacion_o_contraste: "Negación o contraste",
  confusion_de_clase: "Clase confundida",
  texto_corto: "Texto corto",
  tabla_numerica: "Tabla numérica",
  cerca_del_umbral: "Cerca del umbral",
};

function money(value) {
  return Number(value).toFixed(1);
}

function labelName(value) {
  return value === 1 ? "Finanzas" : "Mercado";
}

function tryFragment(text) {
  if (window.opener && !window.opener.closed) {
    window.opener.postMessage({ type: "try-fragment", text }, window.location.origin);
    window.opener.focus();
    return;
  }
  sessionStorage.setItem("try-fragment", text);
  window.location.href = "./index.html";
}

function render(data) {
  const note = document.querySelector("#eval-note");
  const dataset = data.dataset || {};
  note.textContent = data.provisional
    ? `Evaluación provisional. Entrenado con ${dataset.train_size ?? "?"} fragmentos; el test aún usa etiquetas débiles.`
    : `Corrida final. Entrenado con ${dataset.train_size ?? "?"} fragmentos.`;

  const costs = data.costs || {};
  document.querySelector("#eval-kpis").innerHTML = [
    ["Entrenamiento", dataset.train_size],
    ["Modelo / 1000", costs.model_per_1000],
    ["Keyword / 1000", costs.keyword_per_1000],
    ["Mayoría / 1000", costs.majority_per_1000],
  ]
    .map(([name, value]) => `<div class="kpi"><span>${name}</span><strong>${name === "Entrenamiento" ? value ?? "—" : money(value)}</strong></div>`)
    .join("");

  const c = data.confusion || {};
  document.querySelector("#confusion").innerHTML = `
    <div></div><div class="mh">Pred. mercado</div><div class="mh">Pred. finanzas</div>
    <div class="mh">Real mercado</div><div>TN ${c.tn ?? "—"}</div><div>FP ${c.fp ?? "—"}</div>
    <div class="mh">Real finanzas</div><div>FN ${c.fn ?? "—"}</div><div>TP ${c.tp ?? "—"}</div>`;

  const cal = data.calibration || {};
  document.querySelector("#calibration").innerHTML = `
    <p>ECE antes: <strong>${Number(cal.ece_before).toFixed(3)}</strong>. ECE después: <strong>${Number(cal.ece_after).toFixed(3)}</strong>. Temperatura: ${Number(cal.temperature).toFixed(2)}.</p>
    <p>Costo/1000 sin calibrar: ${money(costs.raw_per_1000)}. Con calibración y umbral ${money((data.threshold || 0) * 100)}%: ${money(costs.model_per_1000)}.</p>
    <p>La calibración baja el ECE, pero en esta corrida no baja el costo. La probabilidad ordena mejor de lo que estima una frecuencia exacta.</p>`;

  const filters = document.querySelector("#error-filters");
  const list = document.querySelector("#error-list");
  const groups = Object.entries(data.errors_grouped || {});
  filters.innerHTML =
    `<button type="button" data-cat="all" aria-pressed="true">Todos (${(data.errors || []).length})</button>` +
    groups
      .map(([key, count]) => `<button type="button" data-cat="${key}" aria-pressed="false">${CATEGORIES[key] || key} (${count})</button>`)
      .join("");

  function paint(category) {
    const rows = (data.errors || []).filter((row) => category === "all" || row.category === category);
    list.innerHTML = rows
      .map((row, index) => {
        const kind = row.true_label === 1 ? "FN" : "FP";
        return `<article class="error-item">
          <header>
            <span class="tag ${kind === "FN" ? "bad" : "warn"}">${kind} · ${CATEGORIES[row.category] || row.category}</span>
            <button class="load" type="button" data-error="${index}">Probar</button>
          </header>
          <p>${row.hypothesis || ""} Real: ${labelName(row.true_label)}. Predicho: ${labelName(row.predicted_label)} a ${(row.probability * 100).toFixed(1)}% (umbral ${(row.threshold * 100).toFixed(1)}%).</p>
          <p class="quote">${row.text || ""}</p>
        </article>`;
      })
      .join("");
    list.querySelectorAll("[data-error]").forEach((button) => {
      button.addEventListener("click", () => tryFragment(rows[Number(button.dataset.error)]?.text || ""));
    });
  }

  paint("all");
  filters.querySelectorAll("button").forEach((button) => {
    button.addEventListener("click", () => {
      filters.querySelectorAll("button").forEach((item) => item.setAttribute("aria-pressed", "false"));
      button.setAttribute("aria-pressed", "true");
      paint(button.dataset.cat);
    });
  });

  document.querySelector("#ood-list").innerHTML = (data.ood || [])
    .map((row) => {
      const ok = row.rejected;
      return `<article class="ood-item">
        <header>
          <span class="tag ${ok ? "" : "bad"}">${row.id} · ${ok ? "rechazada" : "no rechazada"}</span>
          <span class="meta">${row.actual_reject_reason || "clasificó"} · ${(row.probability * 100).toFixed(1)}%</span>
        </header>
        <p>Esperado: ${row.expected} (${row.reason_expected || "fuera de dominio"}). ${ok ? "La app rechazó la entrada." : "La app no la rechazó; la clasificó igual."}</p>
        <p class="quote">${row.text || "(vacío)"}</p>
      </article>`;
    })
    .join("");
}

fetch("./evaluation.json?v=2", { cache: "no-cache" })
  .then((response) => {
    if (!response.ok) throw new Error(`HTTP ${response.status}`);
    return response.json();
  })
  .then(render)
  .catch((error) => {
    console.error(error);
    document.querySelector("#eval-note").textContent = "No se pudo cargar evaluation.json.";
  });
```

La estructura de la ventana está en `practica1/pwa/evaluacion.html`:

```html
<!doctype html>
<html lang="es">
  <head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover" />
    <meta name="theme-color" content="#0b1020" />
    <title>Análisis · Triaje financiero</title>
    <link rel="stylesheet" href="./style.css" />
  </head>
  <body class="analysis-body">
    <div class="analysis-shell">
      <header class="analysis-top">
        <div>
          <p class="eyebrow">Rúbrica · sección 4</p>
          <h1>Errores y fuera de dominio</h1>
          <p id="eval-note" class="eval-note">Cargando evaluación…</p>
        </div>
        <a class="ghost" href="./index.html">Volver al clasificador</a>
      </header>

      <div id="eval-kpis" class="kpi-grid"></div>

      <div class="analysis-meta">
        <article class="mini-card">
          <p class="eyebrow">Matriz de confusión</p>
          <div id="confusion" class="matrix"></div>
        </article>
        <article class="mini-card">
          <p class="eyebrow">Calibración y costo</p>
          <div id="calibration" class="stack"></div>
        </article>
      </div>

      <div class="analysis-panes">
        <section class="pane">
          <div class="pane-head">
            <div>
              <p class="eyebrow">Análisis de errores</p>
              <h2>20 casos mal clasificados</h2>
            </div>
            <div id="error-filters" class="filters"></div>
          </div>
          <div id="error-list" class="pane-scroll"></div>
        </section>
        <section class="pane">
          <div class="pane-head">
            <div>
              <p class="eyebrow">Fuera de dominio</p>
              <h2>10 entradas ajenas</h2>
            </div>
          </div>
          <div id="ood-list" class="pane-scroll"></div>
        </section>
      </div>
    </div>
    <script type="module" src="./evaluacion.js"></script>
  </body>
</html>
```

### Verificación de los datos de evaluación

La siguiente celda carga el mismo `evaluation.json` que usa la app y comprueba los conteos.


In [1]:
EVAL_JSON = "{\"provisional\": true, \"test_label_source\": \"weak_provisional\", \"dataset\": {\"fragment_count\": 666, \"total_with_augmentation\": 832, \"train_size\": 500, \"train_augmented\": 166, \"train_positive\": 115, \"train_negative\": 385, \"val_size\": 130, \"val_positive\": 27, \"test_size\": 202, \"test_positive\": 30, \"document_count\": 46, \"team_count\": 34, \"fit_on\": \"train\"}, \"threshold\": 0.26, \"costs\": {\"fp\": 1.0, \"fn\": 5.0, \"model_per_1000\": 514.8514851485148, \"raw_per_1000\": 519.8019801980198, \"majority_per_1000\": 742.5742574257425, \"keyword_per_1000\": 465.34653465346537, \"logistic_per_1000\": 504.950495049505}, \"confusion\": {\"tn\": 138, \"fp\": 34, \"fn\": 14, \"tp\": 16}, \"metrics\": {\"accuracy\": 0.7623762376237624, \"precision\": 0.32, \"recall\": 0.5333333333333333, \"f1\": 0.4}, \"calibration\": {\"temperature\": 5.018911339130734, \"ece_before\": 0.15076843394229233, \"ece_after\": 0.08425164656336487}, \"errors_grouped\": {\"seccion_mixta\": 7, \"negacion_o_contraste\": 7, \"confusion_de_clase\": 5, \"texto_corto\": 1}, \"errors\": [{\"section\": \"vehiculo\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.9046, \"threshold\": 0.26, \"category\": \"seccion_mixta\", \"hypothesis\": \"La sección mezcla propuesta técnica con costos o riesgos.\", \"text\": \"Concepto USD Nota BOM directo 4,012 6 de 18 renglones estimados Mano de obra (100 h × MXN 130 ÷ 20) 650 Supuesto Flete, aduana, seguro y reserva de garantía 814 USD 400 + 3 % + 3 % del precio Costo entregado 5,476 Precio de venta 6,900\"}, {\"section\": \"competencia\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.8947, \"threshold\": 0.26, \"category\": \"negacion_o_contraste\", \"hypothesis\": \"Una negación o contraste cambia el sentido financiero.\", \"text\": \"Capacidad: Motor y Canal de Marco y Modelo Carga / Precio de lista Garantia y soporte operativo Potencia adquisición Arrastre $470,000 MXN - 3 Cilindros 1,250 lbs / $510,000 MXN 2 años limitados. Requiere Polaris Pro SD Intermediario Diésel / 24 HP 2,500 lbs ($23,500 USD - translado a agencia para validez. $25,500 USD) $420,000 MXN - John Deere Gator 3 Cilindros 1,400 lbs / $460,000 MXN Distribuidor 1 año o 1,000 horas. Enfoque XUV855M Diésel / 22.8 HP 1,500 lbs ($21,000 USD - africola agricola, no turistico. $23,000 USD) $490,000 MXN - 2 años a 2,000 horas.\"}, {\"section\": \"suministro\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.7698, \"threshold\": 0.26, \"category\": \"seccion_mixta\", \"hypothesis\": \"La sección mezcla propuesta técnica con costos o riesgos.\", \"text\": \"Adquirir materiales conforme avance la Vincular los cobros a los hitos contractuales. producción. Reserva presupuestaria Control mensual Contemplar el 10 % de contingencia en los Supervisar ingresos, egresos y obligaciones costos. pendientes. Nota financiera: El capital inicial de $5,000,000 MXN y su distribución son estimaciones de planeación. El esquema de pagos está sujeto a negociación con OAH y su viabilidad deberá comprobarse mediante un flujo de efectivo mensual que contemple costos operativos, impuestos y posibles retrasos.\"}, {\"section\": \"operaciones\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.7655, \"threshold\": 0.26, \"category\": \"seccion_mixta\", \"hypothesis\": \"La sección mezcla propuesta técnica con costos o riesgos.\", \"text\": \"Comprar frente a rentar Su solicitud señala que la compra permitiría reducir los costos generales al eliminar la renta. A continuación lo cuantificamos. Concepto Monto Costo de rentar una flota equivalente en el horizonte $82,499,308 Costo total de propiedad de la flota adquirida $59,920,900 Ahorro acumulado $22,578,408 (27.4%) Año en que la compra supera a la renta Año 4 Retorno del cliente sobre su inversión 58.0% En qué plazo conviene la compra. El desembolso inicial hace que la renta resulte más barata los primeros años;\"}, {\"section\": \"operaciones\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.7617, \"threshold\": 0.26, \"category\": \"negacion_o_contraste\", \"hypothesis\": \"Una negación o contraste cambia el sentido financiero.\", \"text\": \"El costo preliminar de fabricación se estima en $225,000 MXN por vehículo básico. Adicionalmente, cada unidad contempla un kit de siete accesorios modulares con un costo estimado de $12,550 MXN. Se incluye una reserva de contingencia del 10% sobre los costos presupuestados para posibles variaciones durante la ejecución del contrato. PROPUESTA COMERCIAL Precio propuesto de $350,000 MXN por unidad equipada, sin IVA. El contrato de 70 unidades asciende a $24,500,000 MXN, con un excedente presupuestado de $5,389,150 y un margen estimado de 22%, considerando la contingencia.\"}, {\"section\": \"vehiculo\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.7527, \"threshold\": 0.26, \"category\": \"seccion_mixta\", \"hypothesis\": \"La sección mezcla propuesta técnica con costos o riesgos.\", \"text\": \"Indicador Valor Precio por vehículo (entregado en Jones Creek) USD 6,900 Costo entregado por vehículo USD 5,476.38 Margen por vehículo USD 1,423.62 (20.6%) Ingresos de 44 vehículos USD 303,600 Utilidad antes de impuestos USD 26,639 (8.8%) Punto de equilibrio Unidad 20 (CapEx total USD 36,000) Servicio anual (44 vehículos) USD 15,400; aporte neto USD 9,400 Valor del contrato a 5 años USD 380,600 (vehículos más servicio)\"}, {\"section\": \"cliente\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.7357, \"threshold\": 0.26, \"category\": \"negacion_o_contraste\", \"hypothesis\": \"Una negación o contraste cambia el sentido financiero.\", \"text\": \"No tenemos un padrón verificado de campamentos juveniles de recreación, por lo que no inventamos un mercado total. Mostramos escenarios del contrato con JCO, basados en el cupo que describe la guía. Escenario Vehículos Ingresos por vehículos (USD) Lógica Año 1 (base) 44 303,600 4 clases × 10 alumnos + 4 de repuesto Ampliación (años 2 a 5) 53 +62,100 Clases de 12 alumnos: 48 en uso + 5 de repuesto Servicio a 5 años (44) — 77,000 USD 350 por vehículo al año Los ingresos de ampliación suponen el mismo precio de USD 6,900; son supuestos del equipo, no pronósticos.\"}, {\"section\": \"ventas\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.7243, \"threshold\": 0.26, \"category\": \"confusion_de_clase\", \"hypothesis\": \"El vocabulario del fragmento se parece a la otra clase.\", \"text\": \"50 vehículos (mes 18) 30% $7,350,000 Total del contrato 100% $24,500,000 ADMINISTRACIÓN Y CONTROL FINANCIERO El anticipo de Outdoor Adventure Holdings (OAH) se destinará principalmente a la compra de materiales y componentes necesarios para iniciar la producción. CIV conservará una reserva de capital de trabajo de $1,800,000 MXN para respaldar sus necesidades operativas, mientras que las adquisiciones se realizarán de manera escalonada conforme al programa de fabricación.\"}, {\"section\": \"competencia\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.7159, \"threshold\": 0.26, \"category\": \"confusion_de_clase\", \"hypothesis\": \"El vocabulario del fragmento se parece a la otra clase.\", \"text\": \"Tiempo de Kubota RTV- 3 Cilindros 1,300 lbs / $530,000 MXN Distribuidor de espera largos por refacciones X1140 Diésel / 24 HP 1,300 lbs ($24,500 USD - maquinaria importadas. $26,500 USD) 3 años integrales. Capacitación Ataraxia AT-X 3 Cilindros 1,074 lbs / $450,000 MXN Venta Directa de técnica para mantenimiento in- Titan Diésel / 23 HP 4,000 lbs ($22,500 USD) Fabrica (B2B) house 3 años integrales.\"}, {\"section\": \"vehiculo\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.7131, \"threshold\": 0.26, \"category\": \"seccion_mixta\", \"hypothesis\": \"La sección mezcla propuesta técnica con costos o riesgos.\", \"text\": \"Qué pasa si los costos o el precio se mueven Escenario Margen por vehículo Equilibrio Utilidad del contrato Base USD 1,424 Unidad 20 USD 26,639 BOM +15 % USD 822 Unidad 35 USD 163 Precio −10 % USD 734 Unidad 39 −USD 3,721 BOM +15 % y precio −10 % USD 132 Unidad 213 −USD 30,203 Para JCO: ¿cuánto debe cobrar para recuperar la inversión? Con 44 vehículos y servicio, JCO paga USD 303,600 el primer año y USD 15,400 al año de servicio, o USD 76,120 al año repartidos en cinco temporadas.\"}, {\"section\": \"competencia\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.5729, \"threshold\": 0.26, \"category\": \"negacion_o_contraste\", \"hypothesis\": \"Una negación o contraste cambia el sentido financiero.\", \"text\": \"Tours Guiados ($2,500 MXN / ~$125 USD): Tarifa promedio estándar de parques de aventura de clase mundial en la región. Cubre el uso del vehículo, guía bilingüe, consumibles y seguros. Mantiene un margen de ganancia conservador del 40% tras deducir la nómina operativa de los guías y paramédicos. Day-Pass / Renta Privada ($3,500 MXN / ~$175 USD): Tarifa premium para movilidad autónoma por las 250,000 hectáreas (sin límite de horas). Al eliminar el sueldo del guía de la ecuación, el margen de ganancia asciende al 50%.\"}, {\"section\": \"ventas\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.4979, \"threshold\": 0.26, \"category\": \"confusion_de_clase\", \"hypothesis\": \"El vocabulario del fragmento se parece a la otra clase.\", \"text\": \"• Avance del proceso: RFP entregado, presentación realizada, finalista y contrato firmado. • Precio promedio logrado contra USD 6,900 y margen contra USD 1,424. • Ocupación de las clases en 2028 (meta inicial de 60%, supuesto). • Referencias y casos de éxito con otros campamentos.\"}, {\"section\": \"operaciones\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.473, \"threshold\": 0.26, \"category\": \"negacion_o_contraste\", \"hypothesis\": \"Una negación o contraste cambia el sentido financiero.\", \"text\": \"la compra la supera a partir del cuarto y la ventaja se amplía desde entonces, porque el activo sigue sirviendo mientras la renta se paga cada mes. Supuestos de los que depende este modelo Supuesto Si falla Cómo lo mitigamos 1,040 horas de operación por unidad El costo por hora y el punto de equilibrio se El contrato de servicio se ajusta por horas reales, no por al año desplazan calendario Consumo medio de 1.8 L/h Varía el costo de combustible del cliente Se sustituye con la medición de la unidad demostrativa (hipótesis) en el mes 3\"}, {\"section\": \"operaciones\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.4727, \"threshold\": 0.26, \"category\": \"seccion_mixta\", \"hypothesis\": \"La sección mezcla propuesta técnica con costos o riesgos.\", \"text\": \"The EXR-26 Balam enters the property long before the first guests arrive. Twenty Balam Build units arrive at month 6 to support the construction of Outdoor Adventure Holdings' 250,000-hectare project, replacing rented vehicles from month 6.\"}, {\"section\": \"vehiculo\", \"true_label\": 0, \"predicted_label\": 1, \"probability\": 0.4687, \"threshold\": 0.26, \"category\": \"seccion_mixta\", \"hypothesis\": \"La sección mezcla propuesta técnica con costos o riesgos.\", \"text\": \"From the coast to the jungle, the EXR-26 Balam is one heavy-duty platform for loose sand, rainy-season mud and white limestone (sascab) roads, built for 40 °C days, humidity and salt air.\"}, {\"section\": \"ingresos\", \"true_label\": 1, \"predicted_label\": 0, \"probability\": 0.0599, \"threshold\": 0.26, \"category\": \"confusion_de_clase\", \"hypothesis\": \"El vocabulario del fragmento se parece a la otra clase.\", \"text\": \"un compromiso de disponibilidad medido y reportado cada mes; el kit de refacciones en piso; y la capacitación y certificación anual de su personal. Un vehículo de catálogo se compra a un distribuidor y se repara donde ese distribuidor indique. En Yucatán, eso se mide en días de tránsito.\"}, {\"section\": \"inversion\", \"true_label\": 1, \"predicted_label\": 0, \"probability\": 0.0609, \"threshold\": 0.26, \"category\": \"confusion_de_clase\", \"hypothesis\": \"El vocabulario del fragmento se parece a la otra clase.\", \"text\": \"La propuesta económica de CIV contempla la fabricación de 70 vehículos utilitarios modulares, equipados con siete accesorios intercambiables y un paquete de servicios que incluye consumibles para las primeras 250 horas de operación, capacitación del personal de OAH y transporte hasta Yucatán.\"}, {\"section\": \"ingresos\", \"true_label\": 1, \"predicted_label\": 0, \"probability\": 0.0774, \"threshold\": 0.26, \"category\": \"negacion_o_contraste\", \"hypothesis\": \"Una negación o contraste cambia el sentido financiero.\", \"text\": \"Vehículo Motor Lista (USD) Control y edad | UTC · Análisis de mercado · Página 1 Polaris RZR cc 7,499 + 400 10 años o más; limitador ajustable y control por app Polaris Ranger cc 6,299 Control de velocidad y geocerca por app Kayo S cc 5,599 Para niños de 10 años o más; limitador de acelerador Hisun Sector cc 5,649 Limitador mecánico ajustable SSR SRU170RS 169 cc 4,459 Sin dato de control de velocidad Son precios de venta al público para familias de la guía de Dirt Wheels de 2024 y del sitio de Polaris; JCO compraría una flota con servicio. Ningún vehículo de esa lista declara un rango desde 7 años con ajuste de talla.\"}, {\"section\": \"riesgos\", \"true_label\": 1, \"predicted_label\": 0, \"probability\": 0.0783, \"threshold\": 0.26, \"category\": \"texto_corto\", \"hypothesis\": \"El fragmento tiene poco contexto para decidir.\", \"text\": \"Fuentes de mercado: Dirt Wheels (guía de UTV juveniles 2024) y sitio de Polaris. | · Thor Cub · Página 10\"}, {\"section\": \"riesgos\", \"true_label\": 1, \"predicted_label\": 0, \"probability\": 0.0839, \"threshold\": 0.26, \"category\": \"negacion_o_contraste\", \"hypothesis\": \"Una negación o contraste cambia el sentido financiero.\", \"text\": \"sin contacto previo ni datos de flota. Fuentes: Dirt Wheels, 2024 Youth UTV Buyer’s Guide; polaris.com (RZR 200); guía Baja SAE 2027 Business Presentation. | UTC · Análisis de mercado · Página 2\"}], \"ood\": [{\"id\": \"ood_01\", \"expected\": \"rechazar\", \"reason_expected\": \"texto vacío\", \"actual_reject_reason\": \"texto_vacio_o_muy_corto\", \"rejected\": true, \"probability\": 0.0, \"text\": \"\"}, {\"id\": \"ood_02\", \"expected\": \"rechazar\", \"reason_expected\": \"otro idioma\", \"actual_reject_reason\": \"fuera_de_vocabulario\", \"rejected\": true, \"probability\": 0.4052, \"text\": \"Hello, I would like to book a flight to New York next week. Please confirm the price and the baggage allowance.\"}, {\"id\": \"ood_03\", \"expected\": \"rechazar\", \"reason_expected\": \"receta, fuera de dominio\", \"actual_reject_reason\": \"idioma_no_espanol\", \"rejected\": true, \"probability\": 0.3377, \"text\": \"Preheat the oven to 180 degrees. Mix flour, sugar and butter, then bake the cake for forty minutes until golden.\"}, {\"id\": \"ood_04\", \"expected\": \"rechazar\", \"reason_expected\": \"código, fuera de dominio\", \"actual_reject_reason\": \"muy_pocas_palabras\", \"rejected\": true, \"probability\": 0.1712, \"text\": \"def fibonacci(n): if n <= 1: return n return fibonacci(n-1) + fibonacci(n-2)\"}, {\"id\": \"ood_05\", \"expected\": \"rechazar\", \"reason_expected\": \"deporte, fuera de dominio\", \"actual_reject_reason\": null, \"rejected\": false, \"probability\": 0.1178, \"text\": \"El equipo local ganó el partido tres a uno con dos goles en el segundo tiempo y subió al primer lugar de la tabla.\"}, {\"id\": \"ood_06\", \"expected\": \"rechazar\", \"reason_expected\": \"texto legal, fuera de dominio\", \"actual_reject_reason\": null, \"rejected\": false, \"probability\": 0.1248, \"text\": \"La Constitución establece que toda persona tiene derecho a la educación y que el Estado debe garantizar su acceso en condiciones de igualdad.\"}, {\"id\": \"ood_07\", \"expected\": \"rechazar\", \"reason_expected\": \"texto de relleno\", \"actual_reject_reason\": \"fuera_de_vocabulario\", \"rejected\": true, \"probability\": 0.1473, \"text\": \"Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua.\"}, {\"id\": \"ood_08\", \"expected\": \"rechazar\", \"reason_expected\": \"solo emojis\", \"actual_reject_reason\": \"texto_vacio_o_muy_corto\", \"rejected\": true, \"probability\": 0.1712, \"text\": \"🎉🎉🎉 🔥🔥🔥 ✨✨✨\"}, {\"id\": \"ood_09\", \"expected\": \"rechazar\", \"reason_expected\": \"texto sin sentido\", \"actual_reject_reason\": \"texto_vacio_o_muy_corto\", \"rejected\": true, \"probability\": 0.1712, \"text\": \"asdf qwer zxcv mnbv poiu lkjh ghfd trew yuiop ñlkj mnbg vcxz\"}, {\"id\": \"ood_10\", \"expected\": \"rechazar\", \"reason_expected\": \"receta en español, fuera de dominio\", \"actual_reject_reason\": \"fuera_de_vocabulario\", \"rejected\": true, \"probability\": 0.1289, \"text\": \"Receta de mole poblano: use chile ancho, chile pasilla, chocolate, tortilla, almendras y especias; muela todo y cocine a fuego lento.\"}]}"
evaluation = json.loads(EVAL_JSON)
print('Errores:', len(evaluation['errors']))
print('Grupos:', evaluation['errors_grouped'])
rejected = sum(1 for x in evaluation['ood'] if x['rejected'])
print('OOD rechazadas:', rejected, 'de', len(evaluation['ood']))
print('Entrenamiento:', evaluation['dataset']['train_size'], '| extra:', evaluation['dataset']['train_augmented'])
print('Costo/1000:', evaluation['costs']['model_per_1000'])


Errores: 20
Grupos: {'seccion_mixta': 7, 'negacion_o_contraste': 7, 'confusion_de_clase': 5, 'texto_corto': 1}
OOD rechazadas: 8 de 10
Entrenamiento: 500 | extra: 166
Costo/1000: 514.8514851485148


## 5. Qué invalidaría la aplicación

No usarla como dictamen del business case, ni con OCR o tablas, ni si el texto no es un fragmento de caso en español. Tampoco si se publica el resultado provisional como si el test ya estuviera auditado.

La app no debe usarse para decidir sin revisión humana: es triaje. El PDF y el video son entregables separados; este notebook no los sustituye.
